# 🎓 AILS — Système de Recommandation Adaptatif Immersif
**Pipeline complet · Entraînement Synthétique → Données Réelles Google Sheets · Temps Réel**

| Ressource | Lien |
|-----------|------|
| 📊 Google Sheet | 🔒 privé (ID stocké dans les Secrets Colab : `AILS_SHEET_ID`) |
| 📱 Form 1 — Profil (Inter-module) | [Ouvrir](https://docs.google.com/forms/d/e/1FAIpQLScv_Aj7WfwUaUBtSBlYrh1zE3pHSuaSDjqAWKBCGuBYqxpRkg/viewform) |
| 📱 Form 2 — Auto-éval (Intra-module) | [Ouvrir](https://docs.google.com/forms/d/e/1FAIpQLSeRDZzG3r4EfHeXDlXPc76ruzKmyBLFo6P9z0JLJTXk2w81fA/viewform) |

**Onglets Sheet :** `Learner_Profile` · `Cognitive_Load` · `Form Responses 1` · `Form Responses 2` · `AILS_Results`

**Ordre d'exécution :**
1. Cellules **1→7** : Installation + chargement + nettoyage données
2. Cellules **8→10** : 🏗️ Génération base synthétique + pré-entraînement modèles
3. Cellules **11→14** : Définition AILS + fusion DB + batch pipeline
4. Cellule **15** : 📊 Métriques d'évaluation complètes (CF + CL)
5. Cellules **16→17** : 🔄 Ré-application sur Form Responses 1 & 2 + comparaison
6. Cellules **18→23** : 🔴 Surveillance temps réel
7. Cellules **21→25** : 🧪 **EXPÉRIMENTATION RÉELLE** — RAG + Exercices Bac + Boucle complète

In [ ]:
import json
import gspread
from google.oauth2.service_account import Credentials
from google.colab import userdata

try:
    secret_key = userdata.get('GOOGLE_SERVICE_ACCOUNT_CREDENTIALS')
    credentials_info = json.loads(secret_key)
    credentials_info['private_key'] = credentials_info['private_key'].replace('\\n', '\n')

    scopes = [
        'https://www.googleapis.com/auth/spreadsheets',
        'https://www.googleapis.com/auth/drive'
    ]

    creds = Credentials.from_service_account_info(credentials_info, scopes=scopes)
    gc = gspread.authorize(creds)

                                                                         
    SHEET_ID = userdata.get('AILS_SHEET_ID')
    sh = gc.open_by_key(SHEET_ID)

    print('✅ Connecté à la base Google Sheet (accès privé)')

except Exception as e:
    print(f'❌ Erreur authentification : {type(e).__name__}')
    print('   → Vérifiez les secrets GOOGLE_SERVICE_ACCOUNT_CREDENTIALS et AILS_SHEET_ID.')

In [ ]:

!pip install -q gspread google-auth pandas numpy scikit-learn matplotlib seaborn scipy requests ipywidgets
print('✅ Dépendances installées')

In [ ]:
import gspread
from google.colab import auth
from google.auth import default
from google.colab import userdata

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

MISTRAL_MODEL   = 'mistral-medium-2312'

SHEET_ID             = userdata.get('AILS_SHEET_ID')
TAB_LEARNER_PROFILE  = 'Learner_Profile'
TAB_COGNITIVE_LOAD   = 'Cognitive_Load'
TAB_FORM_ONBOARDING  = 'Form Responses 1'
TAB_FORM_POSTEX      = 'Form Responses 2'
RESULTS_TAB_NAME     = 'AILS_Results'

USE_LLM = bool(MISTRAL_API_KEY)
print(f"Mode LLM : {'✅ Mistral ACTIVÉ' if USE_LLM else '⚠️  template'}")


In [ ]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_sheet_tab(tab_name):
    try:
        ws   = sh.worksheet(tab_name)
        data = ws.get_all_records()
        df   = pd.DataFrame(data)
        print(f'  ✅ {tab_name:<30} → {df.shape[0]:>4} lignes × {df.shape[1]:>3} colonnes')
        return df
    except gspread.exceptions.WorksheetNotFound:
        print(f"  ⚠️  Onglet '{tab_name}' introuvable — DataFrame vide")
        return pd.DataFrame()

print('📥 Chargement...')
df_profile = load_sheet_tab(TAB_LEARNER_PROFILE)
df_cl_raw  = load_sheet_tab(TAB_COGNITIVE_LOAD)
df_form1   = load_sheet_tab(TAB_FORM_ONBOARDING)
df_form2   = load_sheet_tab(TAB_FORM_POSTEX)

print('\n📋 Colonnes disponibles :')
for name, df in [(TAB_LEARNER_PROFILE,df_profile),(TAB_COGNITIVE_LOAD,df_cl_raw),
                  (TAB_FORM_ONBOARDING,df_form1),(TAB_FORM_POSTEX,df_form2)]:
    if not df.empty:
        print(f'  [{name}]')
        for col in df.columns:
            print(f'    - {col}  ({df[col].dtype})')

In [ ]:

import re

LIKERT_MAP = {
                         
    '1':1,'2':2,'3':3,'4':4,'5':5,
                                                                               
                                                                           
    'strongly disagree':1.0,'disagree':2.0,'slightly disagree':2.5,
    'neutral':3.0,'slightly agree':3.5,'agree':4.0,'strongly agree':5.0,
                     
    'not at all':1,'a little':2,'somewhat':3,'quite a bit':4,'extremely':5,
    'very low':1,'low':2,'moderate':3,'high':4,'very high':5,
}
                                                                                           
                                                                                 
_LIKERT_KEYS_BY_LEN = sorted(LIKERT_MAP, key=len, reverse=True)

def _likert_label(s):
    lo = re.sub(r'\s+', ' ', s.lower().strip())
    if lo in LIKERT_MAP: return LIKERT_MAP[lo]
    for k in _LIKERT_KEYS_BY_LEN:
        if not k.isdigit() and k in lo: return LIKERT_MAP[k]
    return None

def parse_likert(val):
    if pd.isna(val) or str(val).strip() == '': return np.nan
    s = str(val).strip()
    m = re.match(r'^(\d+)', s)
    if m: return int(m.group(1))
    v = _likert_label(s)
    if v is not None: return v
    try: return float(s)
    except: return np.nan


def normalize_col_name(col):
    col = re.sub(r'[^\w\s]', '', str(col).strip())
    col = re.sub(r'\s+', '_', col)
    return col.lower()[:60]

def detect_id_column(df):
    patterns = ['learner_id','learner id','student_id','provided by the experimenter','identifiant']
    for col in df.columns:
        if any(p in col.lower() for p in patterns): return col
    return None

def clean_dataframe(df, name, is_likert=False, impute=True):
                                                                                                                   
    if df.empty: return df
    df = df.copy()
    before = len(df)
    df.replace('', np.nan, inplace=True)
    df.dropna(how='all', inplace=True)
    print(f'  [{name}] Lignes vides : {before-len(df)}')
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'  [{name}] Doublons : {before-len(df)}')
    df.columns = [normalize_col_name(c) for c in df.columns]
    for tc in [c for c in df.columns if 'timestamp' in c]:
        df[tc] = pd.to_datetime(df[tc], errors='coerce')
    id_col = detect_id_column(df)
    if id_col:
        df[id_col] = pd.to_numeric(df[id_col], errors='coerce')
    if is_likert:
        skip = {'timestamp', id_col, 'exercise_id', 'module_id', 'email_address'}
        for col in df.columns:
            if col in skip or 'timestamp' in col or col.endswith('_id'): continue
            if not pd.api.types.is_numeric_dtype(df[col]):                                          
                df[col] = df[col].apply(parse_likert)
                df[col] = pd.to_numeric(df[col], errors='coerce')
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        n_miss = df[col].isna().sum()
        if n_miss > 0 and impute:
            med = df[col].median()
            df[col].fillna(med, inplace=True)
            print(f'  [{name}] Imputation "{col}" : {n_miss} → {med:.2f}')
    empty_after = [c for c in df.columns if df[c].isna().all()]
    if empty_after:
        df.drop(columns=empty_after, inplace=True)
    print(f'  [{name}] ✅ {df.shape[0]} lignes × {df.shape[1]} col.')
    return df

print('🧹 Nettoyage...\n')
df_profile_clean = clean_dataframe(df_profile, TAB_LEARNER_PROFILE)
df_cl_clean      = clean_dataframe(df_cl_raw,  TAB_COGNITIVE_LOAD)
                                                                                                                   
                                                                                       
df_form1_clean   = clean_dataframe(df_form1,   TAB_FORM_ONBOARDING, is_likert=True, impute=False)
df_form2_clean   = clean_dataframe(df_form2,   TAB_FORM_POSTEX,     is_likert=True)
print('\n✅ Nettoyage terminé.')

In [ ]:

def find_col(df, keywords, default=None):
    for col in df.columns:
        if any(k.lower() in col.lower() for k in keywords): return col
    return default

def find_cols_containing(df, keywords):
    return [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]

print('🔍 Mapping colonnes...\n')

COL_LEARNER_ID = find_col(df_profile_clean, ['learner_id','id','student'])
COL_PRIOR      = find_col(df_profile_clean, ['prior','knowledge','niveau_initial','level'])
COL_CONFIDENCE = find_col(df_profile_clean, ['confidence','confiance','confident'])                                             
COL_MOTIVATION = find_col(df_profile_clean, ['motivation','motivat'])
COL_GROUP      = find_col(df_profile_clean, ['group','groupe','class','niveau'])
print(f'Learner_Profile → id={COL_LEARNER_ID} | prior={COL_PRIOR} | conf={COL_CONFIDENCE} | motiv={COL_MOTIVATION}')

COL_CL_LEARNER  = find_col(df_cl_clean, ['learner_id','learner','id','student'])
COL_CL_MODULE   = find_col(df_cl_clean, ['module','module_id'])
COL_CL_EXERCISE = find_col(df_cl_clean, ['exercise','exercice','ex_id'])
COL_HEAD_MV     = find_col(df_cl_clean, ['head','tete','movement','mouvement'])
COL_GAZE        = find_col(df_cl_clean, ['gaze','regard','yeux','oculo'])
COL_INTERACT    = find_col(df_cl_clean, ['interact','revision','answer','event'])
COL_RESP_TIME   = find_col(df_cl_clean, ['response_time','temps','latency','time','duree'])
COL_SCORE       = find_col(df_cl_clean, ['score','grade','note','result','correct'])
COL_ERROR_TYPE  = find_col(df_cl_clean, ['error','erreur','type','dominant'])
print(f'\nCognitive_Load → head={COL_HEAD_MV} | gaze={COL_GAZE}')

COL_F2_LEARNER  = find_col(df_form2_clean, ['learner_id','learner','id','email'])
COL_F2_EXERCISE = find_col(df_form2_clean, ['exercise_id','exercise','exercice'])
                                                                                            
                                                                        


In [ ]:
import re

from sklearn.preprocessing import MinMaxScaler

def minmax_normalize_cols(df, cols, suffix='_norm'):
    cols_valid = [c for c in cols if c and c in df.columns]
    if not cols_valid: return df, None, []
    for c in cols_valid:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    scaler = MinMaxScaler()
    normalized = scaler.fit_transform(df[cols_valid].fillna(df[cols_valid].median()))
    for i, col in enumerate(cols_valid):
        df[col+suffix] = normalized[:,i].round(4)
    return df, scaler, cols_valid

cl_signal_cols = [c for c in [COL_HEAD_MV,COL_GAZE,COL_INTERACT,COL_RESP_TIME] if c]
if cl_signal_cols and not df_cl_clean.empty:
    df_cl_clean, scaler_cl, norm_cols = minmax_normalize_cols(df_cl_clean, cl_signal_cols)
    COL_HEAD_NORM     = COL_HEAD_MV   + '_norm' if COL_HEAD_MV   else None
    COL_GAZE_NORM     = COL_GAZE      + '_norm' if COL_GAZE      else None
    COL_INTERACT_NORM = COL_INTERACT  + '_norm' if COL_INTERACT  else None
    COL_RT_NORM       = COL_RESP_TIME + '_norm' if COL_RESP_TIME else None
    print(f'✅ Signaux VR normalisés : {norm_cols}')
else:
    COL_HEAD_NORM = COL_GAZE_NORM = COL_INTERACT_NORM = COL_RT_NORM = None
    print('⚠️  Cognitive_Load vide — signaux VR non disponibles')

if COL_SCORE and COL_SCORE in df_cl_clean.columns:
    df_cl_clean[COL_SCORE] = pd.to_numeric(df_cl_clean[COL_SCORE], errors='coerce')
    s_min, s_max = df_cl_clean[COL_SCORE].min(), df_cl_clean[COL_SCORE].max()
    df_cl_clean['score_norm'] = ((df_cl_clean[COL_SCORE]-s_min)/(s_max-s_min)).round(4) if s_max>1.0 else df_cl_clean[COL_SCORE]
    COL_SCORE_NORM = 'score_norm'
else:
    COL_SCORE_NORM = None


CLQ10_VERSIONS = {
    'pilot_base_v1': [                                    
        ('Q1',  'How mentally demanding was the exercise?',                         False),
        ('Q2',  'How physically demanding was the exercise?',                       False),
        ('Q3',  'How difficult was the pace of the exercise?',                      False),
        ('Q4',  'How difficult was it to accomplish what was asked?',               False),
        ('Q5',  'How hard did you have to work (effort) to complete the exercise?', False),
        ('Q6',  'I found the instructions clear',                                   True),
        ('Q7',  'I was confident in my answer',                                     True),
        ('Q8',  'I hesitated during the exercise',                                  False),
        ('Q9',  'I felt confused during the exercise',                              False),
        ('Q10', 'How frustrated were you during the exercise?',                     False),
    ],
    'online_form_v2': [
        ('Q1',  'This exercise required a high level of mental effort.',            False),
        ('Q2',  'I found this exercise difficult.',                                 False),
        ('Q3',  'I felt hesitant or unsure while solving this exercise.',           False),
        ('Q4',  'I had to reread the question or instructions several times.',      False),
        ('Q5',  'I felt under time pressure while solving this exercise.',          False),
        ('Q6',  'The instructions were clear.',                                     True),
        ('Q7',  'I felt confident about my answer.',                                True),
        ('Q8',  'I felt frustrated while working on this exercise.',                False),
        ('Q9',  'I had difficulty identifying what was being asked.',               False),
        ('Q10', 'I felt that the amount of information was overwhelming.',          False),
    ],
}
CLQ10_ANALYSIS_VERSION = 'pilot_base_v1'                                                         
CLQ10_ITEMS = CLQ10_VERSIONS[CLQ10_ANALYSIS_VERSION]
for _v, _its in CLQ10_VERSIONS.items():
    assert [q for q, _, _ in _its] == [f'Q{i}' for i in range(1, 11)], _v
    assert {q for q, _, r in _its if r} == {'Q6', 'Q7'}, f'{_v} : inversion ≠ {{Q6, Q7}}'
CLQ10_IDS      = [q for q, _, _ in CLQ10_ITEMS]
CLQ10_REVERSED = {q for q, _, rev in CLQ10_ITEMS if rev}                           
CLQ10_COLS     = [normalize_col_name(t) for _, t, _ in CLQ10_ITEMS]                                             
CLQ10_COLS_BY_VERSION = {v: [normalize_col_name(t) for _, t, _ in its] for v, its in CLQ10_VERSIONS.items()}
assert not set.intersection(*[set(c) for c in CLQ10_COLS_BY_VERSION.values()]), 'intitulés ambigus entre versions'
CLQ10_RAW_COLS = [f'{q}_raw' for q in CLQ10_IDS]                                                                
LIKERT_MIN, LIKERT_MAX = 1, 7

def _likert7(v):
    """Entier 1..7 ou None. Refuse vide/NaN, texte, décimaux (3.5) et hors-borne (0, 8…).
    Volontairement PAS de parse_likert() : ses libellés textuels sont sur 1–5, pas 1–7."""
    if v is None: return None
    txt = str(v).strip()
                                                                                      
                                                                                        
    m = re.match(r'^([1-7])\s*(?:[—–\-:)]|\s)', txt)
    if m:
        return int(m.group(1))
    try:
        x = float(txt)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(x) or x != int(x) or not (LIKERT_MIN <= x <= LIKERT_MAX):
        return None
    return int(x)

def validate_clq10_answers(answers):
    """answers : 10 valeurs dans l'ordre Q1..Q10, ou dict {'Q1':…,'Q10':…}.
    Retourne (ok, vecteur_de_10_entiers | None, raison)."""
    if isinstance(answers, dict):
        answers = [answers.get(q) for q in CLQ10_IDS]
    try:
        answers = list(answers)
    except TypeError:
        return False, None, 'réponses non itérables'
    if len(answers) != len(CLQ10_IDS):
        return False, None, f'{len(answers)} réponses reçues, {len(CLQ10_IDS)} attendues'
    vec = [_likert7(v) for v in answers]
    bad = [CLQ10_IDS[i] for i, v in enumerate(vec) if v is None]
    if bad:
        return False, None, f'réponses manquantes ou invalides (hors 1–7) : {bad}'
    return True, vec, ''

def compute_clq10(answers):
    """CL_q ∈ [0,1] à partir des DIX réponses brutes (1–7), ordre Q1..Q10.
    Recodage 8−s pour Q6 et Q7, normalisation (s−1)/6, moyenne des 10 items.
    Retourne np.nan si le questionnaire est incomplet ou invalide (jamais d'imputation)."""
    ok, vec, _ = validate_clq10_answers(answers)
    if not ok: return np.nan
    z = [((LIKERT_MAX + LIKERT_MIN - s) if q in CLQ10_REVERSED else s) for q, s in zip(CLQ10_IDS, vec)]
    return float(np.mean([(s - 1) / 6. for s in z]))

def detect_clq10_version(row):
    """Version du questionnaire d'une ligne, déterminée UNIQUEMENT par correspondance exacte des intitulés
    normalisés (jamais par position). Q1_raw…Q10_raw (onglet Cognitive_Load) = version pilote.
    Retourne None si ambigu (égalité) ou si aucun intitulé connu n'est présent."""
    keys = {normalize_col_name(k) for k in dict(row)}
    if all(normalize_col_name(r) in keys for r in CLQ10_RAW_COLS): return CLQ10_ANALYSIS_VERSION
    score = {v: sum(c in keys for c in cols) for v, cols in CLQ10_COLS_BY_VERSION.items()}
    best = max(score.values())
    winners = [v for v, n in score.items() if n == best]
    return winners[0] if best > 0 and len(winners) == 1 else None

def extract_clq10_answers(row, version=None):
    """10 réponses brutes (ordre Q1..Q10 de la version détectée), par correspondance EXACTE de l'intitulé.
    Item absent → None (questionnaire invalide). Aucune réaffectation de colonne, aucun repli positionnel."""
    version = version or detect_clq10_version(row)
    if version is None: return [None] * len(CLQ10_IDS)
    norm = {normalize_col_name(k): v for k, v in dict(row).items()}
    out = []
    for col, raw in zip(CLQ10_COLS_BY_VERSION[version], CLQ10_RAW_COLS):
        out.append(norm.get(col, norm.get(normalize_col_name(raw)) if version == CLQ10_ANALYSIS_VERSION else None))
    return out

def find_ails_xlsx(candidates=None):
    """Localise la base locale, quel que soit son nom : noms connus, puis motif AILS_base_corrigee*.xlsx
    (couvre « AILS_base_corrigee (1)(1).xlsx » et « AILS_base_corrigee__1_.xlsx »)."""
    import os, glob
    names = candidates or ['AILS_base_corrigee.xlsx', 'AILS_base_corrigee (1)(1).xlsx', 'AILS_base_corrigee (1).xlsx',
                           'AILS_base_corrigee__1_.xlsx', 'AILS_Database_corrigee.xlsx']
    for base in ('', '/content/', '/content/drive/MyDrive/', '/mnt/user-data/uploads/'):
        for f in names:
            if os.path.exists(base + f): return base + f
        hits = sorted(glob.glob(base + 'AILS_base_corrigee*.xlsx'))
        if hits: return hits[0]
    return None

def clq10_series(df):
    """CL_q ligne par ligne pour un DataFrame (colonnes normalisées ou Q*_raw)."""
    return df.apply(lambda r: compute_clq10(extract_clq10_answers(r)), axis=1)

if not df_form2.empty:
                                                                                                
                                                                                                        
    df_form2_clq_src = df_form2.copy()
    df_form2_clq_src.columns = [normalize_col_name(c) for c in df_form2_clq_src.columns]
    df_form2_clq_src = df_form2_clq_src.loc[df_form2_clean.index]
    absent = [c for c in CLQ10_COLS if c not in df_form2_clq_src.columns]
    if absent:
        raise KeyError(f'Items CL_q introuvables dans Form 2 (renommage de colonnes ?) : {absent}')
    df_form2_clean['clq_score_norm'] = clq10_series(df_form2_clq_src).round(4)
    df_form2_clean['clq_score']      = (df_form2_clean['clq_score_norm'] * 6 + 1).round(4)                                   
    n_inv = int(df_form2_clean['clq_score_norm'].isna().sum())
    print(f'✅ CL_q (compute_clq10, 10 items) : {CLQ10_COLS}')
    print(f'   inversés : {sorted(CLQ10_REVERSED)} | CL_q moyen = {df_form2_clean["clq_score_norm"].mean():.3f} '
          f'| N = {len(df_form2_clean)} | questionnaires invalides/incomplets = {n_inv}')
else:
    print('⚠️  Form Responses 2 vide — CL_q non calculé')

prof_cols = [c for c in [COL_PRIOR,COL_CONFIDENCE,COL_MOTIVATION] if c]
if prof_cols and not df_profile_clean.empty:
    df_profile_clean, _, _ = minmax_normalize_cols(df_profile_clean, prof_cols)
    print(f'✅ Profil normalisé : {prof_cols}')

## 🏗️ Partie 1 — Entraînement sur Base Synthétique Professionnelle

In [ ]:

import numpy as np
import pandas as pd
import math, re, json
from scipy.stats import truncnorm
from datetime import datetime

np.random.seed(42)

SAMPLING_RATE_HZ            = 90
GAZE_DISPERSION_WINDOW_SEC  = 2
NORMALIZATION_WINDOW_SEC    = 60
ALIGNMENT_LATENCY_MS        = 20
GAZE_HIGH_THRESHOLD         = 0.7
HEAD_LOW_THRESHOLD          = 0.3
VR_FEATURE_AGGREGATION_SEC  = 1

MODULES = {
    'M01': {'title': 'Variables & Types',            'difficulty': 1, 'prerequisites': [],                     'theta': 0.75},
    'M02': {'title': 'Conditionnelles',              'difficulty': 1, 'prerequisites': ['M01'],                'theta': 0.75},
    'M03': {'title': 'Boucles',                      'difficulty': 2, 'prerequisites': ['M01', 'M02'],         'theta': 0.75},
    'M04': {'title': 'Tableaux',                     'difficulty': 2, 'prerequisites': ['M01', 'M03'],         'theta': 0.78},
    'M05': {'title': 'Fonctions',                    'difficulty': 3, 'prerequisites': ['M02', 'M03'],         'theta': 0.78},
    'M06': {'title': 'Récursivité',                  'difficulty': 4, 'prerequisites': ['M05'],                'theta': 0.80},
    'M07': {'title': 'Matrices',                     'difficulty': 3, 'prerequisites': ['M04'],                'theta': 0.78},
    'M08': {'title': 'Logique Formelle',             'difficulty': 3, 'prerequisites': ['M02'],                'theta': 0.80},
    'M09': {'title': 'Complexité & Algorithmes de Tri','difficulty': 4,'prerequisites': ['M03', 'M05'],        'theta': 0.82},
    'M10': {'title': 'Graphes',                      'difficulty': 5, 'prerequisites': ['M06', 'M07', 'M09'], 'theta': 0.82},
}
MODULE_ORDER = ['M01','M02','M03','M04','M05','M06','M07','M08','M09','M10']

ALPHA, BETA, GAMMA, DELTA = 0.25, 0.25, 0.30, 0.20

N_WINDOW            = 3
EXP_MASTERY_WINDOW  = 3
EXP_MASTERY_SCORE   = 0.70
EXP_MASTERY_CL_MAX  = 0.50
EXP_MAX_EXERCISES   = 15

                                                                             
                                                                       
POPULATION_CF = {
    'Novice':        {'n': 34, 'prior': (0.22, 0.12), 'conf': (0.25, 0.13), 'motiv': (0.48, 0.14),
                      'h_base': (0.68, 0.08), 'g_base': (0.60, 0.08), 'e_base': (0.52, 0.08), 'r_base': (0.58, 0.08)},
    'Intermédiaire': {'n': 33, 'prior': (0.52, 0.12), 'conf': (0.53, 0.13), 'motiv': (0.68, 0.13),
                      'h_base': (0.38, 0.08), 'g_base': (0.32, 0.08), 'e_base': (0.28, 0.08), 'r_base': (0.34, 0.08)},
    'Avancé':        {'n': 33, 'prior': (0.76, 0.10), 'conf': (0.77, 0.10), 'motiv': (0.81, 0.10),
                      'h_base': (0.16, 0.08), 'g_base': (0.12, 0.08), 'e_base': (0.10, 0.08), 'r_base': (0.15, 0.08)},
}
                                                                        
                                                                     
POPULATION_SYN = {
    'G1_Novice':       {'n': 10, 'label': 'G1 – Novice',
                        'prior': (0.20, 0.10), 'conf': (0.22, 0.10), 'motiv': (0.45, 0.12),
                        'h_base': (0.70, 0.08), 'g_base': (0.62, 0.08), 'e_base': (0.55, 0.08), 'r_base': (0.60, 0.08)},
    'G2_Intermediate': {'n': 10, 'label': 'G2 – Intermediate',
                        'prior': (0.52, 0.12), 'conf': (0.53, 0.13), 'motiv': (0.68, 0.13),
                        'h_base': (0.38, 0.08), 'g_base': (0.32, 0.08), 'e_base': (0.28, 0.08), 'r_base': (0.34, 0.08)},
    'G3_Advanced':     {'n': 10, 'label': 'G3 – Advanced',
                        'prior': (0.76, 0.10), 'conf': (0.77, 0.10), 'motiv': (0.81, 0.10),
                        'h_base': (0.16, 0.08), 'g_base': (0.12, 0.08), 'e_base': (0.10, 0.08), 'r_base': (0.15, 0.08)},
}
POPULATION = POPULATION_CF

def classify_cl(cl_value: float) -> tuple:
    """
    Classifie l'état cognitif selon Table 4 de l'article.
    ✅ Seuils : 0.30 / 0.60 / 0.80 (article Section 3.2)
    Returns: (state: str, action: str)
    """
    cl = float(cl_value)
    cl = max(0.0, min(1.0, cl))
    if cl < 0.30:
        return ('Confidence',  '✨ Excellent ! Exercice plus difficile (d+1).')
    elif cl < 0.60:
        return ('Exploration', '👍 Bonne progression ! Exercice standard.')
    elif cl < 0.80:
        return ('Hésitation',  '⚠️  Difficultés — 2 indices activés.')
    else:
        return ('Confusion',   '🆘 Surcharge — exercice guidé + alerte enseignant.')

def estimate_cl(h: float, g: float, e: float, r: float):
    """
    Formule CL Eq.(1) : CL = α·ĥ + β·ĝ + γ·ê + δ·r̂
    ✅ α=0.25, β=0.25, γ=0.30, δ=0.20 (article Section 3.2)

    Retourne (cl_value: float, cl_state: str, cl_action: str)
    NOTE : compatibilité avec l'ancien usage estimate_cl()[0] préservée.
    """
    cl = round(ALPHA*float(h) + BETA*float(g) + GAMMA*float(e) + DELTA*float(r), 4)
    cl = max(0.0, min(1.0, cl))
    state, action = classify_cl(cl)
    return (cl, state, action)

import shap as _shap
from sklearn.ensemble import GradientBoostingRegressor as _GradientBoostingRegressor

def _generate_shap_background_dataset(n: int = 100, seed: int = 42):
    """
    Génère un dataset d'arrière-plan synthétique réaliste pour SHAP.
    ✅ AJOUTÉ : N=100 lignes x 4 colonnes (ĥ, ĝ, ê, r̂) simulant l'historique
    d'interaction d'apprenants, avec une cible y = score de performance /
    adaptation, corrélé négativement à la charge cognitive (cohérent avec Eq.(1)).
    """
    rng = np.random.RandomState(seed)
    h = rng.beta(2, 2, n)
    g = rng.beta(2, 2, n)
    e = rng.beta(2, 2, n)
    r = rng.beta(2, 2, n)
    X = np.column_stack([h, g, e, r])
    cl_proxy = ALPHA * h + BETA * g + GAMMA * e + DELTA * r
    noise = rng.normal(0, 0.05, n)
    y = np.clip(1.0 - cl_proxy + noise, 0.0, 1.0)
    return X, y

_SHAP_BG_X, _SHAP_BG_Y = _generate_shap_background_dataset(n=100, seed=42)

_SHAP_BG_MODEL = _GradientBoostingRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42,
)
_SHAP_BG_MODEL.fit(_SHAP_BG_X, _SHAP_BG_Y)

_SHAP_BG_SUMMARY = _shap.kmeans(_SHAP_BG_X, min(20, len(_SHAP_BG_X)))
_SHAP_BG_EXPLAINER = _shap.KernelExplainer(_SHAP_BG_MODEL.predict, _SHAP_BG_SUMMARY)

_SHAP_CHANNEL_ORDER = ['head_movement', 'gaze_direction', 'interaction_events', 'response_time']

def compute_shap(h: float, g: float, e: float, r: float) -> dict:
    """
    Calcule les vraies valeurs SHAP (Eq.9 article) via shap.KernelExplainer
    appliqué à un GradientBoostingRegressor entraîné sur le dataset
    d'arrière-plan synthétique _SHAP_BG_X/_SHAP_BG_Y (N=100, colonnes ĥ,ĝ,ê,r̂).
    ✅ RÉÉCRITE : remplace l'ancienne formule triviale φk = wk·x̂k par un vrai
    calcul de Shapley (KernelExplainer + shap.kmeans comme résumé d'arrière-plan
    pour borner le temps de calcul), en conservant le même format de sortie
    (mêmes clés, valeurs arrondies à 4 décimales).
    """
    x = np.array([[float(h), float(g), float(e), float(r)]])
    phi = _SHAP_BG_EXPLAINER.shap_values(x)[0]
    return {channel: round(float(val), 4) for channel, val in zip(_SHAP_CHANNEL_ORDER, phi)}

def cl_from_form2(answers) -> float:
    """CL_q à partir des 10 réponses brutes (1–7, ordre Q1..Q10) — DÉLÈGUE à compute_clq10
    (cellule 7) : recodage 8−s de Q6/Q7 puis (s−1)/6. Retourne np.nan si invalide."""
    return compute_clq10(answers)

def cl_fused(cl_beh: float, cl_tlx: float,
             w_beh: float = 0.60, w_tlx: float = 0.40) -> float:
    """
    Fusion CL comportemental (VR) + CL déclaratif (Form 2 / TLX).
    ✅ Eq. implémentée : CL_fused = 0.60·CL_beh + 0.40·TLX_norm
    Quand CL_beh indisponible (pas de VR) → retourne TLX_norm directement.
    """
    if cl_beh is None:
        return cl_tlx
    return round(w_beh * cl_beh + w_tlx * cl_tlx, 4)

def score_evolution(s_prev: float, cl_current: float,
                    eta: float = 0.18,
                    xi: float = None) -> float:
    """
    Mise à jour score conforme article Section 4.2 :
    S^(t+1) = S^(t) + η · (1 - S^(t)) · (1 - CL_t) · ξ
    η=0.18 (learning rate)
    ξ ~ U(0.8, 1.2) (bruit individuel)
    ✅ AJOUTÉE — manquait dans la version précédente
    """
    if xi is None:
        xi = np.random.uniform(0.8, 1.2)
    delta = eta * (1.0 - s_prev) * (1.0 - cl_current) * xi
    return round(min(1.0, s_prev + delta), 4)

def simulate_learner(group_params: dict, n_modules: int = 10,
                     pop_label: str = 'G1') -> dict:
    """
    Génère un apprenant synthétique avec score evolution conforme article.
    ✅ seed=42 global, η=0.18, ξ~U(0.8,1.2), cap=15 exercices/module
    """
    p = group_params
    prior      = float(np.clip(np.random.normal(*p['prior']),  0.01, 0.99))
    confidence = float(np.clip(np.random.normal(*p['conf']),   0.01, 0.99))
    motivation = float(np.clip(np.random.normal(p.get('motiv', (.5,.1))[0],
                                                p.get('motiv', (.5,.1))[1]), 0.01, 0.99))

    h0 = float(np.clip(np.random.normal(*p['h_base']), 0.01, 0.99))
    g0 = float(np.clip(np.random.normal(*p['g_base']), 0.01, 0.99))
    e0 = float(np.clip(np.random.normal(*p['e_base']), 0.01, 0.99))
    r0 = float(np.clip(np.random.normal(*p['r_base']), 0.01, 0.99))

    cl0, _, _ = estimate_cl(h0, g0, e0, r0)

    tlx_label = float(np.clip(cl0 + np.random.normal(0, 0.08), 0, 1))

    mastered = []
    module_progress = {}
    score_current = prior

    for mid in MODULE_ORDER[:n_modules]:
        prereqs = MODULES[mid]['prerequisites']
        if not set(prereqs).issubset(set(mastered)):
            continue

        score_t = score_current
        n_ex    = 0

        while n_ex < EXP_MAX_EXERCISES:

            state, _ = classify_cl(cl0)

            score_t = score_evolution(score_t, cl0, eta=0.18)

            cl0 = max(0.05, cl0 * (0.95 + np.random.normal(0, 0.02)))
            n_ex += 1

            if n_ex >= N_WINDOW and score_t >= MODULES[mid]['theta'] and cl0 <= EXP_MASTERY_CL_MAX:
                mastered.append(mid)
                break

        module_progress[mid] = {
            'score_pre':     round(score_current, 3),
            'score_post':    round(score_t, 3),
            'score_current': round(score_t, 3),
            'mean_cl':       round(cl0, 3),
            'is_mastered':   mid in mastered,
            'nb_exercises':  n_ex,
        }
        score_current = score_t

    return {
        'group':             pop_label,
        'prior_knowledge':   round(prior, 4),
        'confidence':        round(confidence, 4),
        'motivation':        round(motivation, 4),
        'mean_head_movement':round(h0, 4),
        'mean_gaze_disp':    round(g0, 4),
        'mean_interaction_ev':round(e0, 4),
        'mean_response_time':round(r0, 4),
        'mean_cl':           round(cl0, 4),
        'mean_score':        round(score_current, 4),
        'tlx_label':         round(tlx_label, 4),
        'mastered_modules':  mastered,
        'module_progress':   module_progress,
    }

def get_fusion_weights(n_interactions: int, cl_value: float) -> dict:
    """
    Scheduling dynamique des poids conforme à l'article Eq.(4)(5) :
      Cold-start   (n < 3)    : w1=0.70, w3=0.30
      CL-critical  (CL ≥ 0.60): w1=0.20, w3=0.80
      Intra-rich   (sinon)    : w1=0.40, w3=0.60
    ✅ AJOUTÉE — le code précédent utilisait des poids fixes (0.60/0.40)
    """
    if n_interactions < 3:
        return {'w1': 0.70, 'w3': 0.30, 'w2': 0.60, 'w4': 0.40,
                'schedule': 'cold-start'}
    elif cl_value >= 0.60:

        return {'w1': 0.20, 'w3': 0.80, 'w2': 0.40, 'w4': 0.60,
                'schedule': 'CL-critical'}
    else:
        return {'w1': 0.40, 'w3': 0.60, 'w2': 0.60, 'w4': 0.40,
                'schedule': 'intra-rich'}

def score_exercise_fusion(s_cb: float, cl_value: float,
                          n_interactions: int = 5) -> float:
    """
    ⚠️ EXPÉRIMENTAL / SIMULATION — cette fonction n'est PAS appelée dans le parcours de
    recommandation de l'expérimentation : l'exercice est généré par RAG (generate_exercise_rag),
    seuls les poids get_fusion_weights() influencent la consigne. Ne pas présenter comme validée.
    Score exercice Eq.(4) : Score_Ex = w1·S_CB + w3·(1-CL)
    Avec scheduling dynamique selon n_interactions.
    ✅ CORRIGÉ — utilise get_fusion_weights() au lieu de poids fixes
    """
    w = get_fusion_weights(n_interactions, cl_value)
    return round(w['w1'] * s_cb + w['w3'] * (1.0 - cl_value), 4)

def score_module_fusion(vote_cf: float, compat: float,
                        cl_value: float, n_interactions: int = 5) -> float:
    """
    Score module Eq.(5) : Score_Mod = w2·VoteCF + w4·Compat
    ✅ CORRIGÉ — utilise scheduling dynamique
    """
    w = get_fusion_weights(n_interactions, cl_value)
    return round(w['w2'] * vote_cf + w['w4'] * compat, 4)


                                                                                                  
COLD_START_THRESH_HIGH, COLD_START_THRESH_MID = 0.70, 0.40
def cold_start_state(prior: float):
    """prior ≥ 0,70 → M01–M03 maîtrisés, départ M04 | 0,40 ≤ prior < 0,70 → M01 maîtrisé, départ M02
       | prior < 0,40 → aucun module maîtrisé, départ M01.  Retourne (module_de_départ, [modules_maîtrisés])."""
    p = float(prior)
    if p >= COLD_START_THRESH_HIGH: return 'M04', ['M01', 'M02', 'M03']
    if p >= COLD_START_THRESH_MID:  return 'M02', ['M01']
    return 'M01', []

CURRICULUM_BLACKLIST = [                                                                        
    'pointeur', 'pointer', 'malloc', 'free', 'new ', 'delete ',
    'template', 'héritage', 'heritage', 'polymorphisme', 'surcharge',
    'lambda', 'thread', 'mutex', 'socket', 'arbre avl', 'avl tree',
    'graphe pondéré', 'automate', 'regex', 'sql', 'api rest',
]
REQUIRED_KEYS = {
    'titre', 'contexte', 'questions', 'indices',
    'conseil_pedagogique', 'difficulte_reelle', 'duree_estimee_minutes'
}

def verify_exercise(exercise: dict, expected_difficulty: int = 2, module_id: str = '') -> dict:
    """
    Verification Engine conforme Section 3.3 article.
    Barème de pénalités unifié avec _verify_exercise (Cellule 25) :
      -0.30 clé manquante | -0.40 questions vides | -0.10/question incomplète
      -0.15 barème incorrect (+0.10 si auto-corrigé) | -0.25×min(2,n) concepts interdits
      -0.10 incohérence difficulté
    κ ≥ 0.70 → livraison | [0.60, 0.70) → retry | < 0.60 → fallback+flag
    """
    score = 1.0
    issues = []
    fixed = False

    missing_keys = REQUIRED_KEYS - set(exercise.keys())
    if missing_keys:
        issues.append(f'Clés manquantes : {missing_keys}')
        score -= 0.30

    questions = exercise.get('questions', [])
    if not isinstance(questions, list) or len(questions) == 0:
        issues.append('Aucune question trouvée.')
        score -= 0.40
    else:
        for i, q in enumerate(questions):
            missing_q = {'enonce', 'points'} - set(q.keys())
            if missing_q:
                issues.append(f'Q{i+1} clés manquantes : {missing_q}')
                score -= 0.10

    try:
        total_pts = sum(int(q.get('points', 0)) for q in questions) if questions else 0
        if total_pts != 10:
            score -= 0.15
            if 8 <= total_pts <= 12 and len(questions) >= 3:
                questions[0]['points'] = 3
                questions[1]['points'] = 4
                questions[2]['points'] = 3
                exercise['questions'] = questions
                issues.append(f'Barème auto-corrigé ({total_pts}→10).')
                score += 0.10
                fixed = True
            else:
                issues.append(f'Barème incorrect : {total_pts}/10.')
    except Exception:
        issues.append('Impossible de calculer le barème.')

    full_text  = json.dumps(exercise, ensure_ascii=False).lower()
    violations = [t for t in CURRICULUM_BLACKLIST if t in full_text]
    if violations:
        issues.append(f'Concepts hors programme : {violations}')

    kappa = 1.0
    if missing_keys:   kappa -= 0.30 * len(missing_keys)
    if violations:     kappa -= 0.25 * len(violations)
    if any('Barème incorrect' in i and 'auto-corrigé' not in i for i in issues):
        kappa -= 0.20
    elif any('auto-corrigé' in i for i in issues):
        kappa -= 0.05
    kappa = round(max(0.0, min(1.0, kappa)), 3)

    return {
        'kappa':       kappa,
        'passed':      kappa >= 0.70 and not missing_keys and not violations,
        'needs_retry': 0.60 <= kappa < 0.70,
        'fallback':    kappa < 0.60,
        'issues':      issues,
        'fixed':       fixed,
    }

def generate_population(pop_config: dict, id_offset: int = 0,
                        is_cf_pretrain: bool = False) -> list:
    """
    Génère une liste d'apprenants synthétiques.
    pop_config : POPULATION_CF (N=100) ou POPULATION_SYN (N=30)
    id_offset  : pour éviter les conflits d'ID entre les deux populations
    """
    learners = []
    uid = id_offset
    for group_name, params in pop_config.items():
        label = params.get('label', group_name)
        for _ in range(params['n']):
            l = simulate_learner(params, n_modules=len(MODULES), pop_label=label)
            l['id']           = uid
            l['username']     = f'{"syn_cf" if is_cf_pretrain else "syn"}_learner_{uid:03d}'
            l['is_synthetic'] = True
            learners.append(l)
            uid += 1
    return learners

print('🏗️  Génération population synthétique...')

SYN_LEARNERS_CF = generate_population(POPULATION_CF, id_offset=0,
                                       is_cf_pretrain=True)
print(f'  ✅ CF pre-train : {len(SYN_LEARNERS_CF)} apprenants (N=100)')

SYN_LEARNERS_30 = generate_population(POPULATION_SYN, id_offset=1000,
                                       is_cf_pretrain=False)
print(f'  ✅ Validation   : {len(SYN_LEARNERS_30)} apprenants (N=30, n=10/groupe)')

SYN_LEARNERS = SYN_LEARNERS_CF

CL_TO_DIFF = {
    'Confidence':  3,
    'Exploration': 2,
    'Hésitation':  2,
    'Confusion':   1,
}

CL_ADAPTIVE_MESSAGES = {
    'Confidence':  '✨ Excellent ! Niveau de maîtrise élevé — exercice plus difficile.',
    'Exploration': '👍 Bonne progression ! Continuez au niveau standard.',
    'Hésitation':  '⚠️  Des difficultés détectées — 2 indices activés.',
    'Confusion':   '🆘 Reprenons depuis le début avec un exercice très guidé.\n   Consultez le tutoriel et demandez de l\'aide.',
}

print('\n── Tests de cohérence ──')

for val, expected in [(0.20,'Confidence'),(0.45,'Exploration'),
                       (0.70,'Hésitation'),(0.85,'Confusion')]:
    state, _ = classify_cl(val)
    mark = '✅' if state == expected else '❌'
    print(f'  {mark} classify_cl({val}) = {state}')

cl_val, state, _ = estimate_cl(0.6, 0.5, 0.7, 0.4)
print(f'  ✅ estimate_cl(h=.6,g=.5,e=.7,r=.4) = {cl_val:.3f} [{state}]')

shap = compute_shap(0.6, 0.5, 0.7, 0.4)
print(f'  ✅ SHAP φê={shap["interaction_events"]:.3f} (dominant comme article Table 10)')

s_new = score_evolution(0.50, 0.45, eta=0.18)
print(f'  ✅ score_evolution(0.50, CL=0.45, η=0.18) = {s_new:.3f}')

for n, cl, expected in [(1, 0.5,'cold-start'),(5, 0.7,'CL-critical'),(5, 0.4,'intra-rich')]:
    w = get_fusion_weights(n, cl)
    mark = '✅' if w['schedule'] == expected else '❌'
    print(f'  {mark} fusion_weights(n={n},CL={cl}) → {w["schedule"]}')

ex_ok = {'titre':'T','contexte':'C','questions':[
    {'numero':1,'enonce':'Q1','points':3,'type':'trace'},
    {'numero':2,'enonce':'Q2','points':4,'type':'redaction'},
    {'numero':3,'enonce':'Q3','points':3,'type':'analyse'},
], 'indices':['i1'],'conseil_pedagogique':'cp',
   'difficulte_reelle':2,'duree_estimee_minutes':15}
v = verify_exercise(ex_ok)
print(f'  ✅ verify_exercise(conforme) → κ={v["kappa"]:.2f}, passed={v["passed"]}')

print(f'\n✅ Cellule 8 chargée — {len(MODULES)} modules | {len(SYN_LEARNERS_CF)} syn_CF | {len(SYN_LEARNERS_30)} syn_N30')
print(f'   EXP_MASTERY_CL_MAX={EXP_MASTERY_CL_MAX} | EXP_MAX_EXERCISES={EXP_MAX_EXERCISES}')

In [ ]:

import math
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

PROFILE_KEYS = [
    'prior_knowledge', 'confidence', 'motivation',
    'mean_response_time', 'mean_gaze_disp', 'mean_head_movement',
    'mean_interaction_ev', 'mean_score', 'mean_cl',
]

def build_profile_vector(l: dict) -> np.ndarray:
    vec = [float(l.get(k, 0.5)) for k in PROFILE_KEYS]
    vec.append(len(l.get('mastered_modules', [])) / len(MODULES))
    return np.array(vec)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    dot  = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm) if norm > 0 else 0.0

def get_top_k_peers(learner: dict, all_learners: list, k: int = 10) -> list:
    """Sélectionne les K pairs les plus similaires par cosine similarity."""
    tv   = build_profile_vector(learner)
    sims = [(p, round(cosine_sim(tv, build_profile_vector(p)), 4))
            for p in all_learners if str(p['id']) != str(learner['id'])]
    return sorted(sims, key=lambda x: x[1], reverse=True)[:k]

                                                                           
def validate_dag(modules=MODULES) -> list:
    """Tri topologique (Kahn). Lève ValueError si cycle ou prérequis inconnu."""
    indeg    = {m: 0 for m in modules}
    children = {m: [] for m in modules}
    for m, info in modules.items():
        for p in info['prerequisites']:
            if p not in modules:
                raise ValueError(f"Prérequis inconnu : {p} (dans {m})")
            indeg[m] += 1
            children[p].append(m)
    queue = [m for m, d in indeg.items() if d == 0]
    order = []
    while queue:
        n = queue.pop(0)
        order.append(n)
        for c in children[n]:
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    if len(order) != len(modules):
        raise ValueError("Cycle détecté dans le graphe des prérequis")
    return order

TOPO_ORDER   = validate_dag()
MODULE_DEPTH = {}
for _m in TOPO_ORDER:
    _pre = MODULES[_m]['prerequisites']
    MODULE_DEPTH[_m] = 0 if not _pre else 1 + max(MODULE_DEPTH[p] for p in _pre)
MAX_DEPTH = max(MODULE_DEPTH.values()) or 1
MAX_PRE   = max(len(m['prerequisites']) for m in MODULES.values()) or 1

def missing_prerequisites(mid: str, mastered_set: set) -> list:
    """Prérequis non encore maîtrisés pour accéder à `mid`."""
    return [p for p in MODULES[mid]['prerequisites'] if p not in mastered_set]

def get_candidate_modules(mastered_set: set, return_blocked: bool = False):
    """
    Filtrage des candidats via le DAG (Eq.algo ligne 19) : module non maîtrisé dont
    tous les prérequis sont couverts. return_blocked=True renvoie aussi
    {module: [prérequis manquants]}.
    """
    mastered_set = set(mastered_set)
    candidates, blocked = [], {}
    for mid in MODULES:
        if mid in mastered_set:
            continue
        miss = missing_prerequisites(mid, mastered_set)
        if miss:
            blocked[mid] = miss
        else:
            candidates.append(mid)
    return (candidates, blocked) if return_blocked else candidates

def collaborative_vote(peers: list, mid: str) -> float:
    """
    Vote CF : Σ(sv · ΔSv,mj) — article Eq.(3)
    ✅ Conforme formule article Section 3.4
    """
    total = 0.0
    for p, sim in peers:
        prog = p.get('module_progress', {}).get(mid)
        if prog:
            delta = max(0.0, prog['score_post'] - prog['score_pre'])
            total += sim * delta
    return round(total, 4)

                                                                          
W_NOMINAL, W_DEPTH, W_NPRE = 0.60, 0.25, 0.15                              
W_LEVEL, W_COVER           = 0.60, 0.40                                         
PENALTY_TOO_HARD, PENALTY_TOO_EASY = 1.5, 0.75                       

def intrinsic_difficulty(mid: str) -> float:
    """Difficulté du module dans [0,1] : niveau nominal + profondeur DAG + nb prérequis."""
    m = MODULES[mid]
    d_nom   = (m['difficulty'] - 1) / 4.0
    d_depth = MODULE_DEPTH[mid] / MAX_DEPTH
    d_pre   = len(m['prerequisites']) / MAX_PRE
    return round(W_NOMINAL * d_nom + W_DEPTH * d_depth + W_NPRE * d_pre, 4)

def learner_level(learner: dict) -> float:
    """Niveau courant dans [0,1] : moyenne du niveau initial et du score observé."""
    prior = float(learner.get('prior_knowledge', 0.5))
    score = float(learner.get('mean_score', prior))
    return 0.5 * prior + 0.5 * score

def compat_score(learner: dict, mid: str) -> float:
    """
    Compatibilité niveau/difficulté/prérequis (w4·Compat dans Eq.5), dans [0,1].
    Un module trop difficile est pénalisé plus fort qu'un module trop facile.
    """
    gap = intrinsic_difficulty(mid) - learner_level(learner)
    pen = PENALTY_TOO_HARD if gap > 0 else PENALTY_TOO_EASY
    level_fit = max(0.0, 1.0 - pen * abs(gap))
    pre      = set(MODULES[mid]['prerequisites'])
    mastered = set(learner.get('mastered_modules', []))
    coverage = len(pre & mastered) / len(pre) if pre else 1.0
    return round(W_LEVEL * level_fit + W_COVER * coverage, 4)

def recommend_next_module(learner: dict, all_learners: list,
                          current_cl: float,
                          k: int = 10,
                          n_interactions: int = None) -> dict | None:
    """
    Recommandation inter-module ✅ CORRIGÉE.
    Utilise get_fusion_weights() pour les poids dynamiques Eq.(4)(5).

    n_interactions : nb d'exercices faits dans le module courant
                     (None = auto-détecté depuis module_progress)
    """
    mastered   = set(learner.get('mastered_modules', []))
    candidates = get_candidate_modules(mastered)
    if not candidates:
        return None

    peers = get_top_k_peers(learner, all_learners,
                             k=min(k, len(all_learners)-1))

    if n_interactions is None:
        prog = learner.get('module_progress', {})
        n_interactions = sum(v.get('nb_exercises', 0) for v in prog.values())

    w = get_fusion_weights(n_interactions, current_cl)

    scored = []
    for mid in candidates:
        vote   = collaborative_vote(peers, mid)
        compat = compat_score(learner, mid)

        fscore = score_module_fusion(vote, compat, current_cl, n_interactions)                           
        scored.append({
            'module_id':    mid,
            'title':        MODULES[mid]['title'],
            'difficulty':   MODULES[mid]['difficulty'],
            'vote_cf':      vote,
            'compat_score': compat,
            'final_score':  fscore,
            'schedule':     w['schedule'],
        })

    scored.sort(key=lambda x: x['final_score'], reverse=True)
    best = scored[0]

    return {
        'recommended_module': best['module_id'],
        'recommended_title':  best['title'],
        'final_score':        best['final_score'],
        'vote_cf':            best['vote_cf'],
        'compat_score':       best['compat_score'],
        'schedule':           best['schedule'],
        'top_peers':          [(p['username'], s) for p, s in peers[:5]],
        'mastered_modules':   list(mastered),
        'all_candidates':     scored,
        'fusion_weights':     w,
    }

ERROR_DIRECTIVES = {
    'comprehension': {
        'label':    'COMPRÉHENSION',
        'q1_type':  'définition',
        'q2_type':  'application simple',
        'q3_type':  'comparaison',
        'directive': (
            "L'élève a des difficultés de COMPRÉHENSION. "
            "Q1=définition du concept, Q2=application directe guidée, "
            "Q3=comparer deux approches ou expliquer un code simple."
        ),
    },
    'logique': {
        'label':    'LOGIQUE',
        'q1_type':  'débogage code erroné',
        'q2_type':  'complétion algorithme',
        'q3_type':  "trace d'exécution",
        'directive': (
            "L'élève fait des erreurs de LOGIQUE. "
            "Q1=déboguer un code ERRONÉ fourni, "
            "Q2=compléter un algorithme lacunaire, "
            "Q3=tracer l'exécution pas-à-pas."
        ),
    },
    'syntaxe': {
        'label':    'SYNTAXE',
        'q1_type':  "correction d'erreurs syntaxiques",
        'q2_type':  'réécriture correcte',
        'q3_type':  'écriture depuis zéro',
        'directive': (
            "L'élève a des problèmes de SYNTAXE. "
            "Q1=corriger les erreurs dans un code fourni, "
            "Q2=réécrire proprement, Q3=écrire un algorithme similaire."
        ),
    },
    'calcul': {
        'label':    'CALCUL',
        'q1_type':  'trace numérique',
        'q2_type':  'vérification pas-à-pas',
        'q3_type':  'optimisation',
        'directive': (
            "L'élève fait des erreurs de CALCUL. "
            "Q1=trace numérique complète, "
            "Q2=vérifier un calcul étape par étape, "
            "Q3=identifier l'erreur et corriger."
        ),
    },
    'boundary': {
        'label':    'CAS LIMITE',
        'q1_type':  'définition',
        'q2_type':  'application',
        'q3_type':  'analyse',
        'directive': (
            "Exercice de consolidation — cas limite du module. "
            "Q1=définition du concept-clé, Q2=application standard, "
            "Q3=analyse ou extension vers le module suivant."
        ),
    },

    'general': {
        'label':    'CAS LIMITE',
        'q1_type':  'définition',
        'q2_type':  'application',
        'q3_type':  'analyse',
        'directive': (
            "Exercice de consolidation générale du module. "
            "Q1=réponse courte, Q2=code/algorithme, Q3=analyse/justification."
        ),
    },
}

print('🎯 Validation CF (Leave-One-Out @K) — population N=100')
print('='*65)

hits_at1, hits_at3, ndcg_vals, mrr_vals, valid_count = 0, 0, [], [], 0

for learner in SYN_LEARNERS:
    mastered = learner.get('mastered_modules', [])
    if len(mastered) < 2:
        continue
    true_next   = mastered[-1]
    fake        = {**learner, 'mastered_modules': mastered[:-1]}
    peers_pool  = [l for l in SYN_LEARNERS if l['id'] != learner['id']]
    res         = recommend_next_module(fake, peers_pool, learner['mean_cl'], k=10)
    if not res:
        continue
    valid_count += 1
    ranked = [c['module_id'] for c in res['all_candidates']]
    if true_next in ranked:
        rank = ranked.index(true_next) + 1
        mrr_vals.append(1.0 / rank)
        ndcg_vals.append(1.0 / math.log2(rank + 1))
        if rank == 1: hits_at1 += 1
        if rank <= 3: hits_at3 += 1
    else:
        mrr_vals.append(0.0); ndcg_vals.append(0.0)

CF_METRICS = {
    'Precision@1': round(hits_at1 / valid_count, 4) if valid_count else 0,
    'Precision@3': round(hits_at3 / valid_count, 4) if valid_count else 0,
    'MRR':         round(float(np.mean(mrr_vals)), 4) if mrr_vals else 0,
    'NDCG@3':      round(float(np.mean(ndcg_vals)), 4) if ndcg_vals else 0,
    'N_évalués':   valid_count,
}

print(f'  N évalués    : {CF_METRICS["N_évalués"]}')
print(f'  Precision@1  : {CF_METRICS["Precision@1"]:.4f}  (seuil article ≥ 0.75)')
print(f'  Precision@3  : {CF_METRICS["Precision@3"]:.4f}')
print(f'  MRR          : {CF_METRICS["MRR"]:.4f}')
print(f'  NDCG@3       : {CF_METRICS["NDCG@3"]:.4f}  (seuil article ≥ 0.85)')
print(f'\n✅ Cellule 9 chargée — CF + poids dynamiques + error directives conformes article')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix

cl_beh_vals = np.array([estimate_cl(l['mean_head_movement'],l['mean_gaze_disp'],
                                     l['mean_interaction_ev'],l['mean_response_time'])[0]
                         for l in SYN_LEARNERS])
tlx_vals_syn = np.array([l['tlx_label'] for l in SYN_LEARNERS])

r_syn, p_syn = stats.pearsonr(cl_beh_vals, tlx_vals_syn)
mae_syn      = mean_absolute_error(tlx_vals_syn, cl_beh_vals)
rmse_syn     = np.sqrt(mean_squared_error(tlx_vals_syn, cl_beh_vals))

print('🧠 Validation modèle CL comportemental (N=100 synthétiques)')
print('='*55)
print(f'  Pearson r   = {r_syn:.4f}  (seuil \u2265 0.65 : {"\u2705" if r_syn>=0.65 else "\u26a0\ufe0f"})')
print(f'  p-value     = {p_syn:.2e}')
print(f'  MAE         = {mae_syn:.4f}')
print(f'  RMSE        = {rmse_syn:.4f}')

print('\n  Pearson r par groupe :')
for g in POPULATION:
    idx = [i for i,l in enumerate(SYN_LEARNERS) if l['group']==g]
    r_g,_ = stats.pearsonr(cl_beh_vals[idx], tlx_vals_syn[idx])
    print(f'    {g:<18}: r={r_g:.4f}  {"\u2705" if r_g>=0.65 else "\u26a0\ufe0f"}')

true_states = [classify_cl(t)[0] for t in tlx_vals_syn]
pred_states = [classify_cl(c)[0] for c in cl_beh_vals]
state_acc   = np.mean([t==p for t,p in zip(true_states,pred_states)])
print(f'\n  Accuracy \u00e9tats CL : {state_acc:.4f}')

print('\n  SHAP moyen |\u03c6k| par groupe :')
shap_header = '    {:<18} {:>10} {:>10} {:>10} {:>10}'.format('Groupe','T\u00eate','Regard','Interact.','T.R\u00e9ponse')
print(shap_header)
for g in POPULATION:
    sub = [l for l in SYN_LEARNERS if l['group']==g]
    shaps = [compute_shap(l['mean_head_movement'],l['mean_gaze_disp'],
                          l['mean_interaction_ev'],l['mean_response_time']) for l in sub]
    means = {k: np.mean([abs(s[k]) for s in shaps]) for k in shaps[0]}
    print(f"    {g:<18} {means['head_movement']:>10.4f} {means['gaze_direction']:>10.4f} "          f"{means['interaction_events']:>10.4f} {means['response_time']:>10.4f}")

CL_METRICS_SYN = {
    'pearson_r':   round(r_syn,4), 'p_value': round(p_syn,4),
    'MAE':         round(mae_syn,4), 'RMSE': round(rmse_syn,4),
    'state_accuracy': round(state_acc,4),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
colors_grp = {'Novice':'#e74c3c','Interm\u00e9diaire':'#3498db','Avanc\u00e9':'#27ae60'}
for g in POPULATION:
    idx = [i for i,l in enumerate(SYN_LEARNERS) if l['group']==g]
    ax.scatter(cl_beh_vals[idx], tlx_vals_syn[idx], c=colors_grp[g], label=g, alpha=0.65, s=40)
xs  = np.linspace(0,1,100)
m,b = np.polyfit(cl_beh_vals, tlx_vals_syn, 1)
ax.plot(xs, m*xs+b, 'k--', lw=1.5, label=f'r={r_syn:.3f}')
ax.plot([0,1],[0,1],'gray',lw=0.8,ls=':',label='y=x')
ax.set_title('CL Comportemental vs NASA-TLX', fontweight='bold')
ax.set_xlabel('CL comportemental'); ax.set_ylabel('TLX [0,1]')
ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=.3)

state_labels = ['Confidence','Exploration','H\u00e9sitation','Confusion']
cm  = confusion_matrix(true_states, pred_states, labels=state_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=state_labels, yticklabels=state_labels, annot_kws={'size':9})
axes[1].set_title(f'Matrice de Confusion \u00c9tats CL\n(Acc={state_acc:.3f})', fontweight='bold')
axes[1].set_xlabel('Pr\u00e9dit'); axes[1].set_ylabel('R\u00e9el')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

shap_all = [compute_shap(l['mean_head_movement'],l['mean_gaze_disp'],
                          l['mean_interaction_ev'],l['mean_response_time']) for l in SYN_LEARNERS]
shap_means_g = {k: float(np.mean([abs(s[k]) for s in shap_all])) for k in shap_all[0]}
labels_shap  = ['R\u00e9visions\n(\u00ea)','T.R\u00e9ponse\n(r\u0302)','Regard\n(\u011d)','T\u00eate\n(\u0125)']
vals_shap    = [shap_means_g['interaction_events'], shap_means_g['response_time'],
                shap_means_g['gaze_direction'],     shap_means_g['head_movement']]
bars = axes[2].bar(labels_shap, vals_shap,
                   color=['#e74c3c','#e67e22','#3498db','#9b59b6'], edgecolor='black')
for bar, v in zip(bars, vals_shap):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[2].set_title('SHAP moyen |\u03c6k| par canal', fontweight='bold')
axes[2].set_ylabel('|\u03c6k| moyen'); axes[2].grid(axis='y', alpha=.4)

plt.suptitle('Validation mod\u00e8le CL \u2014 Base synth\u00e9tique (N=100)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('cl_validation_syn.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n\u2705 Mod\u00e8le CL valid\u00e9 \u2014 m\u00e9triques enregistr\u00e9es dans CL_METRICS_SYN')

## 🤝 Partie 2 — Fusion DB + Pipeline AILS Complet

In [ ]:
import shap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler

FEATURE_NAMES = ['score_norm', 'cl_value', 'tendance', 'erreur_norm']

                                   
                                                                            
                                                                       
                                                                             
                                                                            
FEATURE_TO_CHANNEL = {
    'score_norm':  'head_movement',
    'tendance':    'gaze_direction',
    'erreur_norm': 'interaction_events',
    'cl_value':    'response_time',
}

def _build_shap_dataset(learner_db: list):
    """
    Construit X, y depuis l'historique de tous les apprenants.
    Features : [score_norm, cl_value, tendance, erreur_norm]
    Target   : gain de score sur l'exercice suivant (Δ S)
    """
    X_rows, y_rows = [], []
    for l in learner_db:
        history = l.get('exercise_history', [])
        if len(history) < 3:
            continue
        for i in range(2, len(history)):
            window = history[max(0, i - 3):i]
            scores = [r.get('score', 0.5) for r in window]
            score_norm   = float(scores[-1])
            tendance     = float(scores[-1] - scores[0])
            cl_value     = float(l.get('mean_cl', 0.5))
            err_counts   = {}
            for r in window:
                et = r.get('error_type', 'general')
                err_counts[et] = err_counts.get(et, 0) + 1
            erreur_norm = (max(err_counts.values()) if err_counts else 0) / 5.0
            X_rows.append([score_norm, cl_value, tendance, erreur_norm])
            next_score = float(history[i].get('score', score_norm))
            y_rows.append(next_score - score_norm)
    return np.array(X_rows), np.array(y_rows)

def _train_shap_model(learner_db: list):
    """
    Entraîne GradientBoostingRegressor et initialise KernelExplainer.
    Retourne : (model, scaler, explainer) ou (None, None, None) si échec.
    """
    X, y = _build_shap_dataset(learner_db)
    if len(X) < 10:

        print(f'SHAP : données insuffisantes (N={len(X)}) — fallback heuristique')
        return None, None, None

    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    model = GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
    )
    model.fit(X_scaled, y)
    r2 = model.score(X_scaled, y)

    background  = shap.kmeans(X_scaled, min(20, len(X_scaled)))
    explainer   = shap.KernelExplainer(model.predict, background)

    print(f'✅ SHAP KernelExplainer initialisé | N={len(X)} | R²={r2:.3f}')
    return model, scaler, explainer

def _heuristic_phi(score_norm: float, cl_value: float,
                   tendance: float, erreur_norm: float) -> dict:
    return {
        'head_movement':      round(ALPHA * abs(1 - score_norm), 4),
        'gaze_direction':     round(BETA  * max(0.0, -tendance + 0.1), 4),
        'interaction_events': round(GAMMA * erreur_norm, 4),
        'response_time':      round(DELTA * cl_value, 4),
    }


In [ ]:

def safe_get(row, col, default=0.5):
    if col and col in row.index and not pd.isna(row[col]):
        return float(row[col])
    return default

def mean_col(df, col, default=0.5):
    if col and not df.empty and col in df.columns:
        vals = pd.to_numeric(df[col], errors='coerce').dropna()
        return float(vals.mean()) if len(vals) > 0 else default
    return default

def build_real_learner_db():
    """Construit la liste des apprenants depuis les vraies données Google Sheets."""
    DB = []
    id_sources = []
    if not df_form1_clean.empty and 'email_address' in df_form1_clean.columns:
        id_sources = df_form1_clean['email_address'].dropna().unique().tolist()
    elif not df_profile_clean.empty and COL_LEARNER_ID:
        id_sources = df_profile_clean[COL_LEARNER_ID].dropna().unique().tolist()
    elif not df_cl_clean.empty and COL_CL_LEARNER:
        id_sources = df_cl_clean[COL_CL_LEARNER].dropna().unique().tolist()
    elif not df_form2_clean.empty and COL_F2_LEARNER:
        id_sources = df_form2_clean[COL_F2_LEARNER].dropna().unique().tolist()

    for lid in id_sources:
        lid_key = int(lid) if str(lid).replace('.','',1).isdigit() else lid
        form1_row = pd.Series(dtype=object)
        if not df_form1_clean.empty and 'email_address' in df_form1_clean.columns:
            match_f1 = df_form1_clean[df_form1_clean['email_address'] == lid_key]
            if not match_f1.empty: form1_row = match_f1.iloc[0]
        prior      = safe_get(form1_row,'i_already_have_prior_knowledge_in_this_topic', 0.5)
        confidence = safe_get(form1_row,'i_am_confident_in_my_ability_to_solve_exercises_in_this_subject', 0.5)
        motivation = safe_get(form1_row,'i_feel_motivated_when_solving_exercises_in_this_subject', 0.5)
        group      = str(form1_row.get('current_educational_level','Réel'))

        if prior > 1.0: prior = (prior - 1) / 4.
        if confidence > 1.0: confidence = (confidence - 1) / 4.
        if motivation > 1.0: motivation = (motivation - 1) / 4.

        cl_rows = pd.DataFrame()
        if not df_cl_clean.empty and COL_CL_LEARNER:
            cl_rows = df_cl_clean[df_cl_clean[COL_CL_LEARNER] == lid_key]

        mean_h = mean_col(cl_rows, COL_HEAD_NORM)
        mean_g = mean_col(cl_rows, COL_GAZE_NORM)
        mean_e = mean_col(cl_rows, COL_INTERACT_NORM)
        mean_r = mean_col(cl_rows, COL_RT_NORM)

        if cl_rows.empty and not df_form2_clean.empty and 'clq_score_norm' in df_form2_clean.columns and COL_F2_LEARNER:
            f2_rows = df_form2_clean[df_form2_clean[COL_F2_LEARNER] == lid_key]
            mean_cl = mean_col(f2_rows, 'clq_score_norm', 0.5)
        else:
            mean_cl = estimate_cl(mean_h, mean_g, mean_e, mean_r)

        mastered_modules = sorted(set(cold_start_state(prior)[1]))                             

        module_progress = {}
        if not cl_rows.empty and COL_CL_MODULE and COL_SCORE_NORM:
            for mod_id, mod_rows in cl_rows.groupby(COL_CL_MODULE):
                mid = str(mod_id).upper()
                if mid not in MODULES: continue
                scores = pd.to_numeric(mod_rows[COL_SCORE_NORM], errors='coerce').dropna()
                if len(scores) == 0: continue
                cls_m = mod_rows.apply(lambda r: estimate_cl(
                    float(r.get(COL_HEAD_NORM, 0.5)) if COL_HEAD_NORM else 0.5,
                    float(r.get(COL_GAZE_NORM, 0.5)) if COL_GAZE_NORM else 0.5,
                    float(r.get(COL_INTERACT_NORM, 0.5)) if COL_INTERACT_NORM else 0.5,
                    float(r.get(COL_RT_NORM, 0.5)) if COL_RT_NORM else 0.5), axis=1)
                last_s = scores.tolist()[-N_WINDOW:]
                last_c = cls_m.tolist()[-N_WINDOW:]
                is_mastered = (len(last_s)>=N_WINDOW and np.mean(last_s)>=MODULES[mid]['theta']
                               and np.mean(last_c)<=0.50 and last_s[-1]>=last_s[0])
                module_progress[mid] = {
                    'score_pre': round(float(scores.iloc[0]),3),
                    'score_post': round(float(scores.iloc[-1]),3),
                    'score_current': round(float(scores.mean()),3),
                    'mean_cl': round(float(cls_m.mean()),3),
                    'is_mastered': is_mastered, 'nb_exercises': len(mod_rows),
                    'dominant_error': str(mod_rows[COL_ERROR_TYPE].mode().iloc[0]) if COL_ERROR_TYPE and not mod_rows.empty else 'unknown'
                }
                if is_mastered and mid not in mastered_modules:
                    mastered_modules.append(mid)
            mastered_modules = sorted(list(set(mastered_modules)))

        mean_score = (np.mean([v['score_current'] for v in module_progress.values()])
                      if module_progress else round(prior, 3))
        DB.append({
            'id': lid_key, 'username': f'learner_{lid_key}', 'group': group,
            'is_synthetic': False,
            'prior_knowledge': round(prior,4), 'confidence': round(confidence,4),
            'motivation': round(motivation,4),
            'mean_head_movement': mean_h, 'mean_gaze_disp': mean_g,
            'mean_interaction_ev': mean_e, 'mean_response_time': mean_r,
            'mean_score': round(mean_score,3), 'mean_cl': round(mean_cl,4),
            'module_progress': module_progress, 'mastered_modules': mastered_modules,
        })
    return DB

REAL_LEARNERS = build_real_learner_db()
print(f'✅ Apprenants réels : {len(REAL_LEARNERS)}')

DB_LEARNERS = SYN_LEARNERS + REAL_LEARNERS
print(f'✅ DB fusionnée   : {len(DB_LEARNERS)} apprenants ({len(SYN_LEARNERS)} syn + {len(REAL_LEARNERS)} réels)')
print('   → Le CF dispose maintenant d\'une base dense pour des recommandations précises.')

SHAP_MODEL, SHAP_SCALER, SHAP_EXPLAINER = _train_shap_model(DB_LEARNERS)
for l in REAL_LEARNERS:
    cl_state,_ = classify_cl(l['mean_cl'])
    print(f"  {l['id']} | CL={l['mean_cl']:.3f} [{cl_state}] | Maîtrisés={l['mastered_modules']} | Score={l['mean_score']:.3f}")

In [ ]:
import requests, json

import re

def _parse_mistral_json(raw: str):
    """Parser robuste : gère ```json, control chars, JSON partiel."""
    if not raw:
        return None

    cleaned = re.sub(r'```json\s*', '', raw)
    cleaned = re.sub(r'```\s*', '', cleaned)
    cleaned = cleaned.strip()

    m = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if not m:
        return None
    json_str = m.group()

    def fix_control_chars(s):
        result = []
        in_string = False
        escape = False
        for ch in s:
            if escape:
                result.append(ch)
                escape = False
                continue
            if ch == '\\':
                result.append(ch)
                escape = True
                continue
            if ch == '"':
                in_string = not in_string
                result.append(ch)
                continue
            if in_string and ch == '\n':
                result.append('\\n')
                continue
            if in_string and ch == '\r':
                result.append('\\r')
                continue
            if in_string and ch == '\t':
                result.append('\\t')
                continue
            result.append(ch)
        return ''.join(result)

    json_str = fix_control_chars(json_str)

    try:
        return json.loads(json_str)
    except Exception as e:
        print(f"    ⚠️ Parsing échoué après nettoyage : {e}")
        print(f"    Aperçu : {json_str[:300]}")
        return None

def call_mistral(system_prompt, user_prompt, max_tokens=350):

    FALLBACK = (
        '1. PRÉREQUIS : Les modules maîtrisés vous préparent à ce contenu.\n'
        '2. PAIRS SIMILAIRES : Des apprenants comparables ont bien progressé ici.\n'
        '3. DISPONIBILITÉ COGNITIVE : Votre charge cognitive est favorable.'
    )

    if not USE_LLM:
        return FALLBACK

    headers = {
        'Authorization': f'Bearer {MISTRAL_API_KEY}',
        'Content-Type': 'application/json'
    }
    payload = {
        'model': MISTRAL_MODEL,
        'max_tokens': max_tokens,
        'temperature': 0.8,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt}
        ]
    }

    for attempt in range(3):
        try:
            r = requests.post(
                'https://api.mistral.ai/v1/chat/completions',
                headers=headers, json=payload, timeout=30
            )
            if r.status_code == 429:
                wait = (attempt + 1) * 15
                print(f'⚠️ Mistral 429 — attente {wait}s (tentative {attempt+1}/3)')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()['choices'][0]['message']['content'].strip()

        except Exception as ex:
            print(f'⚠️ Mistral erreur tentative {attempt+1}: {ex}')
            if attempt < 2:
                time.sleep(10)

    print('❌ Mistral indisponible — fallback activé')
    return FALLBACK

INTER_SYS = '''
Tu es un tuteur pédagogique IA pour lycéens tunisiens (Bac Informatique).
Justifie la recommandation du prochain module en 3 parties numérotées :
1. PRÉREQUIS : modules déjà maîtrisés qui préparent ce module
2. PAIRS SIMILAIRES : progression mesurée d'apprenants similaires
3. DISPONIBILITÉ COGNITIVE : état cognitif actuel et préparation
Réponds en français, 4-6 phrases, encourageant et précis.
'''

SHAP_LABELS = {
    'head_movement':      'mouvements de tête (ĥ)',
    'gaze_direction':     'direction du regard (ĝ)',
    'interaction_events': 'révisions de réponses (ê)',
    'response_time':      'temps de réponse (r̂)',
}

SHAP_PEDAGOGICAL_MESSAGES = {
    'head_movement': {
        'message':  "Cet exercice cible un concept non encore maîtrisé.",
        'target':   "Niveau de connaissance",
        'channel':  'ĥ',
        'audience': "Apprenant + Enseignant",
    },
    'gaze_direction': {
        'message':  "Tes résultats récents suggèrent un exercice de consolidation.",
        'target':   "Tendance de progression",
        'channel':  'ĝ',
        'audience': "Apprenant + Enseignant",
    },
    'interaction_events': {
        'message':  "Cet exercice se concentre sur ton type d'erreur le plus fréquent.",
        'target':   "Type d'erreur",
        'channel':  'ê',
        'audience': "Apprenant + Enseignant",
    },
    'response_time': {
        'message':  "Un exercice plus simple est proposé pour reprendre confiance.",
        'target':   "Hésitation / État émotionnel",
        'channel':  'r̂',
        'audience': "Apprenant + Enseignant",
    },
}

XAI_DEVELOPER_LOG = []

def _log_xai_event(learner_id, module_id, shap_values, dominant_channel,
                   cl_value, cl_state, pedagogical_msg):
    from datetime import datetime
    XAI_DEVELOPER_LOG.append({
        'timestamp':        datetime.now().isoformat(),
        'learner_id':       learner_id,
        'module_id':        module_id,
        'shap_vector':      shap_values,
        'dominant_channel': dominant_channel,
        'dominant_phi':     shap_values[dominant_channel],
        'cl_value':         round(cl_value, 4),
        'cl_state':         cl_state,
        'pedagogical_msg':  pedagogical_msg,
        'target':           SHAP_PEDAGOGICAL_MESSAGES[dominant_channel]['target'],
    })

def explain_recommendation(learner, result, current_cl, cl_state):
    rec    = MODULES[result['recommended_module']]
    prompt = f'''Apprenant : {learner['username']} (groupe {learner.get('group','?')})
Module recommandé : {result['recommended_module']} — {result['recommended_title']} (difficulté {rec['difficulty']}/5)
Prérequis : {', '.join(rec['prerequisites']) or 'Aucun'}
Modules maîtrisés : {', '.join(result['mastered_modules'])}
Pairs analysés : {len(result['top_peers'])}
Score compatibilité : {result['compat_score']:.2f}/1.0
Vote CF : {result['vote_cf']:.4f}
Score final : {result['final_score']:.4f}
État cognitif : {cl_state} (CL={current_cl:.3f})
'''
    return call_mistral(INTER_SYS, prompt, max_tokens=300)

def translate_shap_to_pedagogy(shap_values, learner_id='?', module_id='?',
                                cl_value=0.5, cl_state='Exploration',
                                verbose=True):

    dominant_channel = max(shap_values, key=lambda k: abs(shap_values[k]))
    dominant_phi     = shap_values[dominant_channel]

    entry          = SHAP_PEDAGOGICAL_MESSAGES[dominant_channel]
    channel_symbol = entry['channel']
    learner_msg    = entry['message']
    teacher_msg    = (
        f"[Enseignant] Canal dominant : {SHAP_LABELS[dominant_channel]} "
        f"| φ={dominant_phi:+.4f} | CL={cl_value:.3f} ({cl_state}) "
        f"| Cible : {entry['target']}"
    )

    _log_xai_event(
        learner_id=learner_id, module_id=module_id,
        shap_values=shap_values, dominant_channel=dominant_channel,
        cl_value=cl_value, cl_state=cl_state, pedagogical_msg=learner_msg,
    )

    if verbose:
        print(f"\n    ┌─ 🔍 XAI — Traduction Pédagogique ({'─'*28})")
        print(f"    │  Canal dominant : {SHAP_LABELS[dominant_channel]}  ({channel_symbol})")
        print(f"    │  φ_{channel_symbol} = {dominant_phi:+.4f}  │  Vecteur SHAP complet :")
        for k, v in shap_values.items():
            marker = ' ◄ dominant' if k == dominant_channel else ''
            print(f"    │    {SHAP_LABELS[k]:<35} φ={v:+.4f}{marker}")
        print(f"    │")
        print(f"    │  💬 Message Apprenant (VR) : « {learner_msg} »")
        print(f"    │  👩‍🏫 Message Enseignant     : {teacher_msg}")
        print(f"    └{'─'*50}")

    return {
        'dominant_channel': dominant_channel,
        'dominant_phi':     dominant_phi,
        'channel_symbol':   channel_symbol,
        'learner_message':  learner_msg,
        'teacher_message':  teacher_msg,
        'target':           entry['target'],
        'shap_vector':      shap_values,
    }

print('💬 Pipeline complet CL → CF → XAI (apprenants réels)')
print('='*65)
for learner in REAL_LEARNERS[:3]:
    h, g, e, r = (learner['mean_head_movement'], learner['mean_gaze_disp'],
                  learner['mean_interaction_ev'], learner['mean_response_time'])
    cl                  = estimate_cl(h, g, e, r)[0]
    shap                = compute_shap(h, g, e, r)
    cl_state, cl_action = classify_cl(cl)
    result              = recommend_next_module(learner, DB_LEARNERS, cl)

    print(f"\n{'═'*65}")
    print(f" 👤 {learner['username']} | Groupe : {learner.get('group','?')}")
    print(f" 🧠 CL = {cl:.4f} → {cl_state} | Action : {cl_action}")

    if not result:
        print(f" 🎓 Curriculum terminé !")
        continue

    xai = translate_shap_to_pedagogy(
        shap_values = shap,
        learner_id  = learner['username'],
        module_id   = result['recommended_module'],
        cl_value    = cl,
        cl_state    = cl_state,
        verbose     = True,
    )

    print(f"\n ✅ Recommandation : {result['recommended_module']} — {result['recommended_title']}")
    print(f"    CF={result['vote_cf']:.4f} | Compat={result['compat_score']:.4f} | Score={result['final_score']:.4f}")

    expl = explain_recommendation(learner, result, cl, cl_state)
    print(f" 💬 XAI Mistral :\n{expl}")

print(f"\n{'═'*65}")
print(f" 📊 Dashboard Développeur — Log XAI ({len(XAI_DEVELOPER_LOG)} événements enregistrés)")
print(f"{'─'*65}")
for entry in XAI_DEVELOPER_LOG:
    print(f"  [{entry['timestamp'][11:19]}] {entry['learner_id']:<20} "
          f"canal={SHAP_LABELS[entry['dominant_channel']]:<35} "
          f"φ={entry['dominant_phi']:+.4f} | CL={entry['cl_value']:.3f} [{entry['cl_state']}]")
print(f"\n  💡 Ce log alimentera les graphiques Waterfall et l'évolution")
print(f"     temporelle des vecteurs SHAP dans le Dashboard Développeur.")

## 🔴 Partie 3 — Surveillance Temps Réel

---
## 🧪 BLOC D'EXPÉRIMENTATION RÉELLE — Cellules 21 → 24
> **Objectif :** Tester le système AILS complet sur de vrais élèves de Baccalauréat.
> **Flux :** Form 1 (profil) → RAG + Mistral (génération exercice) → Affichage exercice → Form 2 (auto-éval CL + score) → Recommandation exercice suivant → (boucle jusqu'à maîtrise) → CF inter-module
>
> ⚠️ **Prérequis :** Cellules 1 → 16 exécutées · `bac.rar` présent sur Google Drive · Liens des deux formulaires Google configurés ci-dessous.

In [ ]:

import json, re

BAC_KNOWLEDGE_BASE = [
  {
    "id": "bac2024c_ex1",
    "source": "Bac 2024 Session Contrôle",
    "titre": "Exercice 1 (3 points)",
    "points": 3,
    "modules": ["M03"],
    "concepts": ["chaines", "recursif", "iteratif", "sous_chaine"],
    "type_algo": "fonction",
    "contexte": (
        "Soit l'algorithme de la fonction 'Inconnue' avec ch1 et ch2 deux chaînes "
        "non vides formées uniquement de lettres minuscules distinctes."
    ),
    "code_fourni": (
        "Fonction Inconnue(ch1, ch2 : Chaîne de caractères) : Chaîne de caractères\n"
        "DEBUT\n"
        "  Si ch1 = \"\" Alors\n"
        "    Retourner \"\"\n"
        "  Sinon Si Pos(ch1[0], ch2) = -1 Alors\n"
        "    Retourner Inconnue(Sous_Chaine(ch1, 1, Long(ch1)), ch2)\n"
        "  Sinon\n"
        "    Retourner ch1[0] + Inconnue(Sous_Chaine(ch1, 1, Long(ch1)), ch2)\n"
        "  FinSi\n"
        "FIN"
    ),
    "questions": [
      "1) Donner une trace d'exécution manuelle pour :\n"
      "   - Cas1 : ch1=\"bac\", ch2=\"courage\" → résultat = \"ac\"\n"
      "   - Cas2 : ch1=\"image\", ch2=\"matrice\" → résultat = \"imae\"",
      "2) En déduire le rôle de la fonction Inconnue.",
      "3) Écrire un algorithme itératif de la fonction Inconnue donnant le même résultat."
    ],
    "bareme": {"Q1": 1.5, "Q2": 0.5, "Q3": 1.0},
  },
  {
    "id": "bac2024c_ex2",
    "source": "Bac 2024 Session Contrôle",
    "titre": "Exercice 2 (3 points)",
    "points": 3,
    "modules": ["M05", "M06"],
    "concepts": ["calcul", "approximation", "boucle", "valeur_approchee"],
    "type_algo": "fonction",
    "contexte": (
        "Pour obtenir une valeur approchée de ln(2), on calcule la somme des n premiers "
        "termes de la suite : ln(2) = 1 - 1/2 + 1/3 - ... + (-1)^(n-1)/n"
    ),
    "code_fourni": None,
    "questions": [
      "Écrire un algorithme d'une fonction CalculLn2(epsilon) qui retourne une valeur "
      "approchée de ln(2). Le calcul s'arrête lorsque la valeur absolue du dernier "
      "terme ajouté est inférieure ou égale à epsilon."
    ],
    "bareme": {"Q1": 3.0},
  },
  {
    "id": "bac2024c_ex3",
    "source": "Bac 2024 Session Contrôle",
    "titre": "Exercice 3 (6 points)",
    "points": 6,
    "modules": ["M01", "M03"],
    "concepts": ["fichiers", "tableaux", "enregistrements", "recursif", "calcul"],
    "type_algo": "procédure",
    "contexte": (
        "Le coefficient binomial Cb(n,k) : Cb(n,k)=0 si n<k ; Cb(n,k)=1 si k=0 ou k=n ; "
        "Cb(n,k)=Cb(n-1,k-1)+Cb(n-1,k) sinon. "
        "On remplit un tableau T depuis un fichier 'Coeff.txt' contenant des couples (n,k). "
        "Chaque enregistrement a 3 champs : n, k (entiers), r (résultat entier)."
    ),
    "code_fourni": None,
    "questions": [
      "1) Déclarer un type pour le tableau T (max 50 couples) ainsi que tout type nécessaire.",
      "2) Écrire un algorithme d'une procédure Remplir(Ch, T) qui remplit le tableau T "
         "depuis le fichier (Ch = chemin physique du fichier)."
    ],
    "bareme": {"Q1": 2.0, "Q2": 4.0},
  },
  {
    "id": "bac2024c_ex4",
    "source": "Bac 2024 Session Contrôle",
    "titre": "Exercice 4 (8 points)",
    "points": 8,
    "modules": ["M01"],
    "concepts": ["tableaux", "matrices", "enregistrements", "binaire"],
    "type_algo": "procédure",
    "contexte": (
        "Dissimulation d'une image Img2 dans Img1. Deux matrices L×C de pixels (R,V,B entiers 0-255). "
        "Pour chaque pixel : convertir la couleur rouge en binaire 8 bits (bin1 et bin2), "
        "remplacer les 4 derniers bits de bin1 par les 4 premiers bits de bin2, "
        "puis reconvertir en décimal."
    ),
    "code_fourni": None,
    "questions": [
      "1) Déclarer un type Pixel (3 champs R,V,B) et un type MatriceImage.",
      "2) Écrire une fonction BinEnDec(B : Tableau[0..7] de 0..1) : Entier.",
      "3) Écrire une fonction DecEnBin(N : Entier) : Tableau[0..7] de 0..1.",
      "4) Écrire une procédure Dissimuler(Img1, Img2 : MatriceImage; L, C : Entier)."
    ],
    "bareme": {"Q1": 1.0, "Q2": 1.5, "Q3": 1.5, "Q4": 4.0},
  },
  {
    "id": "bac2025p_ex1",
    "source": "Bac 2025 Session Principale",
    "titre": "Exercice 1 (3 points)",
    "points": 3,
    "modules": ["M03", "M04"],
    "concepts": ["chaines", "iteratif", "recursif", "conversion_base", "Chr"],
    "type_algo": "fonction",
    "contexte": "Conversion d'un entier N vers la base B. NB : Chr(65)='A', Chr(97)='a', Chr(48)='0'.",
    "code_fourni": (
        "Fonction Inconnue (N, B : Entier) : Chaîne de caractères\n"
        "DEBUT\n"
        "  Si N=0 Alors\n"
        "    Retourner \"0\"\n"
        "  Sinon\n"
        "    CH ← \"\"\n"
        "    Répéter\n"
        "      K ← N Mod B + 48 + ((N Mod B) Div 10) * 7\n"
        "      CH ← Chr(K) + CH\n"
        "      N ← N Div B\n"
        "    Jusqu'à N = 0\n"
        "    Retourner CH\n"
        "  FinSi\n"
        "FIN"
    ),
    "questions": [
      "1a) Valider (V/F) : Inconnue(5, 2) retourne 100 / 31 / 101",
      "1b) Valider (V/F) : Inconnue(31, 16) retourne 1F / 31 / F1",
      "1c) La fonction Inconnue permet de : convertir hexadécimal→décimal / "
          "décimal→base donnée / base donnée→base 10",
      "2) Proposer un algorithme récursif pour la fonction Inconnue."
    ],
    "bareme": {"Q1a": 0.75, "Q1b": 0.75, "Q1c": 0.5, "Q2": 1.0},
  },
  {
    "id": "bac2025p_ex2",
    "source": "Bac 2025 Session Principale",
    "titre": "Exercice 2 (3,5 points)",
    "points": 3.5,
    "modules": ["M04", "M05"],
    "concepts": ["calcul", "suites", "tableaux", "enregistrements", "boucle"],
    "type_algo": "fonction",
    "contexte": (
        "Nombre de Keith : entier K de d chiffres, suite U où U_1..U_d sont les chiffres de K "
        "et U_n = U_{n-1}+...+U_{n-d} pour n>d. K est un nombre de Keith si K apparaît dans "
        "la suite. Ex: Verif(197)=8, Verif(24)=-1."
    ),
    "code_fourni": None,
    "questions": [
      "1) Écrire une fonction Verif(K) qui retourne n si U_n=K ou -1 sinon.",
      "2) En utilisant Verif, écrire un programme qui remplit un tableau NK "
         "d'enregistrements (Nbre:entier, Num:entier) par tous les nombres de Keith < 100000."
    ],
    "bareme": {"Q1": 2.0, "Q2": 1.5},
  },
  {
    "id": "bac2025c_ex2",
    "source": "Bac 2025 Session Contrôle",
    "titre": "Exercice 2 (3,5 points)",
    "points": 3.5,
    "modules": ["M04", "M05"],
    "concepts": ["calcul", "suites", "tableaux", "enregistrements", "boucle"],
    "type_algo": "fonction",
    "contexte": (
        "Soit K un entier de d chiffres (d≥2) ayant la forme C1C2...Cd. Suite U telle que "
        "les d premiers termes sont les chiffres de K et U_n = U_{n-1}+...+U_{n-d} pour n>d. "
        "K est un nombre de Keith si un terme de U est égal à K."
    ),
    "code_fourni": None,
    "questions": [
      "1) Écrire une fonction Verif(K) qui retourne n (position dans U) si K est un "
         "nombre de Keith, -1 sinon.",
      "2) En utilisant Verif, écrire un algorithme d'un programme qui remplit un tableau NK "
         "d'enregistrements (Nbre, Num) par les nombres de Keith < 100000."
    ],
    "bareme": {"Q1": 2.0, "Q2": 1.5},
  },
]

print(f'✅ Knowledge Base chargée : {len(BAC_KNOWLEDGE_BASE)} exercices réels Bac')

MODULE_CONCEPTS_MAP = {
    'M01': ['fichiers', 'enregistrements', 'tableaux', 'matrices', 'binaire'],
    'M02': ['tri', 'tableaux', 'recherche', 'boucle'],
    'M03': ['recursif', 'chaines', 'sous_chaine', 'Chr'],
    'M04': ['suites', 'recursif', 'boucle', 'calcul', 'enregistrements'],
    'M05': ['calcul', 'approximation', 'valeur_approchee', 'arithmetique'],
    'M06': ['approximation', 'optimisation', 'calcul', 'boucle'],
}

def retrieve_fewshots(module_id: str, error_type: str = 'general',
                      n: int = 2, exclude_ids: list = None) -> list:
    """
    Retourne jusqu'à `n` exercices de la KB les plus pertinents.

    Scoring :
      +3  si l'exercice est tagué pour ce module exact
      +2  si concept du module dans l'exercice
      +1  si error_type correspond aux concepts
      −∞  si id dans exclude_ids
    """
    exclude_ids = exclude_ids or []
    target_concepts = set(MODULE_CONCEPTS_MAP.get(module_id, []))

    if not target_concepts:
        mid_lower = module_id.lower()
        for k, vals in MODULE_CONCEPTS_MAP.items():
            if any(v in mid_lower for v in vals):
                target_concepts.update(vals)

    error_bonus = {
        'comprehension': {'recursif', 'iteratif'},
        'logique':       {'boucle', 'recursif', 'calcul'},
        'syntaxe':       {'chaines', 'iteratif'},
        'calcul':        {'calcul', 'approximation', 'suites'},
        'general':       set(),
    }.get(error_type, set())

    scored = []
    for ex in BAC_KNOWLEDGE_BASE:
        if ex['id'] in exclude_ids:
            continue
        ex_concepts = set(ex.get('concepts', []))
        score  = 10 * (module_id in ex.get('modules', []))
        score += 3  * len(target_concepts & ex_concepts)
        score += 1  * len(error_bonus & ex_concepts)
        scored.append((score, ex))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = [ex for score, ex in scored if score > 0][:n]
    if len(results) < n:
        neutral = [ex for score, ex in scored if score == 0]
        results += neutral[:n - len(results)]
    return results[:n]

def format_fewshot_example(ex: dict) -> str:
    """Formate un exercice KB → bloc texte pour injection dans le prompt."""
    lines = [
        f"=== EXEMPLE OFFICIEL BAC — {ex['source']} ===",
        f"Titre : {ex['titre']}",
        f"Contexte : {ex['contexte']}",
    ]
    if ex.get('code_fourni'):
        lines.append(f"\nAlgorithme fourni :\n{ex['code_fourni']}")
    lines.append("\nTravail demandé :")
    for q in ex['questions']:
        lines.append(f"  • {q}")
    total = sum(ex['bareme'].values())
    lines.append(f"\n[Barème : {ex['bareme']} — Total : {total} pts]")
    lines.append("=== FIN EXEMPLE ===\n")
    return '\n'.join(lines)

def build_rag_context(module_id: str, n_docs: int = 2, max_len: int = 1400) -> str:
    """
    Retourne une chaîne de texte prête à être injectée dans le prompt de Cellule 23.
    Récupère `n_docs` exercices réels du Bac pertinents pour `module_id`,
    formate chacun et tronque si nécessaire à `max_len` caractères.

    Usage dans Cellule 23 :
        rag_context = build_rag_context(module_id, n_docs=2, max_len=1400)
    """
    examples = retrieve_fewshots(module_id, error_type='general', n=n_docs)
    if not examples:
        return "(Aucun exemple disponible dans la base pour ce module)"

    blocks = [format_fewshot_example(ex) for ex in examples]
    full_text = '\n'.join(blocks)

    if len(full_text) > max_len:
        full_text = full_text[:max_len] + '\n... [tronqué]'
    return full_text

print('\n── Test build_rag_context ──')
for mid in ['M01', 'M03', 'M05']:
    ctx = build_rag_context(mid, n_docs=2, max_len=400)
    print(f'  {mid} → {len(ctx)} chars | aperçu : {ctx[:80].strip()!r}...')

print('\n── Test retrieve_fewshots ──')
for mid, err in [('M03','logique'), ('M01','calcul'), ('M06','comprehension')]:
    res = retrieve_fewshots(mid, err, n=2)
    ids = [ex['id'] for ex in res]
    print(f'  {mid}/{err} → {ids}')

print('\n✅ Cellule 21 (RAG) chargée — build_rag_context() disponible pour Cellule 23')

In [ ]:

FORM1_URL = 'https://docs.google.com/forms/d/e/1FAIpQLScv_Aj7WfwUaUBtSBlYrh1zE3pHSuaSDjqAWKBCGuBYqxpRkg/viewform'
FORM2_URL = 'https://docs.google.com/forms/d/e/1FAIpQLSeRDZzG3r4EfHeXDlXPc76ruzKmyBLFo6P9z0JLJTXk2w81fA/viewform'

EXP_POLL_INTERVAL   = 20

EXP_MASTERY_WINDOW  = N_WINDOW
EXP_MASTERY_SCORE   = EXP_MASTERY_SCORE

                                                                                        
                                                                                                 

EXP_EXERCISE_SYS = """
Tu es un professeur expert en Informatique pour les élèves de Baccalauréat tunisien (Section Informatique).
Tu génères des exercices UNIQUEMENT en pseudo-code style Bac tunisien ou Python standard.

STYLE OBLIGATOIRE — PSEUDO-CODE BAC TUNISIEN :
- Fonctions : Fonction NomFonction(Param : Type) : TypeRetour
- Procédures : Procédure NomProc(Param : Type)
- Affectation : ←
- Conditions : Si ... Alors ... Sinon ... FinSi
- Boucles : Pour I de X à Y Faire ... Fin Pour | TantQue ... Faire ... Fin TantQue
- Tableaux : T : Tableau[0..N-1] de TypeElement
- Chaînes : Long(), Pos(), Copie(), Concat(), Ch[I]
- Fichiers : Assigner, Ouvrir, Fermer, LireSeq, EcrireSeq, FDF()
- Déclaration : T.D.O.L (Tableau de Déclaration des Objets Locaux)

STRUCTURE OBLIGATOIRE DE L'EXERCICE (exactement comme le Bac) :
{
  "titre": "Exercice X (N points)",
  "contexte": "Description du problème en français",
  "algorithme_fourni": "code pseudo-code si nécessaire (peut être vide)",
  "questions": [
    {
      "numero": 1,
      "points": 2,
      "enonce": "1) Compléter l'entête / Compléter le T.D.O.L / Tracer le tableau / ...",
      "type": "completion|trace|redaction|correction|analyse"
    }
  ],
  "indices": ["indice 1 si CL élevée", "indice 2"],
  "conseil_pedagogique": "conseil court",
  "difficulte_reelle": 2,
  "duree_estimee_minutes": 20,
  "bareme_total": 10
}

TYPES DE QUESTIONS BAC (utiliser ces formulations exactes) :
- Q1 simple : "Compléter l'entête de la fonction/procédure"
- Q1 simple : "Compléter le T.D.O.L (Tableau de Déclaration des Objets Locaux)"
- Q2 moyen  : "Compléter la colonne Résultat affiché du tableau ci-dessous"
- Q2 moyen  : "Tracer le tableau d'évolution des variables pour [appel]"
- Q3 hard   : "En déduire le rôle de cette fonction/procédure"
- Q3 hard   : "Écrire une procédure/fonction qui..."

CONTRAINTES STRICTES :
1. Barème total = exactement 10 points répartis sur 3-4 questions
2. Questions progressives : Q1 facile → Q3 difficile
3. Indice obligatoire si CL > 0.60
4. Contexte réaliste (traitement de chaînes, fichiers élèves, calculs numériques...)
5. FORMAT : JSON uniquement, sans texte avant ni après

LISTE NOIRE (concepts INTERDITS) :
pointeurs, malloc, héritage, polymorphisme, templates, threads,
sockets, arbres AVL, SQL, API REST, lambda, regex avancées
"""

ERROR_DIRECTIVES = {
    'comprehension': {
        'label': 'COMPRÉHENSION',
        'q1_type': 'définition',
        'q2_type': 'application simple',
        'q3_type': 'exemple commenté',
        'directive': "L'élève a des difficultés de COMPRÉHENSION. Q1=définition du concept, Q2=application directe guidée, Q3=expliquer un code simple.",
    },
    'logique': {
        'label': 'LOGIQUE',
        'q1_type': 'débogage code erroné',
        'q2_type': 'complétion algorithme',
        'q3_type': "trace d'exécution",
        'directive': "L'élève fait des erreurs de LOGIQUE. Q1=déboguer un code ERRONÉ fourni (raisonnement procédural), Q2=compléter un algorithme lacunaire, Q3=tracer l'exécution pas-à-pas.",
    },
    'syntaxe': {
        'label': 'SYNTAXE',
        'q1_type': "correction d'erreurs syntaxiques",
        'q2_type': 'réécriture correcte',
        'q3_type': 'écriture depuis zéro',
        'directive': "L'élève a des problèmes de SYNTAXE. Q1=corriger les erreurs dans un code fourni, Q2=réécrire proprement, Q3=écrire un algorithme similaire sans erreur.",
    },
    'calcul': {
        'label': 'CALCUL',
        'q1_type': 'vérification pas-à-pas',
        'q2_type': 'trace numérique',
        'q3_type': 'optimisation',
        'directive': "L'élève fait des erreurs de CALCUL. Q1=vérifier un calcul étape par étape, Q2=trace numérique complète, Q3=identifier l'erreur et corriger.",
    },
    'general': {
        'label': 'GÉNÉRAL',
        'q1_type': 'réponse courte',
        'q2_type': 'code/algorithme',
        'q3_type': 'analyse/justification',
        'directive': "Exercice de consolidation générale du module.",
    },
}

                                                                                      

print('✅ Configuration expérimentation chargée')
print(f'   Form 1 (Profil)      : {FORM1_URL[:60]}...')
print(f'   Form 2 (Auto-éval)   : {FORM2_URL[:60]}...')
print(f'   Seuil maîtrise       : score≥{EXP_MASTERY_SCORE} | CL≤{EXP_MASTERY_CL_MAX} | fenêtre={EXP_MASTERY_WINDOW}')
print(f'   Max exercices/module : {EXP_MAX_EXERCISES}')
print(f'   Directives par erreur : {list(ERROR_DIRECTIVES.keys())}')
print(f'   Blacklist            : {len(CURRICULUM_BLACKLIST)} concepts interdits')


In [ ]:

import gradio as gr
import threading
import time
import json
import re
import numpy as np
import pandas as pd
from datetime import datetime

def _parse_mistral_json(raw: str):
    if not raw:
        return None
    cleaned = re.sub(r'```json\s*', '', raw)
    cleaned = re.sub(r'```\s*', '', cleaned)
    cleaned = cleaned.strip()
    m = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if not m:
        return None
    json_str = m.group()
    result = []
    in_string = False
    escape = False
    i = 0
    while i < len(json_str):
        ch = json_str[i]
        if escape:
            if ch in ('"', '\\', '/', 'b', 'f', 'n', 'r', 't', 'u'):
                result.append(ch)
            else:
                result.append(ch)
            escape = False
            i += 1
            continue
        if ch == '\\':
            result.append(ch)
            escape = True
            i += 1
            continue
        if ch == '"':
            in_string = not in_string
            result.append(ch)
            i += 1
            continue
        if in_string:
            if ch == '\n':
                result.append('\\n')
            elif ch == '\r':
                result.append('\\r')
            elif ch == '\t':
                result.append('\\t')
            elif ord(ch) < 0x20:
                pass
            else:
                result.append(ch)
        else:
            result.append(ch)
        i += 1
    json_str = ''.join(result)
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        try:
            repaired = _repair_truncated_json(json_str)
            return json.loads(repaired)
        except Exception:
            print(f"⚠️ _parse_mistral_json failed: {e} | preview: {json_str[:200]}")
            return None

def _repair_truncated_json(s: str) -> str:
    stack = []
    in_string = False
    escape = False
    for ch in s:
        if escape:
            escape = False
            continue
        if ch == '\\' and in_string:
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if not in_string:
            if ch == '{':
                stack.append('}')
            elif ch == '[':
                stack.append(']')
            elif ch in ('}', ']'):
                if stack and stack[-1] == ch:
                    stack.pop()
    if in_string:
        s += '"'
    s += ''.join(reversed(stack))
    return s

EXP_STATE = {
    'active':                False,
    'learners':              {},
    'processed_f1':          set(),
    'processed_f2':          set(),
    'log':                   [],
    'last_f1_count':         1,
    'last_f2_count':         1,
    'current_exercise':      None,
    'current_exercise_html': '',
    'current_learner_id':    None,
    'shap_message':          '',
    'answer_analysis':       '',
    'waiting_answer':        False,
    'correction_shown':      False,
    'last_error_type':       'general',
    'last_score_norm':       0.5,
    'last_mastered':         False,
                                                             
    'n_gen':                 0,                                                                                         
    'n_first_ok':            0,                                            
    'n_retry':               0,                                                                                                                                                
    'n_fallback':            0,                                        
}

_stop_event = threading.Event()
_exp_thread = [None]

def _exp_log(msg: str, level: str = 'info'):
    ts   = datetime.now().strftime('%H:%M:%S')
    icon = {'info': 'ℹ️', 'ok': '✅', 'warn': '⚠️', 'error': '❌', 'exp': '🧪'}.get(level, '•')
    entry = f"[{ts}] {icon} {msg}"
    EXP_STATE['log'].insert(0, entry)
    EXP_STATE['log'] = EXP_STATE['log'][:100]
    print(entry)

def _clean_row(row: dict) -> dict:
    return {
        re.sub(r'\s+', '_', k.strip().lower()): v
        for k, v in row.items()
    }

def _detect_lid(row: dict) -> str:
    for key in ['learner_id_(provided_by_the_experimenter)',
                'learner_id', 'email_address', 'email', 'id']:
        if key in row and str(row[key]).strip():
            return str(row[key]).strip()
    return 'unknown'

def _init_learner_state(lid, prior: float, conf: float,
                        motiv: float, group: str) -> dict:
    start_mod, mastered = cold_start_state(prior)                             

    st = {
        'id':               lid,
        'group':            group,
        'prior':            prior,
        'confidence':       conf,
        'motivation':       motiv,
        'mean_cl':          0.5,
        'current_module':   start_mod,
        'mastered_modules': mastered,
        'module_scores':    [],
        'module_cls':       [],
        'module_progress':  {},
        'exercise_history': [],
        'exercise_index':   0,
        'ex_count_module':  0,
        'last_exercise':    None,
        'waiting_form2':    False,
        'seen_exercises':   [],
        'last_shap_features': [],
    }
    EXP_STATE['learners'][lid] = st
    return st

def _fetch_new_sheet_rows(tab_name: str, last_count_key: str) -> list:
    try:
        ws   = sh.worksheet(tab_name)
        all_ = ws.get_all_values()
        cur  = len(all_)
        prev = EXP_STATE[last_count_key]
        if cur <= prev:
            return []
        headers  = all_[0]
        new_rows = all_[prev:cur]
        EXP_STATE[last_count_key] = cur
        return [
            dict(zip(headers, r + [''] * (len(headers) - len(r))))
            for r in new_rows
        ]
    except Exception as ex:
        _exp_log(f'Error reading {tab_name}: {ex}', 'error')
        return []

def _write_result_to_sheet(row_dict: dict):
    try:
        try:
            ws = sh.worksheet(RESULTS_TAB_NAME)
        except Exception:
            ws = sh.add_worksheet(title=RESULTS_TAB_NAME, rows=1000, cols=20)
            ws.append_row([
                'timestamp', 'learner_id', 'module_id', 'exercise_index',
                'exercise_title', 'difficulty', 'cl_value', 'cl_state',
                'score_raw', 'score_norm', 'mastered', 'next_action',
                'error_type', 'shap_message', 'recommendation', 'clq_raw', 'cl_smoothed',
            ])
        ws.append_row([
            row_dict.get(k, '') for k in [
                'timestamp', 'learner_id', 'module_id', 'exercise_index',
                'exercise_title', 'difficulty', 'cl_value', 'cl_state',
                'score_raw', 'score_norm', 'mastered', 'next_action',
                'error_type', 'shap_message', 'recommendation', 'clq_raw', 'cl_smoothed',
            ]
        ])
    except Exception as ex:
        _exp_log(f'Sheet write error: {ex}', 'error')

def _check_mastery(st: dict, mid: str = None) -> bool:
                                                                  
                                                             
    sc  = st['module_scores']
    cl  = st['module_cls']
    win = EXP_MASTERY_WINDOW
    if len(sc) < win:
        return False
    last_sc = sc[-win:]
    last_cl = cl[-win:]
    mid_key = mid or st.get('current_module')
    theta   = MODULES[mid_key]['theta'] if mid_key in MODULES else EXP_MASTERY_SCORE
    return (
        np.mean(last_sc) >= theta
        and np.mean(last_cl) <= EXP_MASTERY_CL_MAX
        and last_sc[-1] >= last_sc[0]
    )

ERROR_TO_CONCEPTS = {
    'comprehension': ['reading_problem_statement', 'identifying_variables', 'data_typing', 'algorithm_specification'],
    'logic':         ['conditional_structures', 'iterative_loops', 'conditional_loops', 'nested_structures', 'edge_cases'],
    'syntax':        ['bac_pseudo_code', 'assignment', 'variable_declaration', 'writing_functions', 'bac_conventions'],
    'calcul':        ['arithmetic_expressions', 'operators', 'operator_precedence', 'numeric_types', 'precision'],
    'boundary':      ['edge_cases', 'extreme_values', 'stopping_conditions', 'infinite_loops', 'error_handling'],
    'general':       ['algorithmic_methodology', 'problem_decomposition', 'solution_planning', 'code_review'],
}

def get_weak_concepts(st: dict) -> list:
    history = st.get('exercise_history', [])
    if not history:
        prior = st.get('prior', 0.5)
        if prior < 0.35:   return ERROR_TO_CONCEPTS['comprehension'][:3]
        elif prior < 0.60: return ERROR_TO_CONCEPTS['logic'][:3]
        else:              return ERROR_TO_CONCEPTS['syntax'][:2]

    error_weights = {}
    recency_window = 5
    for i, rec in enumerate(history):
        et     = rec.get('error_type', 'general')
        score  = rec.get('score', 0.5)
        weight = (1.0 - score)
        if i >= len(history) - recency_window:
            weight *= 2.0
        error_weights[et] = error_weights.get(et, 0.0) + weight

    concept_scores = {}
    for et, w in error_weights.items():
        for concept in ERROR_TO_CONCEPTS.get(et, ERROR_TO_CONCEPTS['general']):
            concept_scores[concept] = concept_scores.get(concept, 0.0) + w

    sorted_concepts = sorted(concept_scores, key=lambda c: concept_scores[c], reverse=True)
    weak = sorted_concepts[:6]
    _exp_log(f'Weak concepts: {weak[:3]}', 'info')
    return weak if weak else ERROR_TO_CONCEPTS['general'][:3]

def _exercise_similarity(ex_new: dict, ex_seen: dict) -> float:
    def _tokenize(text):
        if not text: return set()
        words = re.sub(r'[^\w\s]', ' ', str(text).lower()).split()
        return {w for w in words if len(w) >= 4}

    def _jaccard(a, b):
        if not a and not b: return 0.0
        union = a | b
        return len(a & b) / len(union) if union else 0.0

    def _concat_q(ex):
        return ' '.join(q.get('enonce', '') for q in ex.get('questions', []))

    return round(
        0.30 * _jaccard(_tokenize(ex_new.get('titre', '')),    _tokenize(ex_seen.get('titre', '')))
      + 0.40 * _jaccard(_tokenize(ex_new.get('contexte', '')), _tokenize(ex_seen.get('contexte', '')))
      + 0.30 * _jaccard(_tokenize(_concat_q(ex_new)),          _tokenize(_concat_q(ex_seen))),
        4
    )

def _is_too_similar(ex_new: dict, seen_exercises: list, threshold: float = 0.60) -> bool:
    for seen in seen_exercises:
        sim = _exercise_similarity(ex_new, seen)
        if sim >= threshold:
            _exp_log(f'Exercise rejected (sim={sim:.2f}) with "{seen.get("titre", "?")}"', 'warn')
            return True
    return False

def generate_full_solution(exercise: dict, module_id: str) -> str:
    prompt = f"""
You are a Tunisian Baccalaureate Computer Science teacher.
Exercise: {json.dumps(exercise, ensure_ascii=False)}
Provide ONLY the complete solution.
JSON format: {{"solution_complete": "complete Bac pseudo-code"}}
"""
    raw = call_mistral("You are an official examiner.", prompt, max_tokens=1200)
    data = _parse_mistral_json(raw)
    return data.get("solution_complete", "") if data else ""

def _build_analyse_sys(error_type: str, weak_concepts: list, module_id: str) -> str:
    directive = {
        'comprehension': "The student does not understand the problem statement. Explain each basic concept. Be very pedagogical.",
        'logic':         "The student makes logic errors. Show the reasoning step by step.",
        'syntax':        "The student understands the logic but makes Bac syntax mistakes. Correct each mistake precisely.",
        'calcul':        "The student makes calculation errors. Trace each numerical step.",
        'general':       "Complete and pedagogical correction.",
    }.get(error_type, "Complete and pedagogical correction.")
    concepts = ', '.join(weak_concepts[:3]) if weak_concepts else 'general algorithms'
    return f"""You are an official corrector for the Tunisian Baccalaureate in Computer Science.

STUDENT PROFILE:
- Dominant error type: {error_type}
- Weak concepts: {concepts}
- Module: {module_id}
- Pedagogical directive: {directive}

RULES:
- Correct each question separately with the points obtained
- Explain WHY each mistake is wrong
- Full correction in Tunisian Bac pseudo-code
- Final advice must mention: {concepts}

BAC PSEUDO-CODE STYLE:
- Assignment: ←
- Conditions:  If ... Then ... Else ... EndIf
- Loops:       For I from X to Y Do ... EndFor / While ... Do ... EndWhile
- Functions:   Function Name(Param : Type) : ReturnType

Return ONLY this JSON (no text before or after):
{{
  "score_estime": <integer 0-10>,
  "type_erreur": "<comprehension|logic|syntax|calcul|general>",
  "resume_erreur": "<2-3 sentences targeting weak concepts>",
  "correction_detaillee": {{
    "explication": "<3-5 sentences explaining WHY the mistakes relate to {concepts}>",
    "code_corrige": "<COMPLETE and correct Bac pseudo-code>",
    "points_par_question": "<Q1=X/Y pts: reason | Q2=X/Y pts: reason | Q3=X/Y pts: reason>",
    "conseil": "<advice mentioning {concepts}>"
  }}
}}"""

def _build_exercise_specific_fallback(exercise: dict) -> str:
    lines = [f"// Correction for: {exercise.get('titre', 'Exercise')}"]
    lines.append("// (Locally generated correction — Mistral unavailable)")
    lines.append("")
    alg = exercise.get('algorithme_fourni') or exercise.get('algorithm_fourni')
    if alg and str(alg).strip() not in ('', 'null', 'None'):
        lines.append("// Provided algorithm to complete:")
        lines.append(str(alg))
        lines.append("")
    for i, q in enumerate(exercise.get('questions', [])):
        lines.append(
            f"// Q{q.get('numero', i+1)} ({q.get('points', '?')} pts): "
            f"{q.get('enonce', '')[:80]}"
        )
        lines.append("// → [Answer to be constructed]")
        lines.append("")
    conseil = exercise.get('conseil_pedagogique', '')
    if conseil:
        lines.append(f"// Advice: {conseil}")
    return '\n'.join(lines)

def analyze_student_answer(answer_text: str, exercise: dict, module_id: str) -> dict:
    if not answer_text or len(answer_text.strip()) < 5:
        correction = generate_full_solution(exercise, module_id)
        points_detail = ' | '.join([
            f"Q{q.get('numero', i+1)} = 0/{q.get('points', '?')} pts: no answer provided"
            for i, q in enumerate(exercise.get('questions', []))
        ])
        return {
            'score_estime': 0,
            'type_erreur':  'comprehension',
            'resume_erreur': f"No answer provided for \"{exercise.get('titre', '?')}\".",
            'correction_detaillee': {
                'explication': (
                    f"This exercise covered: {exercise.get('contexte', '')[:200]}. "
                    f"Here is the full correction for all {len(exercise.get('questions', []))} questions."
                ),
                'code_corrige':        correction,
                'points_par_question': points_detail,
                'conseil': exercise.get('conseil_pedagogique', 'Tunisian Bac algorithmic logic.'),
            },
        }

    questions_text = '\n'.join([
        f"Q{q.get('numero', i+1)} ({q.get('points', '?')} pts): {q.get('enonce', '')}"
        for i, q in enumerate(exercise.get('questions', []))
    ])
    user_prompt = f"""Module: {module_id} — {MODULES.get(module_id, {}).get('title', '?')}
Exercise: {exercise.get('titre', '')}
Context: {exercise.get('contexte', '')}
Provided algorithm: {exercise.get('algorithme_fourni') or '(none)'}
Questions: {questions_text}
Student answer: {answer_text[:2000]}
Return ONLY the requested JSON."""

    lid              = EXP_STATE.get('current_learner_id')
    st_local         = EXP_STATE['learners'].get(lid, {}) if lid else {}
    error_type_st    = st_local.get('exercise_history', [{}])[-1].get('error_type', 'general')\
                       if st_local.get('exercise_history') else 'general'
    weak_concepts_st = get_weak_concepts(st_local) if st_local else ['algorithmic_methodology']

    sys_prompt = _build_analyse_sys(error_type_st, weak_concepts_st, module_id)
    raw        = call_mistral(sys_prompt, user_prompt, max_tokens=1200)
    parsed     = _parse_mistral_json(raw)

    if parsed is not None:
        try:
            parsed['score_estime'] = max(0, min(10, round(float(parsed.get('score_estime', 5)))))
        except:
            parsed['score_estime'] = 5

        type_map = {
            'comprehension': 'comprehension', 'compréhension': 'comprehension',
            'logique': 'logic', 'logic': 'logic', 'syntaxe': 'syntax',
            'syntax': 'syntax', 'calcul': 'calcul', 'calculation': 'calcul',
            'general': 'general', 'général': 'general', 'boundary': 'boundary',
        }
        raw_type = str(parsed.get('type_erreur', 'general')).lower().strip()
        parsed['type_erreur'] = 'general'
        for key, val in type_map.items():
            if key in raw_type:
                parsed['type_erreur'] = val
                break

        if 'correction_detaillee' not in parsed:
            parsed['correction_detaillee'] = {
                'explication': parsed.get('correction_courte', ''),
                'code_corrige': '', 'points_par_question': '', 'conseil': '',
            }
        corr = parsed['correction_detaillee']
        if not corr.get('explication'):
            corr['explication'] = "The provided answer does not correctly address the questions."
        if not corr.get('points_par_question'):
            corr['points_par_question'] = "Q1=0 pts | Q2=0 pts | Q3=0 pts"
        if not corr.get('conseil'):
            corr['conseil'] = "Re-read the problem statement and use Tunisian Bac pseudo-code."
        return parsed

    txt    = answer_text.lower().strip()
    n_mots = len(txt.split())
    score  = 0
    if n_mots >= 20: score += 2
    if n_mots >= 50: score += 2
    if 'begin' in txt or 'start' in txt or 'debut' in txt or 'début' in txt: score += 1
    if 'end' in txt or 'fin' in txt: score += 1
    if '←' in answer_text: score += 1
    if 'if' in txt and 'then' in txt: score += 1
    if 'for' in txt and 'do' in txt: score += 1
    if 'function' in txt or 'procedure' in txt: score += 1
    score = min(10, score)
    error = 'comprehension' if score == 0 else 'logic' if score <= 3 else 'syntax' if score <= 6 else 'general'

    solution_officielle = exercise.get('_solution_officielle', '')
    if not solution_officielle:
        solution_officielle = generate_full_solution(exercise, module_id)
    code_fallback = solution_officielle if solution_officielle else _build_exercise_specific_fallback(exercise)

    points_fallback = ' | '.join([
        f"Q{q.get('numero', i+1)}=0/{q.get('points', '?')} pts"
        for i, q in enumerate(exercise.get('questions', []))
    ])

    return {
        'score_estime': score, 'type_erreur': error,
        'resume_erreur': f"Answer analysed locally ({n_mots} words).",
        'correction_detaillee': {
            'explication': f"Here is the full correction for the exercise \"{exercise.get('titre', '?')}\".",
            'code_corrige': code_fallback,
            'points_par_question': points_fallback,
            'conseil': exercise.get('conseil_pedagogique', 'Check: Begin/End, ←, If/Then/EndIf.'),
        },
    }

def get_shap_explanation(error_type: str, cl_value: float,
                         score_norm: float, module_id: str) -> str:

    lid      = EXP_STATE.get('current_learner_id')
    st_local = EXP_STATE['learners'].get(lid, {}) if lid else {}
    history  = st_local.get('exercise_history', [])

    if len(history) >= 2:
        scores_recents = [r.get('score', 0.5) for r in history[-3:]]
        tendance = scores_recents[-1] - scores_recents[0]
    else:
        tendance = 0.0

    error_counts = {}
    for r in history[-5:]:
        et = r.get('error_type', 'general')
        error_counts[et] = error_counts.get(et, 0) + 1
    erreur_count = max(error_counts.values()) if error_counts else 0

    x_hat = {
        'h': round(abs(1 - score_norm), 4),
        'g': round(max(0, -tendance + 0.1), 4),
        'e': round(min(1.0, erreur_count / 3.0), 4),
        'r': round(cl_value, 4),
    }
    features = list(x_hat.keys())
    n = len(features)

    if history:
        v_base = float(np.mean([r.get('score', 0.5) for r in history]))
    else:
        v_base = st_local.get('prior', 0.5)

    global SHAP_MODEL, SHAP_SCALER, SHAP_EXPLAINER
    use_real_shap = (
        'SHAP_EXPLAINER' in globals() and SHAP_EXPLAINER is not None
        and 'SHAP_SCALER' in globals() and SHAP_SCALER is not None
    )

    if use_real_shap:

        x_real = np.array([[score_norm, cl_value, tendance, erreur_count / 3.0]])
        x_real_scaled = SHAP_SCALER.transform(x_real)
        try:
            shap_vals = SHAP_EXPLAINER.shap_values(x_real_scaled)[0]

            phi = {
                'h': round(float(shap_vals[0]), 6),
                'g': round(float(shap_vals[2]), 6),
                'e': round(float(shap_vals[3]), 6),
                'r': round(float(shap_vals[1]), 6),
            }
            v_base = float(SHAP_MODEL.predict(SHAP_SCALER.transform(
                np.array([[0.5, 0.5, 0.0, 0.0]])))[0])
            v_full = float(SHAP_MODEL.predict(x_real_scaled)[0])
            sum_phi = sum(phi.values())
        except Exception:
            use_real_shap = False

    if not use_real_shap:

        phi = _heuristic_phi(score_norm, cl_value, tendance, erreur_count / 3.0)

        CANAL_TO_SHORT = {'head_movement': 'h', 'gaze_direction': 'g',
                           'interaction_events': 'e', 'response_time': 'r'}
        phi = {CANAL_TO_SHORT[k]: v for k, v in phi.items()}
        v_full = v_base + sum(phi.values())
        sum_phi = sum(phi.values())

    dominant = max(phi, key=lambda k: abs(phi[k]))

    CANAL_NOMS = {'h': 'head_movement', 'g': 'gaze_direction',
                  'e': 'interaction_events', 'r': 'response_time'}
    dominant_nom = CANAL_NOMS[dominant]
    if lid and lid in EXP_STATE['learners']:
        EXP_STATE['learners'][lid]['last_shap_features'] = [dominant_nom]

    efficacite = round(abs(sum_phi - (v_full - v_base)), 6)

    n_ex      = len(history)
    score_pct = round(score_norm * 100)

    CANAL_CONTEXTE = {
        'h': f"Score de {score_pct}% sur {n_ex} exercice(s) — niveau de connaissance faible.",
        'g': f"Tendance negative sur les {min(3,n_ex)} derniers exercices (tendance={tendance:+.2f}).",
        'e': f"Erreur de type '{error_type}' repetee {erreur_count} fois recemment.",
        'r': f"Charge cognitive elevee (CL={cl_value:.2f}) — eleve fatigue ou en difficulte.",
    }
    contexte = CANAL_CONTEXTE[dominant]

                                                                           
                                                                          
    _PED_LABELS = {
        'h': 'Niveau de connaissance',
        'g': 'Tendance des scores',
        'e': 'Erreurs recurrentes',
        'r': 'Charge cognitive',
    }
    dominant_label = _PED_LABELS[dominant]
    prompt = (
        f"Tu es un tuteur pedagogique bienveillant pour un eleve du Bac Informatique tunisien.\n\n"
        f"Situation : {contexte}\n\n"
        f"Le signal pedagogique dominant est : {dominant_label} "
        f"(contribution phi={phi[dominant]:+.4f}).\n\n"
        f"Ecris UNE seule phrase courte (max 20 mots) a la 2eme personne du singulier "
        f"qui explique a l'eleve POURQUOI cet exercice lui est recommande.\n"
        f"REGLES ABSOLUES :\n"
        f"- Ne mentionne JAMAIS les lettres h, g, e, r, ni les mots canal, SHAP, Shapley, phi.\n"
        f"- Utilise uniquement des mots scolaires (score, exercice, erreur, niveau, comprehension...).\n"
        f"- Retourne UNIQUEMENT la phrase, sans guillemets."
    )

    try:
        phrase = call_mistral("Tu es un assistant pedagogique.", prompt, max_tokens=60)
        phrase = phrase.strip().strip('"').strip("'").split('\n')[0]
        if not phrase or len(phrase) < 10:
            raise ValueError("vide")
    except Exception:
                                                                      
        FALLBACK = {
            'h': "Ton score montre que tu as besoin de plus de pratique sur ce module.",
            'g': "Tes resultats recents sont en baisse — cet exercice t'aidera a consolider les bases.",
            'e': "Tu repetes ce type d'erreur — cet exercice le cible directement.",
            'r': "Ta charge cognitive est elevee — voici un exercice plus simple pour retrouver confiance.",
        }
        phrase = FALLBACK[dominant]

    cl_state, _ = classify_cl(cl_value)
    cl_icon = '🟢' if cl_value < 0.30 else '🟡' if cl_value < 0.60 else '🟠' if cl_value < 0.80 else '🔴'

    CANAL_LABELS = {
        'h': ('ĥ', 'Niveau de connaissance'),
        'g': ('ĝ', 'Tendance des scores'),
        'e': ('ê', 'Erreurs recurrentes'),
        'r': ('r̂', 'Charge cognitive'),
    }
    dominant_marker = lambda k: '**' if k == dominant else ''

    table_rows = ''
    for k in features:
        sym, label = CANAL_LABELS[k]
        dm = dominant_marker(k)
        table_rows += (
            f"| {dm}{sym}{dm} | {label} | {x_hat[k]:.4f} "
            f"| {phi[k]:+.6f} | {dm}{'Dominant' if k==dominant else ''}{dm} |\n"
        )

    return (
        f"**Pourquoi cet exercice ? (SHAP — Shapley Values)**\n\n"
        f"> {phrase}\n\n"
        f"---\n\n"
        f"**Table SHAP — Valeurs de Shapley exactes**\n\n"
        f"| Canal | Signification | x̂k | φk (Shapley) | Role |\n"
        f"|:---:|:---|:---:|:---:|:---:|\n"
        f"{table_rows}\n"
        f"**Propriete d'efficacite** : Σφk = {sum_phi:+.6f} ≈ v(F)-v(∅) = {v_full-v_base:+.6f} "
        f"(erreur={efficacite:.2e})\n\n"
        f"**Valeur de base** v(∅) = {v_base:.4f} | "
        f"**Prediction** v(F) = {v_full:.4f}\n\n"
        f"---\n"
        f"*CL : {cl_icon} **{cl_value:.3f}** [{cl_state}]"
        f" | Score : **{score_pct}%**"
        f" | Erreur : **{error_type}**"
        f" | Canal dominant : **{dominant}** (φ={phi[dominant]:+.6f})*"
    )

CONCEPTS_INTERDITS = [
    'pointer', 'malloc', 'free', 'heap', 'stack overflow',
    'recursion', 'recursive', 'object', 'class',
    'inheritance', 'polymorphism', 'interface', 'thread',
    'socket', 'api', 'framework', 'external library', 'import',
    'complexity', 'big o', 'graph', 'binary tree', 'linked list',
    'hash', 'python dictionary', 'lambda', 'generator', 'decorator',
    'exception', 'try except', 'sql', 'database', 'network',
]

REQUIRED_KEYS = {'titre', 'contexte', 'questions', 'indices', 'conseil_pedagogique',
                 'difficulte_reelle', 'duree_estimee_minutes'}
REQUIRED_QUESTION_KEYS = {'numero', 'points', 'enonce'}

SAFE_TEMPLATES = {
    'M01': {
        'titre': 'Basic Exercise — Variables and Types',
        'contexte': (
            'A program must read an integer entered by the user, '
            'double it and display the result.'
        ),
        'algorithme_fourni': None,
        'difficulte_reelle': 1,
        'duree_estimee_minutes': 10,
        'kappa': 0.95,
        'questions': [
            {'numero': 1, 'points': 3, 'enonce': 'Declare the necessary variables in Bac pseudo-code.'},
            {'numero': 2, 'points': 4, 'enonce': 'Write the complete algorithm that reads N, computes N*2 and displays the result.'},
            {'numero': 3, 'points': 3, 'enonce': 'Trace the variable evolution table for N=5.'},
        ],
        'indices': [
            'Use: Variable N: Integer',
            'Assignment with ←',
            'Write to display output',
        ],
        'conseil_pedagogique': 'Master variable declaration and assignment before anything else.',
        '_solution_officielle': '',
        '_safe_template': True,
    },
    'M02': {
        'titre': 'Exercise — Conditional Structures',
        'contexte': 'A shop applies a 10% discount if the total exceeds 100 DT.',
        'algorithme_fourni': None,
        'difficulte_reelle': 1,
        'duree_estimee_minutes': 12,
        'kappa': 0.95,
        'questions': [
            {'numero': 1, 'points': 3, 'enonce': 'Declare the necessary variables.'},
            {'numero': 2, 'points': 4, 'enonce': 'Write the algorithm using If/Then/Else/EndIf.'},
            {'numero': 3, 'points': 3, 'enonce': 'Test with amount=150 and amount=80.'},
        ],
        'indices': ['Use If Amount > 100 Then', 'Discount ← Amount * 0.1', "Don't forget EndIf"],
        'conseil_pedagogique': 'The If/Then/EndIf structure is fundamental.',
        '_solution_officielle': '',
        '_safe_template': True,
    },
    'M03': {
        'titre': 'Exercise — Loops',
        'contexte': 'Calculate the sum of the first N natural integers.',
        'algorithme_fourni': None,
        'difficulte_reelle': 1,
        'duree_estimee_minutes': 12,
        'kappa': 0.95,
        'questions': [
            {'numero': 1, 'points': 3, 'enonce': 'Write the algorithm using a For loop.'},
            {'numero': 2, 'points': 4, 'enonce': 'Trace the table for N=4.'},
            {'numero': 3, 'points': 3, 'enonce': 'Rewrite using While.'},
        ],
        'indices': ['For I from 1 to N Do', 'Sum ← Sum + I', 'Initialise Sum ← 0 before the loop'],
        'conseil_pedagogique': 'Master For/EndFor and While/EndWhile.',
        '_solution_officielle': '',
        '_safe_template': True,
    },
    'M04': {
        'titre': 'Exercise — Arrays and Functions',
        'contexte': 'An array T of 5 integers. Compute the average and find the maximum.',
        'algorithme_fourni': (
            'Function Maximum(T: Array[0..4] of Integer): Integer\n'
            'T.D.O.L\n'
            '  Max: Integer\n'
            '  i  : Integer\n'
            'Begin\n'
            '  Max ← T[0]\n'
            '  For i from 1 to 4 Do\n'
            '    If T[i] > Max Then Max ← T[i] EndIf\n'
            '  EndFor\n'
            '  Maximum ← Max\n'
            'End'
        ),
        'difficulte_reelle': 2,
        'duree_estimee_minutes': 15,
        'kappa': 0.95,
        'questions': [
            {'numero': 1, 'points': 3, 'enonce': 'Complete the T.D.O.L of the Maximum function.'},
            {'numero': 2, 'points': 4, 'enonce': 'Write a function Average(T, N) that returns the mean.'},
            {'numero': 3, 'points': 3, 'enonce': 'Write the main program that calls both functions.'},
        ],
        'indices': ['Function Average(T:Array[0..N-1] of Real, N:Integer):Real', 'Initialise Sum ← 0', 'Return Sum/N'],
        'conseil_pedagogique': 'Master arrays, T.D.O.L and parameter passing.',
        '_solution_officielle': '',
        '_safe_template': True,
    },
}

SAFE_TEMPLATES['M05'] = {
    'titre': 'Exercise — Functions',
    'contexte': 'Write a reusable function to determine whether an integer N is prime.',
    'algorithme_fourni': None,
    'difficulte_reelle': 3,
    'duree_estimee_minutes': 15,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Write the T.D.O.L of a function EstPremier(N: Integer): Boolean.'},
        {'numero': 2, 'points': 4, 'enonce': 'Write the body of EstPremier using a loop testing divisors from 2 to N-1.'},
        {'numero': 3, 'points': 3, 'enonce': 'Write the main program that calls EstPremier and displays the result for N=17.'},
    ],
    'indices': ['Function EstPremier(N:Integer):Boolean', 'A number is prime if no divisor between 2 and N-1 divides it', 'Use a boolean flag updated inside the loop'],
    'conseil_pedagogique': 'Master function signatures, parameter passing and return values.',
    '_solution_officielle': '',
    '_safe_template': True,
}
SAFE_TEMPLATES['M06'] = {
    'titre': 'Exercise — Recursion',
    'contexte': 'Compute N! (factorial of N) using a recursive function.',
    'algorithme_fourni': None,
    'difficulte_reelle': 4,
    'duree_estimee_minutes': 15,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Identify the base case and the recursive case for Factorielle(N).'},
        {'numero': 2, 'points': 4, 'enonce': 'Write the recursive function Factorielle(N: Integer): Integer.'},
        {'numero': 3, 'points': 3, 'enonce': 'Trace the recursive call stack for N=4.'},
    ],
    'indices': ['Base case: Factorielle(0) = 1', 'Recursive case: Factorielle(N) = N * Factorielle(N-1)', 'Each call must reduce N to guarantee termination'],
    'conseil_pedagogique': 'Master the base case / recursive case decomposition before tracing the call stack.',
    '_solution_officielle': '',
    '_safe_template': True,
}
SAFE_TEMPLATES['M07'] = {
    'titre': 'Exercise — Matrices',
    'contexte': 'A matrix M of size 3x3 contains integers. Compute the sum of its diagonal elements.',
    'algorithme_fourni': None,
    'difficulte_reelle': 3,
    'duree_estimee_minutes': 15,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Declare the matrix M: Array[0..2, 0..2] of Integer.'},
        {'numero': 2, 'points': 4, 'enonce': 'Write the algorithm that sums the diagonal elements M[i,i].'},
        {'numero': 3, 'points': 3, 'enonce': 'Extend the algorithm to also compute the sum of all elements of M.'},
    ],
    'indices': ['Use nested For loops for rows and columns', 'Diagonal elements satisfy i = j', 'Initialise Sum ← 0 before any loop'],
    'conseil_pedagogique': 'Master double-indexed access M[i,j] and nested loops.',
    '_solution_officielle': '',
    '_safe_template': True,
}
SAFE_TEMPLATES['M08'] = {
    'titre': 'Exercise — Formal Logic',
    'contexte': 'A program must evaluate the truth value of a compound logical proposition (P AND Q) OR (NOT P).',
    'algorithme_fourni': None,
    'difficulte_reelle': 3,
    'duree_estimee_minutes': 12,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Declare boolean variables P and Q.'},
        {'numero': 2, 'points': 4, 'enonce': 'Write the algorithm that evaluates (P AND Q) OR (NOT P) and displays the result.'},
        {'numero': 3, 'points': 3, 'enonce': 'Build the truth table for the proposition for all combinations of P and Q.'},
    ],
    'indices': ['Boolean operators: ET (AND), OU (OR), NON (NOT)', 'Evaluate sub-expressions before combining them', 'A truth table has 2^n rows for n boolean variables'],
    'conseil_pedagogique': 'Master boolean operators and operator precedence before combining propositions.',
    '_solution_officielle': '',
    '_safe_template': True,
}
SAFE_TEMPLATES['M09'] = {
    'titre': 'Exercise — Complexity & Sorting Algorithms',
    'contexte': 'An array T of N integers must be sorted in ascending order using the Bubble Sort algorithm.',
    'algorithme_fourni': None,
    'difficulte_reelle': 4,
    'duree_estimee_minutes': 18,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Write the Bubble Sort algorithm for array T of size N.'},
        {'numero': 2, 'points': 4, 'enonce': 'Trace the algorithm execution for T = [5, 2, 4, 1].'},
        {'numero': 3, 'points': 3, 'enonce': 'Give the worst-case time complexity of Bubble Sort and justify it.'},
    ],
    'indices': ['Compare adjacent elements and swap if out of order', 'Repeat the pass N-1 times in the worst case', 'Worst case complexity is O(N²)'],
    'conseil_pedagogique': 'Master comparison/swap logic and basic complexity analysis (O(N²)).',
    '_solution_officielle': '',
    '_safe_template': True,
}
SAFE_TEMPLATES['M10'] = {
    'titre': 'Exercise — Graphs',
    'contexte': 'A graph G represents 5 cities connected by direct roads, stored as an adjacency matrix.',
    'algorithme_fourni': None,
    'difficulte_reelle': 5,
    'duree_estimee_minutes': 20,
    'kappa': 0.95,
    'questions': [
        {'numero': 1, 'points': 3, 'enonce': 'Represent the graph G using an adjacency matrix Array[0..4, 0..4] of Boolean.'},
        {'numero': 2, 'points': 4, 'enonce': 'Write an algorithm that computes the degree (number of neighbours) of a given vertex.'},
        {'numero': 3, 'points': 3, 'enonce': 'Write an algorithm that lists all vertices directly connected to a given vertex.'},
    ],
    'indices': ['Adjacency matrix entry M[i,j] = True means an edge exists between i and j', 'The degree of vertex i is the number of True values in row i', 'Iterate over the row corresponding to the vertex'],
    'conseil_pedagogique': 'Master adjacency matrix representation before tackling traversal algorithms.',
    '_solution_officielle': '',
    '_safe_template': True,
}

for _mid in MODULE_ORDER:
    if _mid not in SAFE_TEMPLATES:
        SAFE_TEMPLATES[_mid] = {
            'titre': f"Basic Exercise — {MODULES.get(_mid, {}).get('title', _mid)}",
            'contexte': f"Generic fallback exercise for module {_mid} ({MODULES.get(_mid, {}).get('title', '?')}) — no dedicated safe template yet.",
            'algorithme_fourni': None,
            'difficulte_reelle': 2,
            'duree_estimee_minutes': 12,
            'questions': [
                {'numero': 1, 'points': 3, 'enonce': f"Recall the key concept of module {_mid}."},
                {'numero': 2, 'points': 4, 'enonce': 'Write a short algorithm illustrating this concept.'},
                {'numero': 3, 'points': 3, 'enonce': 'Trace your algorithm on a simple example.'},
            ],
            'indices': ['Review the module lecture notes.'],
            'conseil_pedagogique': f'Review the fundamentals of {MODULES.get(_mid, {}).get("title", _mid)}.',
            '_solution_officielle': '',
            '_safe_template': True,
        }

def _verify_exercise(ex: dict, expected_difficulty: int, module_id: str) -> dict:
    """
    Verifies a LLM-generated exercise.
    Returns: {'kappa': float, 'flags': list, 'fixed': dict}
    """
    flags    = []
    score    = 1.0
    ex_fixed = dict(ex)

    missing_keys = REQUIRED_KEYS - set(ex.keys())
    if missing_keys:
        flags.append(f"MISSING_KEY:{missing_keys}")
        score -= 0.30

    questions = ex.get('questions', [])
    if not questions:
        flags.append("EMPTY_QUESTIONS")
        score -= 0.40
    else:
        for i, q in enumerate(questions):
            missing_q = REQUIRED_QUESTION_KEYS - set(q.keys())
            if missing_q:
                flags.append(f"Q{i+1}_MISSING_KEY:{missing_q}")
                score -= 0.10

    if questions:
        total_pts = sum(q.get('points', 0) for q in questions)
        if total_pts != 10:
            flags.append(f"TOTAL_POINTS:{total_pts}!=10")
            score -= 0.15

            if len(questions) == 3:
                ex_fixed['questions'][0]['points'] = 3
                ex_fixed['questions'][1]['points'] = 4
                ex_fixed['questions'][2]['points'] = 3
                flags.append("POINTS_AUTO_CORRECTED:3-4-3")
            elif len(questions) == 2:
                ex_fixed['questions'][0]['points'] = 5
                ex_fixed['questions'][1]['points'] = 5
            score += 0.10

    texte_complet = (
        str(ex.get('titre', '')) + ' ' +
        str(ex.get('contexte', '')) + ' ' +
        ' '.join(str(q.get('enonce', '')) for q in questions)
    ).lower()

    mots_interdits_trouves = [m for m in CONCEPTS_INTERDITS if m in texte_complet]
    if mots_interdits_trouves:
        flags.append(f"FORBIDDEN_CONCEPTS:{mots_interdits_trouves[:3]}")
        score -= 0.25 * len(mots_interdits_trouves[:2])

    diff_reelle = ex.get('difficulte_reelle', 2)
    try:
        diff_reelle = int(diff_reelle)
    except (ValueError, TypeError):
        diff_reelle = 2
        ex_fixed['difficulte_reelle'] = 2

    if abs(diff_reelle - expected_difficulty) > 1:
        flags.append(f"INCONSISTENT_DIFFICULTY:{diff_reelle}!={expected_difficulty}")
        score -= 0.10
        ex_fixed['difficulte_reelle'] = expected_difficulty

    kappa = round(max(0.0, min(1.0, score)), 3)
    ex_fixed['kappa'] = kappa

    return {'kappa': kappa, 'flags': flags, 'fixed': ex_fixed}

_TEACHER_FLAGS = []

def _flag_teacher(module_id: str, difficulty: int, kappa: float,
                  flags: list, raw_ex: dict):
    """Records an incident for teacher review."""
    incident = {
        'timestamp':  datetime.now().isoformat(),
        'module_id':  module_id,
        'difficulty': difficulty,
        'kappa':      kappa,
        'flags':      flags,
        'titre':      raw_ex.get('titre', '?'),
    }
    _TEACHER_FLAGS.append(incident)
    _exp_log(
        f"TEACHER FLAG | kappa={kappa:.2f} | {flags[:2]} | '{raw_ex.get('titre','?')[:40]}'",
        'warn'
    )

def generate_exercise_rag(
    module_id: str,
    difficulty: int,
    error_type: str      = 'general',
    weak_concepts: list  = None,
    shap_features: list  = None,
    cl_value: float      = 0.5,
    seen_exercises: list = None,
    max_retries: int     = 3,
) -> dict:
    weak_concepts  = weak_concepts  or ['algorithmic_methodology']
    shap_features  = shap_features  or []
    seen_exercises = seen_exercises or []

    module_info = MODULES.get(module_id, {})
    mod_title   = module_info.get('title', module_id)

    SHAP_DIRECTIVES = {
        'head_movement':      "The learner has a low level. Build an INTRODUCTORY exercise with very guided steps.",
        'gaze_direction':     "Negative trend. Build a CONSOLIDATION exercise covering the basics.",
        'interaction_events': "The learner repeats the same mistake. Build an exercise that PRECISELY TARGETS this error.",
        'response_time':      "The learner hesitates. Build a SIMPLER exercise to restore confidence.",
    }
    shap_directive = ' | '.join(
        SHAP_DIRECTIVES[f] for f in shap_features if f in SHAP_DIRECTIVES
    ) or "Build an exercise adapted to the detected level."

    fw = get_fusion_weights(len(seen_exercises), cl_value)
    n_hints_eq4 = 3 if (cl_value > 0.50 or difficulty == 1) else 2
    if cl_value >= 0.80:
        cl_directive = f"⚠️ Very high CL (schedule={fw['schedule']}). Short, simple exercise with {n_hints_eq4} mandatory hints. Reduce difficulty by one level."
    elif cl_value >= 0.60:
        cl_directive = f"Moderately high CL (schedule={fw['schedule']}). Add {n_hints_eq4} hints and guide the learner."
    elif cl_value < 0.30:
        cl_directive = f"Very low CL (schedule={fw['schedule']}). You may slightly increase complexity."
    else:
        cl_directive = f"Normal CL (schedule={fw['schedule']}). Standard difficulty."

    concepts_str = ', '.join(weak_concepts[:4]) if weak_concepts else 'general algorithms'

    try:
        rag_context = build_rag_context(module_id, n_docs=2)
    except Exception as e:
        _exp_log(f'RAG error: {e}', 'warn')
        rag_context = ""

    system_prompt = EXP_EXERCISE_SYS

    user_prompt = (
        f"TARGET MODULE: {module_id} — {mod_title}\n"
        f"DIFFICULTY: {difficulty}/3\n"
        f"PREVIOUS ERROR TYPE: {error_type}\n"
        f"WEAK CONCEPTS TO TARGET: {concepts_str}\n"
        f"SHAP DIRECTIVE: {shap_directive}\n"
        f"COGNITIVE LOAD DIRECTIVE: {cl_directive}\n"
        + (f"\nBAC EXAMPLES (inspiration only):\n{rag_context}" if rag_context else "")
        + f"""

RULES:
1. NEW context DIFFERENT from previous exercises.
2. Questions targeting EXACTLY the weak concepts: {concepts_str}
3. Tunisian Bac pseudo-code ONLY (←, If/Then/EndIf, For/EndFor, While/EndWhile).
4. 3 progressive questions (comprehension → application → synthesis).
5. If difficulty=1 or high CL: 3 mandatory hints.
6. conseil_pedagogique must mention the targeted weak concept.

Return ONLY this JSON (no text before or after):
{{
  "titre": "<unique title>",
  "contexte": "<original scenario, 3-4 sentences>",
  "algorithme_fourni": "<pseudo-code if relevant, otherwise null>",
  "difficulte_reelle": {difficulty},
  "duree_estimee_minutes": <10-20>,
  "questions": [
    {{"numero": 1, "points": 3, "enonce": "<question on {weak_concepts[0] if weak_concepts else 'the main concept'}>" }},
    {{"numero": 2, "points": 4, "enonce": "<application question>"}},
    {{"numero": 3, "points": 3, "enonce": "<synthesis question>"}}
  ],
  "indices": ["<hint 1>", "<hint 2>", "<hint 3>"],
  "conseil_pedagogique": "<advice mentioning the targeted weak concept>"
}}"""
    )

    for attempt in range(1, max_retries + 1):
        EXP_STATE['n_gen'] += 1
        if attempt > 1:
            EXP_STATE['n_retry'] += 1
            _exp_log(f'Attempt {attempt}/{max_retries}…', 'warn')

        raw = call_mistral(system_prompt, user_prompt, max_tokens=2000)
        ex  = _parse_mistral_json(raw)

        if ex is None:
            _exp_log(f'JSON parsing failed attempt {attempt}', 'error')
            continue
        if not ex.get('titre') or not ex.get('questions'):
            _exp_log('Incomplete exercise', 'warn')
            continue

        verif = _verify_exercise(ex, difficulty, module_id)
        kappa = verif['kappa']
        flags = verif['flags']
        ex    = verif['fixed']

        if flags:
            _exp_log(f'Verification: kappa={kappa:.2f} | flags={flags[:2]}', 'warn')

        if kappa < 0.60:
            _flag_teacher(module_id, difficulty, kappa, flags, ex)
            safe = SAFE_TEMPLATES.get(module_id, SAFE_TEMPLATES.get('M04'))
            EXP_STATE['n_fallback'] += 1
            _exp_log(f'kappa={kappa:.2f} < 0.60 → safe template activated', 'warn')
            return dict(safe)

        if kappa < 0.70:
            _flag_teacher(module_id, difficulty, kappa, flags, ex)

        if _is_too_similar(ex, seen_exercises, threshold=0.60):
            continue

        if attempt == 1:
            EXP_STATE['n_first_ok'] += 1
        _exp_log(f'Exercise OK: kappa={kappa:.2f} | "{ex.get("titre","?")}" (diff={difficulty})', 'ok')
        try:
            sol_prompt = (
                f"You are a Tunisian Bac Computer Science teacher.\n"
                f"Exercise: {json.dumps(ex, ensure_ascii=False)}\n"
                f"Give the COMPLETE solution question by question.\n"
                f'JSON format: {{"solution_complete": "complete Bac pseudo-code"}}'
            )
            raw_sol = call_mistral("You are an expert examiner.", sol_prompt, max_tokens=1200)
            sol = _parse_mistral_json(raw_sol)
            if sol:
                ex['_solution_officielle'] = sol.get('solution_complete', '')
        except Exception as e:
            ex['_solution_officielle'] = ''
        return ex

    _exp_log(f'Generation failed after {max_retries} attempts', 'error')
    return None

                                                                                    
GEN_REQUESTS = []
_generate_exercise_rag_impl = generate_exercise_rag
def generate_exercise_rag(*a, **k):
    keys = ['n_gen', 'n_first_ok', 'n_retry', 'n_fallback']
    b = {x: EXP_STATE.get(x, 0) for x in keys}
    out = _generate_exercise_rag_impl(*a, **k)
    d = {x: EXP_STATE.get(x, 0) - b[x] for x in keys}
    outcome = ('failed_none' if out is None else 'fallback' if d['n_fallback'] > 0
               else 'first_attempt' if d['n_first_ok'] > 0 else 'after_retry')
    GEN_REQUESTS.append(dict(module=(a[0] if a else k.get('module_id')), attempts=d['n_gen'], outcome=outcome))
    return out
generate_exercise_rag._wrapped = True

def _build_exercise_html(ex: dict, st: dict) -> str:
    lid      = st['id']
    mid      = st['current_module']
    cl       = st['mean_cl']
    ex_idx   = st['exercise_index']
    mastered = ' → '.join(st['mastered_modules']) if st['mastered_modules'] else 'none'
    cl_color = '#27ae60' if cl < 0.30 else '#e67e22' if cl < 0.60 else '#e74c3c'
    cl_state, _ = classify_cl(cl)

    questions_html = ''
    for i, q in enumerate(ex.get('questions', [])):
        questions_html += (
            f"<div style='margin:10px 0;'>"
            f"<p style='margin:0 0 6px;font-size:14px;'>"
            f"<strong>{q.get('numero', i+1)})</strong> {q.get('enonce', '')}"
            f"<span style='color:#888;font-size:12px;'> &nbsp;({q.get('points', '?')} pts)</span>"
            f"</p></div>"
        )

    code_html = ''
    alg = ex.get('algorithme_fourni') or ex.get('algorithm_fourni')
    if alg and str(alg).strip() not in ('', 'null', 'None'):
        code_html = (
            "<div style='margin:12px 0;'>"
            "<p style='margin:0 0 6px;font-weight:bold;font-size:14px;'>Provided algorithm:</p>"
            "<div style='background:#f8f8f8;border:1px solid #ddd;border-radius:4px;"
            "padding:12px 16px;font-family:Consolas,monospace;font-size:13px;"
            f"white-space:pre;line-height:1.6;color:#222;'>{alg}</div>"
            "</div>"
        )

    hints_html = ''
    if cl >= 0.60 and ex.get('indices'):
        items = ''.join(
            f"<li style='margin:3px 0;font-size:13px;'>{h}</li>"
            for h in ex['indices']
        )
        hints_html = (
            "<div style='margin-top:14px;padding:10px 14px;background:#fffbea;"
            "border:1px solid #f0c040;border-radius:4px;'>"
            "<strong style='font-size:13px;'>💡 Hints:</strong>"
            f"<ul style='margin:6px 0 0 16px;padding:0;'>{items}</ul>"
            "</div>"
        )

    kappa = 0.0
    try: kappa = float(ex.get('kappa', 0))
    except: pass

    return (
        "<div style='font-family:Georgia,serif;max-width:760px;margin:0 auto;"
        "background:white;border:1px solid #ccc;border-radius:6px;overflow:hidden;'>"
        "<div style='background:#1a3a6b;color:white;padding:10px 18px;'>"
        f"<div style='font-size:15px;font-weight:bold;'>{ex.get('titre', 'Exercise')}</div>"
        f"<div style='font-size:11px;opacity:0.80;margin-top:3px;'>Difficulty {ex.get('difficulte_reelle', '?')} / 3"
        f" &nbsp;|&nbsp; ⏱ {ex.get('duree_estimee_minutes', '?')} min &nbsp;|&nbsp; κ = {kappa:.2f}</div>"
        "</div>"
        "<div style='background:#f0f4f8;padding:6px 18px;font-size:12px;color:#444;border-bottom:1px solid #ddd;'>"
        f"👤 <strong>{lid}</strong> &nbsp;|&nbsp; Module: <strong>{mid}</strong>"
        f" &nbsp;|&nbsp; Exercise no. <strong>{ex_idx}</strong>"
        f" &nbsp;|&nbsp; Mastered: <em>{mastered}</em>"
        f" &nbsp;|&nbsp; CL: <strong style='color:{cl_color};'>{cl:.3f}</strong> [{cl_state}]"
        "</div>"
        "<div style='padding:16px 22px;'>"
        "<p style='margin:0 0 4px;font-weight:bold;font-size:14px;'>Context:</p>"
        f"<p style='margin:0 0 14px;font-size:14px;line-height:1.6;text-align:justify;'>{ex.get('contexte', '')}</p>"
        f"{code_html}"
        "<p style='margin:0 0 8px;font-weight:bold;font-size:14px;'>❓ Questions (total: 10 pts):</p>"
        f"{questions_html}"
        f"{hints_html}"
        "<div style='margin-top:14px;padding:8px 12px;background:#f5f5f5;"
        "border-left:3px solid #aaa;border-radius:3px;font-size:12px;color:#555;'>"
        f"<em>💬 <strong>Focus:</strong> {ex.get('conseil_pedagogique', '')}</em>"
        "</div>"
        "</div>"
        "<div style='padding:10px 18px;background:#fff3e0;border-top:2px solid #e65100;text-align:center;'>"
        "<div style='font-weight:bold;color:#e65100;font-size:13px;margin-bottom:6px;'>"
        "⚠️ After answering below, also complete Form 2:</div>"
        f"<a href='{FORM2_URL}' target='_blank' style='display:inline-block;padding:8px 24px;background:#e65100;"
        "color:white;text-decoration:none;border-radius:6px;font-weight:bold;font-size:13px;'>"
        "📋 Form 2 — Self-assessment →</a>"
        f"<div style='font-size:11px;color:#666;margin-top:5px;'>Module: <strong>{mid}</strong> | Exercise no. <strong>{ex_idx}</strong></div>"
        "</div>"
        "</div>"
    )

def _generate_exercise_for_student(st: dict):
    lid      = st['id']
    mid      = st['current_module']
    cl       = st['mean_cl']
    strategy = st.get('next_strategy', 'PROGRESSION_NORMALE')

    history    = st.get('exercise_history', [])
    error_type = history[-1].get('error_type', 'general') if history else 'general'
    last_score = history[-1].get('score', 0.5) if history else st.get('prior', 0.5)

    if strategy == 'MEME_FAUTE_EXERCICE_DIFFERENT' or cl >= 0.80 or last_score < 0.20:
        difficulty = 1
    elif strategy == 'RENFORCEMENT_SIMPLE' or cl >= 0.60:
        difficulty = 1
    elif strategy == 'PROGRESSION_NORMALE':
        difficulty = 2
    elif strategy == 'AVANCEMENT_OU_MAITRISE':
        difficulty = min(3, st.get('ex_count_module', 0) // 2 + 1)
    else:
        difficulty = 2

    weak_concepts  = get_weak_concepts(st)
    shap_features  = st.get('last_shap_features', [])
    seen_exercises = st.get('seen_exercises', [])

    _exp_log(
        f'{lid} | mod={mid} | diff={difficulty} | cl={cl:.3f} | '
        f'strategie={strategy} | erreur={error_type} | concepts={weak_concepts[:2]}',
        'exp',
    )

    ex = generate_exercise_rag(
        module_id      = mid,
        difficulty     = difficulty,
        error_type     = error_type,
        weak_concepts  = weak_concepts,
        shap_features  = shap_features,
        cl_value       = cl,
        seen_exercises = seen_exercises,
        max_retries    = 3,
    )

    if ex is None:
        EXP_STATE['current_exercise_html'] = (
            "<p style='color:#e74c3c;padding:20px;text-align:center;'>"
            "Impossible de generer un exercice. Reessayez.</p>"
        )
        return

    st['exercise_index']  = st.get('exercise_index', 0) + 1
    st['ex_count_module'] = st.get('ex_count_module', 0) + 1
    st['last_exercise']   = ex
    st['waiting_form2']   = False                                                                     
    st['pending_exercise'] = None

    st['seen_exercises'].append(ex)

    EXP_STATE['current_exercise']      = ex
    EXP_STATE['current_learner_id']    = lid
    EXP_STATE['waiting_answer']        = True
    EXP_STATE['correction_shown']      = False
    EXP_STATE['answer_analysis']       = ''
    EXP_STATE['shap_message']          = ''
    EXP_STATE['current_exercise_html'] = _build_exercise_html(ex, st)

    _exp_log(
        f'{lid} | Ex #{st["exercise_index"]} pret : "{ex.get("titre","?")}" '
        f'(diff={difficulty}, strategie={strategy}, CL={cl:.3f})',
        'ok'
    )

def _handle_form1(raw_row: dict):
    row = _clean_row(raw_row)
    lid = _detect_lid(row)
    if lid in EXP_STATE['processed_f1']:
        return
    EXP_STATE['processed_f1'].add(lid)

    def _pv(keys, default=0.5):
        for k, v in row.items():
            if any(p in k for p in keys):
                try:
                    n = float(parse_likert(v))
                    return round((n - 1) / 4., 4) if n > 1.0 else round(n, 4)
                except: pass
        return default

    prior = _pv(['prior', 'knowledge', 'already', 'connaissance'])
    conf  = _pv(['confidence', 'confiance', 'confident'])
    motiv = _pv(['motivation', 'motivat'])
    group = str(row.get('current_educational_level', row.get('group', 'Bac')))

    st = _init_learner_state(lid, prior, conf, motiv, group)
    _exp_log(f'Form 1: {lid} | prior={prior:.2f} | start={st["current_module"]}', 'exp')
    _generate_exercise_for_student(st)

def _handle_form2(raw_row: dict):
    """Traite UNE soumission de Form 2 (questionnaire post-exercice).
    Responsabilités : valider le questionnaire → CLq brut (compute_clq10) → lissage séparé →
    mise à jour de l'état → maîtrise / exercice suivant.
    NE ré-enregistre PAS le score ni l'événement d'exercice (faits une seule fois par cb_submit_answer).
    La clé n'entre dans processed_f2 qu'APRÈS un traitement réussi."""
    row = _clean_row(raw_row)
    lid = _detect_lid(row)
    if lid not in EXP_STATE['learners']:
        _exp_log(f'Form 2 from {lid} without Form 1 — ignored', 'warn')
        return

    st     = EXP_STATE['learners'][lid]
    ex_idx = st['exercise_index']
    key    = (str(lid), ex_idx)
    if key in EXP_STATE['processed_f2']:
        _exp_log(f'Form 2 {lid} ex#{ex_idx}: doublon — ignoré (aucun double comptage)', 'warn')
        return
    pend = st.get('pending_exercise')
    if not st.get('waiting_form2', False) or not pend or pend.get('exercise_index') != ex_idx:
        _exp_log(f'Form 2 {lid}: aucun exercice en attente de questionnaire — état inchangé', 'warn')
        return

                                                                                            
    clq_version = detect_clq10_version(row)
    ok, vec, reason = validate_clq10_answers(extract_clq10_answers(row, clq_version))
    if not ok and clq_version is None: reason = 'version du questionnaire non reconnue (intitulés inconnus) ; ' + reason
    if not ok:
        _exp_log(f'Form 2 {lid}: questionnaire rejeté, CLq NON mis à jour — {reason}', 'warn')
        return                                                                                       
    clq_raw = round(compute_clq10(vec), 4)
    cl_state, cl_action = classify_cl(clq_raw)

                                                                           
    st['clq_raw']     = clq_raw
    st['mean_cl']     = round(st['mean_cl'] * 0.7 + clq_raw * 0.3, 4)
    st['cl_smoothed'] = st['mean_cl']
    st['module_cls'].append(clq_raw)
    rec = st['exercise_history'][-1]
    st['clq_version'] = clq_version
    rec.update({'cl': clq_raw, 'clq_raw': clq_raw, 'cl_smoothed': st['cl_smoothed'], 'clq_version': clq_version})

    score_raw, score_norm, error_type = pend['score_raw'], pend['score_norm'], pend['error_type']
    shap_msg = get_shap_explanation(error_type, clq_raw, score_norm, st['current_module'])
    EXP_STATE['shap_message'] = shap_msg
    _exp_log(f'Form 2: {lid} | CLq_brut={clq_raw:.3f} ({clq_version}) | CL_lissé={st["cl_smoothed"]:.3f} | '
             f'Score={score_raw}/10 (enregistré à la soumission)', 'exp')

    mastered_now = _check_mastery(st, mid=st.get('current_module'))
    if st['ex_count_module'] >= EXP_MAX_EXERCISES and not mastered_now:
        mastered_now = True
        _exp_log(f'{lid}: max exercises reached → forced progression', 'warn')

    mid = st['current_module']
    st['module_progress'][mid] = {
        'score_pre':      round(st['exercise_history'][0]['score'], 3) if st['exercise_history'] else 0.5,
        'score_post':     round(score_norm, 3),
        'score_current':  round(float(np.mean(st['module_scores'])), 3),
        'mean_cl':        round(float(np.mean(st['module_cls'])), 3),
        'is_mastered':    mastered_now,
        'nb_exercises':   st['ex_count_module'],
        'dominant_error': error_type,
    }

    _write_result_to_sheet({
        'timestamp': datetime.now().isoformat(), 'learner_id': str(lid),
        'module_id': mid, 'exercise_index': ex_idx,
        'exercise_title': pend.get('title', ''), 'difficulty': pend.get('difficulty', ''),
        'cl_value': clq_raw, 'cl_state': cl_state, 'score_raw': score_raw,
        'score_norm': score_norm, 'mastered': mastered_now,
        'next_action': 'NEXT_MODULE' if mastered_now else 'NEXT_EXERCISE',
        'error_type': error_type, 'shap_message': shap_msg[:200], 'recommendation': cl_action,
        'clq_raw': clq_raw, 'cl_smoothed': st['cl_smoothed'],
    })

                                                                                  
    st['waiting_form2']    = False
    st['pending_exercise'] = None
    EXP_STATE['processed_f2'].add(key)

    if mastered_now:
        _exp_log(f'{lid} has MASTERED {mid}!', 'ok')
        st['mastered_modules'].append(mid)
        st['module_scores'] = []
        st['module_cls']    = []
        st['ex_count_module'] = 0
        _recommend_next_module_display(st, clq_raw, cl_state)
    else:
        _generate_exercise_for_student(st)

def _recommend_next_module_display(st: dict, cl_value: float, cl_state: str):
    lid = st['id']
    tmp = {
        'id': lid, 'username': str(lid), 'group': st['group'],
        'prior_knowledge': st['prior'], 'confidence': st['confidence'],
        'motivation': st['motivation'], 'mean_cl': st['mean_cl'],
        'mean_score': float(np.mean([v['score_current'] for v in st['module_progress'].values()])) if st['module_progress'] else st['prior'],
        'mean_head_movement': 0.5, 'mean_gaze_disp': 0.5,
        'mean_interaction_ev': cl_value, 'mean_response_time': cl_value,
        'mastered_modules': list(st['mastered_modules']),
        'module_progress': st['module_progress'],
    }
    n_inter = sum(v.get('nb_exercises', 0) for v in st['module_progress'].values())
    res_cf  = recommend_next_module(tmp, DB_LEARNERS, cl_value, n_interactions=n_inter)

    if not res_cf:
        EXP_STATE['current_exercise_html'] = (
            "<div style='max-width:800px;margin:20px auto;padding:24px;background:#e8f5e9;"
            "border:3px solid #2e7d32;border-radius:12px;text-align:center;'>"
            f"<h2 style='color:#1b5e20;'>🎓 Congratulations, {lid}!</h2>"
            "<p>Baccalaureate Computer Science curriculum completed!</p>"
            f"<p><strong>{len(st['mastered_modules'])}/{len(MODULES)} modules mastered</strong></p>"
            "</div>"
        )
        EXP_STATE['correction_shown'] = False
        _exp_log(f'{lid}: Curriculum COMPLETED!', 'ok')
        return

    next_mid   = res_cf['recommended_module']
    next_title = res_cf['recommended_title']
    cf_score   = res_cf['final_score']
    schedule   = res_cf.get('schedule', '')
    expl       = explain_recommendation(tmp, res_cf, cl_value, cl_state)

                                                                                            
                                                                                     
    _prev_expl = EXP_STATE.get('shap_message', '')
    _cf_block = (
        f"**Recommandation de module (CF)** — {next_mid}\n"
        f"• CF Schedule: **{schedule}**\n• CF Score: **{cf_score:.4f}**\n"
        f"• Collaborative vote: **{res_cf['vote_cf']:.4f}**\n• Compatibility: **{res_cf['compat_score']:.4f}**"
    )
    EXP_STATE['shap_message'] = (_prev_expl + "\n\n---\n" + _cf_block) if _prev_expl else (
        "*(Explication SHAP indisponible pour ce changement de module — détail CF uniquement)*\n\n" + _cf_block)
    EXP_STATE['current_exercise_html'] = (
        "<div style='font-family:Arial;max-width:800px;margin:16px auto;'>"
        "<div style='padding:20px;background:#1565c0;color:white;border-radius:12px;margin-bottom:12px;'>"
        f"<h3 style='margin:0 0 10px;'>🏆 Module {st['mastered_modules'][-1]} Mastered!</h3>"
        "<div style='background:rgba(255,255,255,0.15);padding:12px;border-radius:8px;'>"
        "<strong>📚 Next module recommended by CF:</strong><br/>"
        f"<span style='font-size:18px;font-weight:bold;'>{next_mid} — {next_title}</span><br/>"
        f"<small>CF Score: {cf_score:.4f} | Schedule: {schedule} | CL: {cl_value:.3f}</small>"
        "</div></div>"
        "<div style='padding:16px;background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;'>"
        "<strong>💬 AI Justification (3 parts — prerequisites / peers / CL):</strong>"
        f"<div style='margin-top:8px;white-space:pre-line;font-size:13px;'>{expl}</div>"
        "</div>"
        "<div style='margin-top:10px;padding:10px;background:#e8f5e9;border-radius:8px;"
        "text-align:center;font-size:13px;color:#2e7d32;'>"
        f"✅ Click <strong>➡️ Next exercise</strong> to start module {next_mid}"
        "</div></div>"
    )

    st['current_module']  = next_mid
    st['ex_count_module'] = 0
    EXP_STATE['correction_shown'] = True
    EXP_STATE['last_mastered']    = True
    DB_LEARNERS.append({**tmp, 'is_synthetic': False})
    _exp_log(f'{lid} → {next_mid} | CF={cf_score:.4f}', 'ok')

def _run_experiment_loop():
    _exp_log('Monitoring started', 'ok')
    try:
        EXP_STATE['last_f1_count'] = len(sh.worksheet(TAB_FORM_ONBOARDING).get_all_values())
        EXP_STATE['last_f2_count'] = len(sh.worksheet(TAB_FORM_POSTEX).get_all_values())
    except Exception as ex:
        _exp_log(f'Error initialising counters: {ex}', 'error')

    while not _stop_event.is_set():
        try:
            for raw in _fetch_new_sheet_rows(TAB_FORM_ONBOARDING, 'last_f1_count'):
                _exp_log('New Form 1 response!', 'exp')
                _handle_form1(raw)
            for raw in _fetch_new_sheet_rows(TAB_FORM_POSTEX, 'last_f2_count'):
                _exp_log('New Form 2 response!', 'exp')
                _handle_form2(raw)
        except Exception as ex:
            _exp_log(f'Loop error: {ex}', 'error')
        _stop_event.wait(EXP_POLL_INTERVAL)

    _exp_log('Monitoring stopped.', 'warn')

MOCK_F1 = {
    'Learner ID (provided by the experimenter)': 'test.student@gmail.com',
    'Email Address':                             'test.student@gmail.com',
    'current_educational_level':                 'Senior Year',
    'i_already_have_prior_knowledge_in_this_topic':                        '2',
    'i_am_confident_in_my_ability_to_solve_exercises_in_this_subject':    '3',
    'i_feel_motivated_when_solving_exercises_in_this_subject':             '4',
}
MOCK_F2 = {                                                                                                      
    'Email Address':                                                'test.student@gmail.com',
    'Exercise ID':                                                 '1',
    'This exercise required a high level of mental effort.':       '4 — Neutral',
    'I found this exercise difficult.':                            '3 — Slightly disagree',
    'I felt hesitant or unsure while solving this exercise.':      '3 — Slightly disagree',
    'I had to reread the question or instructions several times.': '2 — Disagree',
    'I felt under time pressure while solving this exercise.':     '2 — Disagree',
    'The instructions were clear.':                                '6 — Agree',
    'I felt confident about my answer.':                           '5 — Slightly agree',
    'I felt frustrated while working on this exercise.':           '2 — Disagree',
    'I had difficulty identifying what was being asked.':          '2 — Disagree',
    'I felt that the amount of information was overwhelming.':     '3 — Slightly disagree',
}

def get_stats() -> str:
    n_active   = sum(1 for s in EXP_STATE['learners'].values() if s.get('waiting_form2'))
    n_mastered = sum(len(s['mastered_modules']) for s in EXP_STATE['learners'].values())
    status     = "🟢 Running" if EXP_STATE['active'] else "⏸ Stopped"
    return (
        f"**{status}** &nbsp;|&nbsp; 👤 Learners: **{len(EXP_STATE['learners'])}**"
        f" &nbsp;|&nbsp; 📝 On exercise: **{n_active}**"
        f" &nbsp;|&nbsp; ✅ Modules mastered: **{n_mastered}**"
    )

def get_log_text()      -> str: return '\n'.join(EXP_STATE['log'][:30])
def get_shap_text()     -> str: return EXP_STATE.get('shap_message', '')
def get_analysis_text() -> str: return EXP_STATE.get('answer_analysis', '')
def get_exercise_html() -> str:
    return EXP_STATE.get(
        'current_exercise_html',
        "<p style='color:#999;text-align:center;padding:40px;'>⏳ Waiting for the first exercise...</p>",
    )

def cb_start():
    if not EXP_STATE['active']:
        _stop_event.clear()
        EXP_STATE['active'] = True
        t = threading.Thread(target=_run_experiment_loop, daemon=True)
        _exp_thread[0] = t
        t.start()
        _exp_log('▶ Experiment started', 'ok')
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_stop():
    _stop_event.set()
    EXP_STATE['active'] = False
    _exp_log('⏹ Experiment stopped', 'warn')
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_test_f1():
    test_lid = 'test.student@gmail.com'
    EXP_STATE['processed_f1'].discard(test_lid)
    if test_lid in EXP_STATE['learners']:
        del EXP_STATE['learners'][test_lid]
    _exp_log('🧪 Testing Form 1...', 'info')
    _handle_form1(MOCK_F1)
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_test_f2():
    _exp_log('🧪 Testing Form 2...', 'info')
    _handle_form2(MOCK_F2)
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_refresh():
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_export():
    rows = []
    for st in EXP_STATE['learners'].values():
        for rec in st['exercise_history']:
            rows.append({
                'learner_id': st['id'], 'group': st['group'],
                'module': st['current_module'], 'score': rec['score'],
                'cl': rec['cl'], 'difficulty': rec['difficulty'],
                'error_type': rec.get('error_type', ''),
                'mastered_modules': '|'.join(st['mastered_modules']),
            })
    if rows:
        path = '/content/ails_results.csv'
        pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig')
        _exp_log(f'Export → {path} ({len(rows)} rows)', 'ok')
    else:
        _exp_log('No data to export.', 'warn')
    return get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text()

def cb_submit_answer(answer_text: str, learner_id_input: str):
    if not EXP_STATE.get('waiting_answer', False):
        return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(),
                "⚠️ No active exercise.", "", gr.update(), gr.update(), gr.update())

    lid = EXP_STATE.get('current_learner_id') or learner_id_input.strip()
    if not lid or lid not in EXP_STATE['learners']:
        return (get_stats(), get_log_text(), get_exercise_html(), '',
                "⚠️ Learner not recognised.", "", gr.update(), gr.update(), gr.update())

    st = EXP_STATE['learners'][lid]
    ex = EXP_STATE.get('current_exercise')
    if not ex:
        return (get_stats(), get_log_text(), get_exercise_html(), '',
                "⚠️ Exercise not found.", "", gr.update(), gr.update(), gr.update())

    _exp_log(f'Analysing answer for {lid}...', 'exp')
    EXP_STATE['waiting_answer'] = False


    analysis   = analyze_student_answer(answer_text, ex, st['current_module'])
    error_type = analysis.get('type_erreur', 'general')
    score_raw  = analysis.get('score_estime', 0)
    score_norm = round(score_raw / 10., 4)
    resume     = analysis.get('resume_erreur', '')

    corr        = analysis.get('correction_detaillee', {})
    explication = corr.get('explication', '')         if isinstance(corr, dict) else ''
    code_corr   = corr.get('code_corrige', '')         if isinstance(corr, dict) else ''
    points      = corr.get('points_par_question', '')  if isinstance(corr, dict) else ''
    conseil     = corr.get('conseil', '')              if isinstance(corr, dict) else ''

                                                                                                      
                                                                                             
    ex_idx = st['exercise_index']
    st['module_scores'].append(score_norm)
    st['exercise_history'].append({
        'exercise_index': ex_idx, 'score': score_norm, 'cl': None, 'clq_raw': None,
        'cl_smoothed': st.get('cl_smoothed', st['mean_cl']),
        'difficulty': ex.get('difficulte_reelle', 2), 'error_type': error_type,
    })
    st['pending_exercise'] = {
        'exercise_index': ex_idx, 'score_raw': score_raw, 'score_norm': score_norm,
        'error_type': error_type, 'title': ex.get('titre', ''),
        'difficulty': ex.get('difficulte_reelle', ''),
    }
    st['waiting_form2'] = True                                                                            
    if len(DB_LEARNERS) % 5 == 0:
        SHAP_MODEL, SHAP_SCALER, SHAP_EXPLAINER = _train_shap_model(DB_LEARNERS)
        _exp_log('SHAP model updated with real data', 'ok')

    mastered_now = False                                                                          
    shap_msg     = ''                                                                                     
    EXP_STATE['shap_message'] = ''
    mid = st['current_module']
    _exp_log(f'{lid}: score ex#{ex_idx} enregistré ({score_raw}/10) — en attente de Form 2', 'ok')
    next_msg = (f"📝 Score recorded. Please submit the **post-exercise questionnaire (Form 2)** — "
                f"your next exercise is generated once it is processed. *(error: {error_type})*")

    if isinstance(code_corr, dict):
        code_corr = '\n'.join(f"// {k}\n{v}" for k, v in code_corr.items())
    elif not isinstance(code_corr, str):
        code_corr = str(code_corr) if code_corr else ''

    code_lines = '\n'.join('    ' + l for l in code_corr.splitlines()) if code_corr else '    (no correction available)'

    analysis_html = (
        f"**📊 Answer Analysis**\n\n"
        f"• **Estimated score**: {score_raw}/10\n"
        f"• **Error type**: `{error_type}`\n"
        f"• **Summary**: {resume}\n\n"
        f"---\n\n"
        f"**Pedagogical explanation:**\n{explication}\n\n"
        f"**Full correction (Bac pseudo-code):**\n\n"
        f"```\n{code_lines}\n```\n\n"
        f"**Points per question:**\n{points}\n\n"
        f"**Advice:**\n> {conseil}\n\n"
        f"---\n\n{next_msg}"
    )

    EXP_STATE['answer_analysis']  = analysis_html
    EXP_STATE['correction_shown'] = True
    EXP_STATE['last_error_type']  = error_type
    EXP_STATE['last_score_norm']  = score_norm
    EXP_STATE['last_mastered']    = mastered_now

    _exp_log(f'Correction displayed: {lid} | {error_type} | {score_raw}/10', 'ok')
    return (
        get_stats(), get_log_text(), get_exercise_html(), shap_msg, analysis_html,
        "",
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
    )

def cb_next_exercise(learner_id_input: str):
    lid = EXP_STATE.get('current_learner_id') or learner_id_input.strip()
    if not lid or lid not in EXP_STATE['learners']:
        return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(),
                "⚠️ No active learner.", "", gr.update(), gr.update(), gr.update())
    if not EXP_STATE.get('correction_shown', False):
        return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(),
                "⚠️ Please submit your answer first.", "", gr.update(), gr.update(), gr.update())

    st = EXP_STATE['learners'][lid]
    if st.get('waiting_form2', False):
        return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(),
                "⚠️ Please submit the post-exercise questionnaire (Form 2) first.", "",
                gr.update(), gr.update(), gr.update())
    EXP_STATE['correction_shown'] = False

    last_mastered_module = st['mastered_modules'][-1] if st['mastered_modules'] else None
    if EXP_STATE.get('last_mastered', False) and st.get('current_module') != last_mastered_module:
        _exp_log(f'{lid}: module transition in progress...', 'info')
        time.sleep(2)
    elif EXP_STATE.get('waiting_answer', False):
        _exp_log(f'{lid}: next exercise already generated after Form 2', 'info')
    else:
        _exp_log(f'{lid}: generating next exercise (gap={EXP_STATE["last_error_type"]})', 'exp')
        threading.Thread(target=_generate_exercise_for_student, args=(st,), daemon=True).start()
        time.sleep(1)
        EXP_STATE['answer_analysis'] = ''
        EXP_STATE['shap_message']    = ''

    return (
        get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text(),
        "",
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=False),
    )

def cb_student_login(learner_id_input: str):
    lid = learner_id_input.strip()
    if not lid:
        return (get_stats(), get_log_text(),
                "<p style='color:#e74c3c;padding:20px;'>⚠️ Please enter your email address.</p>",
                '', '', '', gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))

    if lid in EXP_STATE['learners']:
        if EXP_STATE.get('current_learner_id') == lid and EXP_STATE.get('current_exercise_html'):
            return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(),
                    get_analysis_text(), '', gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))
        _generate_exercise_for_student(EXP_STATE['learners'][lid])
        return (get_stats(), get_log_text(), get_exercise_html(), '', '',
                '', gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))

    form1_filled = False
    try:
        ws   = sh.worksheet(TAB_FORM_ONBOARDING)
        rows = ws.get_all_records()
        for row in rows:
            row_lid = str(row.get('Learner ID (provided by the experimenter)',
                                   row.get('Email Address', ''))).strip()
            if row_lid == str(lid):
                form1_filled = True
                _handle_form1(row)
                break
    except Exception as ex:
        _exp_log(f'Sheet verification error: {ex}', 'error')

    if not form1_filled:
        html = (
            "<div style='max-width:800px;margin:20px auto;padding:28px;"
            "background:#e3f2fd;border:2px solid #1565c0;border-radius:12px;text-align:center;'>"
            f"<h3 style='color:#1565c0;margin:0 0 12px;'>Welcome, {lid}!</h3>"
            "<p style='font-size:15px;margin:0 0 16px;'>Please first complete the "
            "<strong>profile form</strong> (Form 1):</p>"
            f"<a href='{FORM1_URL}' target='_blank' style='display:inline-block;padding:12px 32px;"
            "background:#1565c0;color:white;text-decoration:none;border-radius:8px;"
            "font-size:16px;font-weight:bold;'>Fill in Form 1 — Initial Profile</a>"
            "<p style='color:#666;font-size:12px;margin:14px 0 0;'>After submitting, come back here "
            "and click <strong>Start</strong>.</p>"
            "</div>"
        )
        return (get_stats(), get_log_text(), html, '', '',
                '', gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))

    time.sleep(1)
    return (get_stats(), get_log_text(), get_exercise_html(), get_shap_text(), get_analysis_text(),
            '', gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))

custom_css = r"""
* { font-family: 'Segoe UI', Arial, sans-serif !important; }
.gradio-container {
    max-width: 1100px;
    margin: 0 auto;
    background: #f4f6fb;
}
.ails-header {
    background: linear-gradient(135deg, #1a237e 0%, #1565c0 100%);
    color: white;
    padding: 22px 30px;
    border-radius: 14px;
    margin-bottom: 18px;
    box-shadow: 0 4px 18px rgba(21,101,192,0.18);
}
.card {
    background: white;
    border-radius: 12px;
    padding: 20px 24px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.07);
    margin-bottom: 16px;
}
button.primary {
    background: #1565c0 !important;
    border-radius: 8px;
    font-weight: 600;
    font-size: 15px;
}
button.primary:hover { background: #0d47a1; }
textarea, input[type=text] {
    border-radius: 8px;
    border: 1.5px solid #c5cae9;
    font-size: 14px;
    background: #fafbff;
}
textarea:focus, input:focus {
    border-color: #1565c0 !important;
    box-shadow: 0 0 0 2px rgba(21,101,192,0.12);
}
"""

with gr.Blocks(title="🎓 AILS", css=custom_css) as demo:

    gr.HTML("""
    <div style='padding:22px 30px; background:linear-gradient(135deg,#1a237e 0%,#1565c0 100%);
                color:white; border-radius:14px; margin-bottom:16px;
                box-shadow:0 4px 18px rgba(21,101,192,0.18);'>
        <div style='display:flex; align-items:center; gap:14px;'>
            <span style='font-size:36px;'>🎓</span>
            <div>
                <h2 style='margin:0; font-size:22px; font-weight:700; letter-spacing:0.3px;'>
                    AILS — Adaptive Learning &amp; Intelligent System
                </h2>
                <p style='margin:4px 0 0; font-size:13px; opacity:0.85;'>
                    Baccalaureate in Computer Science · Tunisia &nbsp;|&nbsp;
                    LLM + RAG + Collaborative Filtering + SHAP
                </p>
            </div>
        </div>
    </div>
    """)

    stats_md = gr.Markdown(value=get_stats(), elem_id="stats_bar")

    gr.HTML("""
    <div style='background:white; border-radius:12px; padding:18px 24px;
                box-shadow:0 2px 10px rgba(0,0,0,0.07); margin-bottom:14px;'>
    """)
    with gr.Row():
        learner_id_box = gr.Textbox(
            label="📧 Email address",
            placeholder="e.g. student@gmail.com",
            scale=4,
        )
        btn_login = gr.Button("▶ Start", variant="primary", scale=1, size="lg")
    gr.HTML("</div>")

    with gr.Accordion("⚙️ Teacher controls", open=False):
        with gr.Row():
            btn_start   = gr.Button("▶ Start",        variant="primary",   size="sm")
            btn_stop    = gr.Button("⏹ Stop",          variant="stop",      size="sm")
            btn_test_f1 = gr.Button("🧪 Test Form 1", variant="secondary", size="sm")
            btn_test_f2 = gr.Button("🧪 Test Form 2", variant="secondary", size="sm")
            btn_refresh = gr.Button("🔄 Refresh",                           size="sm")
            btn_export  = gr.Button("💾 CSV",                               size="sm")

        gr.HTML(f"""
        <div style='background:#e8f4fd; border:1px solid #90caf9; padding:10px 14px;
                    border-radius:8px; font-size:13px; margin-top:8px;'>
            <strong>📤 Links:</strong> &nbsp;
            <a href='{FORM1_URL}' target='_blank' style='color:#1565c0;'>
                📋 Form 1 — Initial Profile
            </a>
            &nbsp;|&nbsp;
            <a href='{FORM2_URL}' target='_blank' style='color:#1565c0;'>
                📋 Form 2 — Self-assessment
            </a>
        </div>
        """)

    gr.HTML("""
    <div style='display:flex; align-items:center; gap:8px; margin:18px 0 8px;'>
        <span style='font-size:20px;'>📝</span>
        <h3 style='margin:0; font-size:17px; font-weight:700; color:#1a237e;'>
            Current exercise
        </h3>
    </div>
    """)
    exercise_html_box = gr.HTML(value=get_exercise_html(), elem_id="exercise_box")

    with gr.Group(visible=True) as answer_group:
        gr.HTML("""
        <div style='display:flex; align-items:center; gap:8px; margin:18px 0 8px;'>
            <span style='font-size:20px;'>✍️</span>
            <h3 style='margin:0; font-size:17px; font-weight:700; color:#1a237e;'>
                Your answer
            </h3>
        </div>
        """)
        answer_box = gr.Textbox(
            placeholder=(
                "Write your answer here...\n"
                "Example:\nVariable N: Integer\nBegin\n  Read N\n  Write N\nEnd"
            ),
            lines=10,
            label="",
            show_label=False,
        )
        btn_submit = gr.Button("📤 Submit my answer", variant="primary", size="lg")

    gr.HTML("""
    <div style='padding:10px 16px; background:#e8f0fe; border-left:4px solid #1565c0;
                border-radius:6px; font-size:12px; color:#333; margin:10px 0;'>
        📌 <strong>Guide:</strong>
        (1) Answer → <strong>📤 Submit</strong> → read the correction
        (2) Click <strong>➡️ Next exercise</strong> to continue
        (3) The system adapts to your gaps until you master the module
        (4) The <strong>inter-module CF</strong> then recommends the next module
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            gr.HTML("""
            <div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                <span style='font-size:18px;'>📊</span>
                <h3 style='margin:0; font-size:16px; font-weight:700; color:#1a237e;'>Correction</h3>
            </div>
            """)
            analysis_md = gr.Markdown(
                value="*Waiting for your answer...*",
                elem_id="analysis_box",
            )
        with gr.Column(scale=2):
            gr.HTML("""
            <div style='display:flex; align-items:center; gap:8px; margin-bottom:6px;'>
                <span style='font-size:18px;'>🔍</span>
                <h3 style='margin:0; font-size:16px; font-weight:700; color:#1a237e;'>
                    Why this exercise?
                </h3>
            </div>
            """)
            shap_md = gr.Markdown(
                value="*Will be displayed after your first answer.*",
                elem_id="shap_box",
            )

    btn_next = gr.Button("➡️ Next exercise", variant="huggingface", size="lg", visible=False)

    with gr.Accordion("📋 Real-time log", open=False):
        log_box = gr.Textbox(value=get_log_text(), lines=8, interactive=False, show_label=False)

    OUTPUTS      = [stats_md, log_box, exercise_html_box, shap_md, analysis_md]
    OUTPUTS_FULL = OUTPUTS + [answer_box, answer_group, btn_submit, btn_next]

    btn_login.click(fn=cb_student_login,  inputs=[learner_id_box],              outputs=OUTPUTS_FULL)
    btn_submit.click(fn=cb_submit_answer, inputs=[answer_box, learner_id_box],  outputs=OUTPUTS_FULL)
    btn_next.click(fn=cb_next_exercise,   inputs=[learner_id_box],              outputs=OUTPUTS_FULL)
    btn_start.click(fn=cb_start,     outputs=OUTPUTS)
    btn_stop.click(fn=cb_stop,       outputs=OUTPUTS)
    btn_test_f1.click(fn=cb_test_f1, outputs=OUTPUTS)
    btn_test_f2.click(fn=cb_test_f2, outputs=OUTPUTS)
    btn_refresh.click(fn=cb_refresh, outputs=OUTPUTS)
    btn_export.click(fn=cb_export,   outputs=OUTPUTS)

demo.launch(share=True, debug=True)

In [ ]:
                                                                                
                                                                                              
                                                                                            
                                                                                
import unittest, contextlib, copy
from unittest import mock

@contextlib.contextmanager
def _patched(**kw):
    g = globals(); old = {k: g.get(k) for k in kw}
    g.update(kw)
    try: yield
    finally: g.update(old)

def _form2_row(answers, lid='t@x.org', version=None):
    """Ligne Form 2 au format du Google Form (« n — libellé »), à partir de 10 entiers 1–7."""
    lab = {1:'Strongly disagree',2:'Disagree',3:'Slightly disagree',4:'Neutral',5:'Slightly agree',6:'Agree',7:'Strongly agree'}
    row = {'Email Address': lid, 'Exercise ID': '1'}
    for (q, title, _), a in zip(CLQ10_VERSIONS[version or CLQ10_ANALYSIS_VERSION], answers):
        row[title] = a if not isinstance(a, int) else f'{a} — {lab[a]}'
    return row

def _fresh_learner(lid='t@x.org'):
    st = _init_learner_state(lid, 0.5, 0.5, 0.5, 'Bac')
    st.update(exercise_index=1, ex_count_module=1, last_exercise={'titre':'Ex','difficulte_reelle':2},
              waiting_form2=False, pending_exercise=None)
    EXP_STATE['learners'] = {lid: st}; EXP_STATE['processed_f2'] = set()
    EXP_STATE.update(current_learner_id=lid, current_exercise={'titre':'Ex','difficulte_reelle':2}, waiting_answer=True)
    return st

class TestCLq(unittest.TestCase):
    def test01_dix_reponses_valides(self):
        a = [4,3,3,2,2,6,5,2,2,3]
        z = [s if i not in (5,6) else 8-s for i,s in enumerate(a)]
        self.assertAlmostEqual(compute_clq10(a), sum((s-1)/6 for s in z)/10)
    def test02_inversion_q6_q7(self):
        base = [4]*10; hi = list(base); hi[5] = 7; hi[6] = 7
        self.assertLess(compute_clq10(hi), compute_clq10(base))                                              
        x = list(base); x[0] = 7
        self.assertGreater(compute_clq10(x), compute_clq10(base))                                                        
    def test03_tout_a_1(self):
                                                                            
        self.assertAlmostEqual(compute_clq10([1]*10), 0.2)
        a = [1]*10; a[5] = a[6] = 7;  self.assertAlmostEqual(compute_clq10(a), 0.0)                     
    def test04_tout_a_7(self):
        a = [7]*10; a[5] = a[6] = 1;  self.assertAlmostEqual(compute_clq10(a), 1.0)                     
        self.assertAlmostEqual(compute_clq10([7]*10), 0.8)
    def test05_reponse_manquante(self):
        a = [4]*10; a[3] = None;      self.assertTrue(np.isnan(compute_clq10(a)))
        self.assertTrue(np.isnan(compute_clq10([4]*9)))
    def test06_hors_intervalle(self):
        for bad in (0, 8, 3.5, 'abc', ''):
            a = [4]*10; a[2] = bad;   self.assertTrue(np.isnan(compute_clq10(a)), bad)
    def test10_hors_ligne_vs_application(self):
        a = [5,2,6,3,1,7,4,2,5,3]
        df = pd.DataFrame([{f'Q{i+1}_raw': v for i, v in enumerate(a)}])
        off = float(clq10_series(df).iloc[0])
        app = compute_clq10(validate_clq10_answers(extract_clq10_answers(_clean_row(_form2_row(a))))[1])
        self.assertEqual(off, app)                                                                 

class TestFlux(unittest.TestCase):
    def _run(self, fn, *a):
        stubs = dict(analyze_student_answer=lambda *x, **k: {'type_erreur':'logic','score_estime':7},
                     _write_result_to_sheet=lambda r: WRITES.append(r),
                     _generate_exercise_for_student=lambda st: GEN.append(1),
                     _recommend_next_module_display=lambda *x, **k: REC.append(1),
                     _check_mastery=lambda *x, **k: False,
                     get_shap_explanation=lambda *x, **k: 'expl', _exp_log=lambda *x, **k: None)
        with _patched(**stubs): return fn(*a)
    def setUp(self):
        global WRITES, GEN, REC
        WRITES, GEN, REC = [], [], []
        self.st = _fresh_learner()
        self._run(cb_submit_answer, 'ma réponse', 't@x.org')
    def test07_form2_valide_traite_une_fois(self):
        self.assertTrue(self.st['waiting_form2'])                                                            
        self.assertNotIn(('t@x.org', 1), EXP_STATE['processed_f2'])                                          
        self._run(_handle_form2, _form2_row([4,3,3,2,2,6,5,2,2,3]))
        self.assertIn(('t@x.org', 1), EXP_STATE['processed_f2'])
        self.assertEqual(len(WRITES), 1); self.assertFalse(self.st['waiting_form2'])
    def test08_double_soumission(self):
        r = _form2_row([4,3,3,2,2,6,5,2,2,3])
        self._run(_handle_form2, r); snap = copy.deepcopy((self.st['module_scores'], self.st['module_cls'], self.st['mean_cl']))
        self._run(_handle_form2, r)
        self.assertEqual(snap, (self.st['module_scores'], self.st['module_cls'], self.st['mean_cl']))
        self.assertEqual(len(WRITES), 1); self.assertEqual(len(self.st['exercise_history']), 1)
        self.assertEqual(len(self.st['module_scores']), 1)                                                  
    def test09_sans_exercice_en_attente(self):
        self.st['waiting_form2'] = False; self.st['pending_exercise'] = None
        before = copy.deepcopy((self.st['mean_cl'], self.st['module_cls']))
        self._run(_handle_form2, _form2_row([4]*10))
        self.assertEqual(before, (self.st['mean_cl'], self.st['module_cls'])); self.assertEqual(WRITES, [])
    def test11_questionnaire_invalide_ne_modifie_rien(self):
        before = (self.st['mean_cl'], list(self.st['module_cls']))
        bad = _form2_row([4,3,3,2,2,6,5,2,2,3]); bad[CLQ10_ITEMS[3][1]] = '9 — ?'
        self._run(_handle_form2, bad)
        self.assertEqual(before, (self.st['mean_cl'], self.st['module_cls']))
        self.assertTrue(self.st['waiting_form2']); self.assertNotIn(('t@x.org', 1), EXP_STATE['processed_f2'])
        self._run(_handle_form2, _form2_row([4,3,3,2,2,6,5,2,2,3]))                                                
        self.assertEqual(len(WRITES), 1)
    def test12_clq_brut_distinct_du_lisse(self):
        self._run(_handle_form2, _form2_row([7,7,7,7,7,1,1,7,7,7]))                       
        self.assertEqual(self.st['clq_raw'], 1.0)
        self.assertAlmostEqual(self.st['cl_smoothed'], 0.5*0.7 + 1.0*0.3)
        self.assertEqual(WRITES[0]['clq_raw'], 1.0); self.assertNotEqual(WRITES[0]['clq_raw'], WRITES[0]['cl_smoothed'])

class TestStructureBase(unittest.TestCase):
    def test14_correspondance_exacte_des_items(self):
        a = [1,2,3,4,5,6,7,1,2,3]                                                                           
        for v in CLQ10_VERSIONS:
            row = _form2_row(a, version=v)
            self.assertEqual(detect_clq10_version(row), v)
            self.assertEqual([_likert7(x) for x in extract_clq10_answers(row)], a, v)
    def test15_ordre_des_colonnes_sans_effet(self):
        a = [5,2,6,3,1,7,4,2,5,3]; row = _form2_row(a)
        shuffled = dict(reversed(list(row.items())))
        self.assertEqual(compute_clq10(validate_clq10_answers(extract_clq10_answers(shuffled))[1]),
                         compute_clq10(validate_clq10_answers(extract_clq10_answers(row))[1]))
    def test16_versions_melangees_ou_inconnues_rejetees(self):
        a = [4]*10; p, o = _form2_row(a, version='pilot_base_v1'), _form2_row(a, version='online_form_v2')
        mix = {k: v for k, v in list(p.items())[:7]}; mix.update({k: v for k, v in list(o.items())[7:]})
        self.assertTrue(np.isnan(compute_clq10(extract_clq10_answers(mix))))                                                               
        self.assertIsNone(detect_clq10_version({'foo': 1}))
    def test17_inversion_uniquement_items_positifs(self):
        positifs = {'pilot_base_v1': ('I found the instructions clear', 'I was confident in my answer'),
                    'online_form_v2': ('The instructions were clear.', 'I felt confident about my answer.')}
        for v, its in CLQ10_VERSIONS.items():
            self.assertEqual({q for q, _, r in its if r}, {'Q6', 'Q7'}, v)                                        
            self.assertEqual((its[5][1], its[6][1]), positifs[v], v)                                                  
    def test18_base_reelle_56_scores(self):
        p = find_ails_xlsx()
        if p is None: self.skipTest('base xlsx absente')
        cl_ = pd.read_excel(p, 'Cognitive_Load'); f2_ = pd.read_excel(p, 'Form Responses 2')
        self.assertEqual(len(cl_), 56)
        self.assertEqual(detect_clq10_version(f2_.iloc[0].to_dict()), CLQ10_ANALYSIS_VERSION)
        k = ['learner_id', 'day_index', 'Exercise_ID']
        f2r = f2_.rename(columns={'Day index': 'day_index', f2_.columns[2]: 'Exercise_ID'})
        chk = f2r.merge(cl_, on=k, suffixes=('', '_cl'), validate='one_to_one')
        self.assertEqual(len(chk), 56)
        for q, (_, title, _) in zip(CLQ10_IDS, CLQ10_ITEMS):                                                              
            self.assertTrue((chk[f'{q}_raw'] == chk[title]).all(), q)
        rec = clq10_series(chk[[t for _, t, _ in CLQ10_ITEMS]])                                                    
        self.assertTrue(rec.notna().all())
        self.assertLessEqual(float(np.abs(rec.values - chk.CL_Score.values).max()), 1e-4)

class TestSelectionJours(unittest.TestCase):
    def test13_jours_attendus(self):
        d = pd.DataFrame({'day_index': [0,0,22,22,22,23]})
        d0, d22 = (d.day_index == 0).values, (d.day_index == 22).values
        self.assertEqual((int(d0.sum()), int(d22.sum()), int((~(d0|d22)).sum())), (2, 3, 1))                                              

_suite = unittest.TestSuite()
for _c in (TestCLq, TestFlux, TestSelectionJours, TestStructureBase): _suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(_c))
_res = unittest.TextTestRunner(verbosity=2).run(_suite)
assert _res.wasSuccessful(), '❌ des tests ont échoué'
print('✅ tests unitaires OK — NE PAS conclure à une validation tant que le test de bout en bout (cellule suivante) n’a pas été exécuté.')


In [ ]:
                                                                                
                                                
                                                                                              
                                                                                
WRITES, GEN, REC = [], [], []
st = _fresh_learner('demo@x.org')
stubs = dict(analyze_student_answer=lambda *x, **k: {'type_erreur':'logic','score_estime':8},
             _write_result_to_sheet=lambda r: WRITES.append(r),
             _generate_exercise_for_student=lambda s: GEN.append('next_exercise'),
             _recommend_next_module_display=lambda *x, **k: GEN.append('next_module'),
             get_shap_explanation=lambda *x, **k: 'expl', _exp_log=lambda *x, **k: None)
with _patched(**stubs):
    cb_submit_answer('réponse élève', 'demo@x.org')
    print('1) après exercice : scores=', st['module_scores'], '| waiting_form2=', st['waiting_form2'], '| CL lissé=', st['mean_cl'], '(inchangé)')
    answers = [5,4,4,3,3,5,4,3,3,4]
    _handle_form2(_form2_row(answers, 'demo@x.org'))
    print('2) après Form 2   : CLq brut=', st['clq_raw'], '| CL lissé=', st['cl_smoothed'], '| waiting_form2=', st['waiting_form2'])
    print('3) enregistrements :', len(WRITES), 'ligne résultat | historique :', st['exercise_history'])
    print('4) décision        :', GEN)
    _handle_form2(_form2_row(answers, 'demo@x.org'))                 
    print('5) répétition ignorée : lignes =', len(WRITES), '| scores =', st['module_scores'], '| CLq =', st['module_cls'])


### 📊 Résultats et Analyse Post-Expérimentation

Après avoir collecté suffisamment de données, exécutez la cellule ci-dessous pour analyser les résultats.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import re, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})
BLUE=  '#1a237e'; RED=   '#c0392b'
GREEN= '#1e8449'; ORANGE='#e67e22'
GRAY=  '#7f8c8d'; LGRAY= '#bdc3c7'
GCL = {'G1':RED,  'G2':BLUE,  'G3':GREEN}
GRP = {'G1':'G1 Novice','G2':'G2 Interm.','G3':'G3 Advanced'}

print("📥 Lecture Google Sheets...")

def read_tab(name):
    try:
        ws  = sh.worksheet(name)
        raw = ws.get_all_values()
        if not raw or len(raw) < 2:
            print(f"  ⚠️  {name}: vide")
            return pd.DataFrame()
        headers = raw[0]
        rows    = [r for r in raw[1:]
                   if any(str(c).strip() for c in r)]
        df = pd.DataFrame(rows, columns=headers)
        print(f"  ✅ {name}: {len(df)} lignes × "
              f"{len(df.columns)} colonnes")
        return df
    except Exception as e:
        print(f"  ❌ {name}: {e}")
        return pd.DataFrame()

df_f1 = read_tab('Form Responses 1')
df_f2 = read_tab('Form Responses 2')
df_r  = read_tab('AILS_Results')

LIKERT_MAP = {
                         
    '1':1,'2':2,'3':3,'4':4,'5':5,
                                                                               
                                                                           
    'strongly disagree':1.0,'disagree':2.0,'slightly disagree':2.5,
    'neutral':3.0,'slightly agree':3.5,'agree':4.0,'strongly agree':5.0,
                     
    'not at all':1,'a little':2,'somewhat':3,'quite a bit':4,'extremely':5,
    'very low':1,'low':2,'moderate':3,'high':4,'very high':5,
}
                                                                                           
                                                                                 
_LIKERT_KEYS_BY_LEN = sorted(LIKERT_MAP, key=len, reverse=True)

def _likert_label(s):
    lo = re.sub(r'\s+', ' ', s.lower().strip())
    if lo in LIKERT_MAP: return LIKERT_MAP[lo]
    for k in _LIKERT_KEYS_BY_LEN:
        if not k.isdigit() and k in lo: return LIKERT_MAP[k]
    return None

def parse_likert(v):
    """Même table que la cellule 5 (source unique) : '4', '4 - Agree', 'Strongly agree' (=5), etc."""
    s = str(v).strip()
    m = re.match(r'^(\d+)', s)
    if m: return float(m.group(1))
    r = _likert_label(s)
    return float(r) if r is not None else np.nan

REVERSE_ITEMS = ['confident', 'clear', 'confian']

def compute_tlx_independent(row_f2):
    vals = []
    for col_name, val in row_f2.items():
        cn = col_name.lower().strip()
        if not any(kw in cn for kw in [
            'mental', 'effort', 'frustrat',
            'time pressure', 'confident',
            'difficult', 'hesit', 'reread',
            'overload', 'had to', 'identify'
        ]):
            continue
        n = parse_likert(val)
        if np.isnan(n):
            continue
        if any(k in cn for k in REVERSE_ITEMS):
            n = 8 - n
        vals.append(n)
    if not vals:
        return np.nan
    raw = float(np.mean(vals))
    return round((raw - 1) / 6, 4)

def find_col(df, keywords):
    for c in df.columns:
        if any(k.lower() in c.lower() for k in keywords):
            return c
    return None

C_EMAIL_F1 = find_col(df_f1, ['email','Email'])
C_PRIOR    = find_col(df_f1, ['prior','already have',
                               'knowledge in this'])
C_CONF     = find_col(df_f1, ['confident','ability to solve'])
C_MOTIV    = find_col(df_f1, ['motivat'])
C_LEVEL    = find_col(df_f1, ['educational level',
                               'current educational'])
C_SUBJECT  = find_col(df_f1, ['subject','domain'])

print(f"\n  Form 1 colonnes clés :")
print(f"    email={C_EMAIL_F1}")
print(f"    prior={C_PRIOR}")
print(f"    conf={C_CONF}")
print(f"    level={C_LEVEL}")

C_EMAIL_F2  = find_col(df_f2, ['email','Email'])
C_EXID_F2   = find_col(df_f2, ['exercise','Exercise'])

CL_POSITIVE_KW = ['mental','difficult','hesit','confus',
                   'effort','frustrat','charge','pressure',
                   'overload','reread','had to','time']
CL_REVERSE_KW  = ['clear','confident','confian']

print(f"\n  Form 2 colonnes clés :")
print(f"    email={C_EMAIL_F2}")
print(f"    exercise_id={C_EXID_F2}")

C_EMAIL_R = find_col(df_r, ['email','learner_id','learner'])
C_SCORE_R = find_col(df_r, ['score_norm','score_raw','score'])
C_CL_R    = find_col(df_r, ['cl_value','cl_q','cl_state','cl'])
C_PRE_R   = find_col(df_r, ['pre_score','score_pre'])
C_POST_R  = find_col(df_r, ['post_score','score_post'])
C_MOD_R   = find_col(df_r, ['module_id','module'])
C_ERR_R   = find_col(df_r, ['error_type','error'])

print(f"\n  AILS_Results colonnes clés :")
print(f"    email={C_EMAIL_R} | score={C_SCORE_R} "
      f"| cl={C_CL_R}")

def norm_email(s):
    return str(s).strip().lower()

if C_EMAIL_F1:
    df_f1[C_EMAIL_F1] = df_f1[C_EMAIL_F1].apply(norm_email)
if C_EMAIL_F2:
    df_f2[C_EMAIL_F2] = df_f2[C_EMAIL_F2].apply(norm_email)
if C_EMAIL_R and not df_r.empty:
    df_r[C_EMAIL_R] = df_r[C_EMAIL_R].apply(norm_email)

all_emails = df_f1[C_EMAIL_F1].replace(
    '',np.nan).dropna().unique().tolist()\
    if C_EMAIL_F1 else []

print(f"\n  Apprenants Form 1 : {len(all_emails)}")

print("\n📊 Construction profils apprenants...")
learners = []

for email in all_emails:
    if not email or email == 'nan':
        continue

    row_f1 = df_f1[
        df_f1[C_EMAIL_F1] == email
    ].iloc[0] if len(df_f1[df_f1[C_EMAIL_F1]==email]) > 0\
     else pd.Series(dtype=object)

    prior = 0.5
    if C_PRIOR and not row_f1.empty:
        v = parse_likert(row_f1.get(C_PRIOR,''))
        if not np.isnan(v):
            prior = round((v-1)/4, 4)

    conf = 0.5
    if C_CONF and not row_f1.empty:
        v = parse_likert(row_f1.get(C_CONF,''))
        if not np.isnan(v):
            conf = round((v-1)/4, 4)

    motiv = 0.5
    if C_MOTIV and not row_f1.empty:
        v = parse_likert(row_f1.get(C_MOTIV,''))
        if not np.isnan(v):
            motiv = round((v-1)/4, 4)

    level   = str(row_f1.get(C_LEVEL,   'Unknown'))\
              if C_LEVEL   else 'Unknown'
    subject = str(row_f1.get(C_SUBJECT, 'Unknown'))\
              if C_SUBJECT else 'Unknown'

    g = ('G1' if prior < 0.35 else
         'G2' if prior < 0.65 else 'G3')

    cl_vals_all  = []
    tlx_vals_all = []
    n_form2      = 0

    if C_EMAIL_F2:
        f2_sub  = df_f2[df_f2[C_EMAIL_F2] == email]
        n_form2 = len(f2_sub)

        for _, row_f2 in f2_sub.iterrows():

                                                                             
            cl_q = compute_clq10(extract_clq10_answers(row_f2))
            if not np.isnan(cl_q):
                cl_vals_all.append(round(cl_q, 4))
                                                                                       
                                                                                                     

    mean_cl_f2  = round(float(np.mean(cl_vals_all)),  4)\
                  if cl_vals_all  else np.nan
    mean_tlx_f2 = round(float(np.mean(tlx_vals_all)), 4)\
                  if tlx_vals_all else np.nan
    cl_init_f2  = cl_vals_all[0]  if cl_vals_all else np.nan
    cl_final_f2 = cl_vals_all[-1] if cl_vals_all else np.nan

                                                                                                       
                                                                                                            
    pre_score  = round(prior * 100, 1)
    post_score = pre_score
    dg         = 0.0
    n_ex       = 0
    mean_cl_sys= mean_cl_f2 if not np.isnan(
                     mean_cl_f2 if cl_vals_all
                     else np.nan) else 0.5
    modules_done = []
    errors_done  = []

    if C_EMAIL_R and not df_r.empty:
        r_sub = df_r[df_r[C_EMAIL_R]==email].copy()

        if 'exercise_index' in r_sub.columns:
            r_sub['exercise_index'] = pd.to_numeric(
                r_sub['exercise_index'], errors='coerce')
            r_sub = r_sub.sort_values('exercise_index')

        if not r_sub.empty:
            n_ex = len(r_sub)

            if C_SCORE_R:
                sc = pd.to_numeric(
                    r_sub[C_SCORE_R],
                    errors='coerce').dropna().tolist()
                if sc:

                    if max(sc) > 1.5:
                        sc = [s/10 for s in sc]
                                                                                    
                                                                            
                                                                                       
                                                                                    
                    pre_score  = round(sc[0]*100,  1)
                    post_score = round(
                        float(np.mean(sc[-3:]))*100, 1)\
                        if len(sc)>=3 else round(sc[-1]*100,1)
                    if pre_score < 100:
                        dg = (post_score - pre_score) /\
                             (100 - pre_score)
                        dg = round(max(0,min(1,dg)),4)

            if C_CL_R:
                cl_s = pd.to_numeric(
                    r_sub[C_CL_R],
                    errors='coerce').dropna().tolist()
                if cl_s:
                    mean_cl_sys = round(
                        float(np.mean(cl_s)),4)

            if C_PRE_R:
                vp = pd.to_numeric(
                    r_sub[C_PRE_R],
                    errors='coerce').dropna()
                if len(vp)>0:
                    pre_score = round(float(vp.iloc[0]),1)
            if C_POST_R:
                vp = pd.to_numeric(
                    r_sub[C_POST_R],
                    errors='coerce').dropna()
                if len(vp)>0:
                    post_score = round(float(vp.iloc[-1]),1)
                if pre_score < 100:
                    dg = (post_score-pre_score)/\
                         (100-pre_score)
                    dg = round(max(0,min(1,dg)),4)

            if C_MOD_R:
                modules_done = r_sub[C_MOD_R]\
                    .astype(str).tolist()
            if C_ERR_R:
                errors_done = r_sub[C_ERR_R]\
                    .astype(str).tolist()

    final_cl = mean_cl_f2\
               if not np.isnan(mean_cl_f2\
                   if cl_vals_all else np.nan)\
               else (mean_cl_sys
                     if mean_cl_sys != 0.5
                     else round(prior*0.6+0.2, 3))

    learners.append({
        'id':         email,
        'group':      g,
        'prior':      prior,
        'confidence': conf,
        'motivation': motiv,
        'pre':        pre_score,
        'post':       post_score,
        'dg':         dg,
        'mean_cl':    final_cl,
        'cl_init':    cl_init_f2  if cl_vals_all else final_cl,
        'cl_final':   cl_final_f2 if cl_vals_all else final_cl,
        'tlx':        mean_tlx_f2,
        'n_ex':       n_ex,
        'n_form2':    n_form2,
        'has_gradio': n_ex > 0,
        'modules':    modules_done,
        'errors':     errors_done,
        'cl_series':  cl_vals_all,
    })

df_l = pd.DataFrame(learners)

print(f"\n{'═'*70}")
print(f"  RÉSULTATS COMPLETS — {len(df_l)} apprenants")
print(f"{'═'*70}")

print(f"\n  {'Email':<28} {'G':>2} {'Pr':>4} "
      f"{'Pre':>5} {'Post':>5} {'ΔG':>6} "
      f"{'CL':>5} {'F2':>3} {'Ex':>3} Grad")
print("  " + "-"*70)
for _, r in df_l.sort_values('group').iterrows():
    print(f"  {r['id'][:26]:<26} {r['group']:>2} "
          f"{r['prior']:>4.2f} "
          f"{r['pre']:>5.1f} {r['post']:>5.1f} "
          f"{r['dg']:>6.3f} {r['mean_cl']:>5.3f} "
          f"{r['n_form2']:>3} {r['n_ex']:>3} "
          f"{'✅' if r['has_gradio'] else '❌'}")

print(f"\n  Résumé par groupe :")
print(f"  {'Groupe':<14} {'N':>3} {'Prior':>6} "
      f"{'Pre':>6} {'Post':>6} {'ΔG':>7} "
      f"{'CL':>6} {'F2':>5} {'Grad':>5}")
print("  " + "-"*65)
for g in ['G1','G2','G3']:
    sub = df_l[df_l['group']==g]
    if sub.empty: continue
    print(f"  {GRP[g]:<14} {len(sub):>3} "
          f"{sub['prior'].mean():>6.2f} "
          f"{sub['pre'].mean():>6.1f} "
          f"{sub['post'].mean():>6.1f} "
          f"{sub['dg'].mean():>7.3f} "
          f"{sub['mean_cl'].mean():>6.3f} "
          f"{sub['n_form2'].sum():>5} "
          f"{sub['has_gradio'].sum():>5}")

all_sub = df_l
print(f"  {'Full cohort':<14} {len(all_sub):>3} "
      f"{all_sub['prior'].mean():>6.2f} "
      f"{all_sub['pre'].mean():>6.1f} "
      f"{all_sub['post'].mean():>6.1f} "
      f"{all_sub['dg'].mean():>7.3f} "
      f"{all_sub['mean_cl'].mean():>6.3f} "
      f"{all_sub['n_form2'].sum():>5} "
      f"{all_sub['has_gradio'].sum():>5}")

print(f"\n{'═'*60}")
print(f"  🧪 TESTS STATISTIQUES")
print(f"{'═'*60}")

stats_out = {}

diff_v = (df_l['post'] - df_l['pre']).values / 100
valid  = diff_v[~np.isnan(diff_v)]
if len(valid) >= 5 and np.any(valid != 0):
    try:
        W, p_w = wilcoxon(valid, alternative='two-sided')
        d_w    = float(np.mean(valid)/
                       np.std(valid,ddof=1))
        stats_out['wilcoxon'] = {'W':W,'p':p_w,'d':d_w}
        print(f"\n  1. Learning Gain (Wilcoxon) :")
        print(f"     W={W:.1f} | p={p_w:.4f} | d={d_w:.2f}")
        print(f"     {'✅ p<0.05' if p_w<0.05 else '⚠️  ns'}")
    except Exception as e:
        print(f"  ⚠️  Wilcoxon: {e}")
        W, p_w, d_w = 0, 1, 0
        stats_out['wilcoxon'] = {'W':0,'p':1,'d':0}
else:
    print(f"  ⚠️  Wilcoxon: données insuffisantes "
          f"(n={len(valid)})")
    W, p_w, d_w = 0, 1, 0
    stats_out['wilcoxon'] = {'W':0,'p':1,'d':0}

cl_i = df_l['cl_init'].dropna().values
cl_f = df_l['cl_final'].dropna().values
n_cl = min(len(cl_i), len(cl_f))
if n_cl >= 5:
    t_s, p_t = stats.ttest_rel(
        cl_i[:n_cl], cl_f[:n_cl])
    red = (np.mean(cl_i)-np.mean(cl_f)) /\
           np.mean(cl_i) * 100
    d_cl = float(
        (np.mean(cl_i)-np.mean(cl_f)) /
        np.sqrt((np.std(cl_i,ddof=1)**2+
                 np.std(cl_f,ddof=1)**2)/2))
    stats_out['cl'] = {'t':t_s,'p':p_t,
                        'd':d_cl,'red':red}
    print(f"\n  2. CL Reduction (paired t-test) :")
    print(f"     CL init={np.mean(cl_i):.3f} → "
          f"CL final={np.mean(cl_f):.3f}")
    print(f"     Réd={red:.1f}% | "
          f"t({n_cl-1})={t_s:.3f} | "
          f"p={p_t:.4f} | d={d_cl:.2f}")
    print(f"     {'✅ p<0.05' if p_t<0.05 else '⚠️  ns'}")
else:
    print(f"  ⚠️  CL t-test: n={n_cl} insuffisant")
    p_t, d_cl, red = 1, 0, 0

print(f"\n  3. Pearson CL_q vs NASA-TLX (sous-échelles indépendantes) :")

mask_valid = (~df_l['mean_cl'].isna() & ~df_l['tlx'].isna())
n_valid = mask_valid.sum()

if n_valid >= 5:
    cl_arr  = df_l.loc[mask_valid, 'mean_cl'].values
    tlx_arr = df_l.loc[mask_valid, 'tlx'].values
    r_cl_tlx, p_cl_tlx = stats.pearsonr(cl_arr, tlx_arr)
    stats_out['pearson_tlx'] = {
        'r': round(r_cl_tlx, 4),
        'p': round(p_cl_tlx, 4),
        'n': n_valid
    }
    print(f"     N : {n_valid}")
    print(f"     r = {r_cl_tlx:.4f}  |  p = {p_cl_tlx:.4f}")
    print(f"     {'✅ p<0.05' if p_cl_tlx<0.05 else '⚠️  ns'}")
else:
    r_p, p_p = stats.pearsonr(
        df_l['mean_cl'].dropna().values,
        (1 - df_l['prior']).dropna().values)
    stats_out['pearson_proxy'] = {'r': round(r_p,4), 'p': round(p_p,4)}
    print(f"     r(CL, 1-Prior) = {r_p:.4f}  |  p = {p_p:.4f}")

if len(df_l) < 3:
    print("\n❌ Pas assez de données pour les figures")
else:

    fig, axes = plt.subplots(1, 2, figsize=(13,5.5))
    fig.suptitle(
        f'Figure 4 — Learning Gain '
        f'(Real Data, $N={len(df_l)}$)',
        fontsize=13, fontweight='bold')

    ax = axes[0]
                                                                  
                                                                
                                                            
                                                               
    conds = {
        'Static\nBaseline': {
            'pre':45.2,'post':54.1,'dg':0.09,
            'sp':6.2,'ss':7.1},
        'Standard\nITS': {
            'pre':45.7,'post':63.2,'dg':0.17,
            'sp':5.8,'ss':6.8},
        f'Full AILS\n(N={len(df_l)})': {
            'pre':  df_l['pre'].mean(),
            'post': df_l['post'].mean(),
            'dg':   df_l['dg'].mean(),
            'sp':   df_l['pre'].std(),
            'ss':   df_l['post'].std()},
    }
    x  = np.arange(3); w = 0.32
    bc = [LGRAY, ORANGE, BLUE]
    for i,(cn,cv) in enumerate(conds.items()):
        ax.bar(x[i]-w/2, cv['pre'],  w,
               color=LGRAY, edgecolor='black',
               linewidth=0.8,
               yerr=cv['sp'], capsize=4)
        ax.bar(x[i]+w/2, cv['post'], w,
               color=bc[i], edgecolor='black',
               linewidth=0.8, alpha=0.9,
               yerr=cv['ss'], capsize=4)
        ax.text(x[i]+w/2, cv['post']+cv['ss']+1.5,
                f"ΔG={cv['dg']:.3f}",
                ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                color=bc[i])

    ax.set_xticks(x)
    ax.set_xticklabels(list(conds.keys()), fontsize=9)
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 120)
    ax.set_title('(a) Condition Comparison', fontsize=10)
    ax.legend(handles=[
        mpatches.Patch(color=LGRAY, label='Pre-test'),
        mpatches.Patch(color=BLUE,  label='Post-test'),
    ], fontsize=9, loc='upper left')
    ax.yaxis.grid(True, alpha=0.35, linestyle='--')
    ax.set_axisbelow(True)

    ax = axes[1]
    plot_data,plot_pos,plot_colors,plot_lbl=[],[],[],[]
    pos=1
    for g in ['G1','G2','G3']:
        sub = df_l[df_l['group']==g]
        if sub.empty: pos+=3; continue
        plot_data.append(sub['pre'].values)
        plot_pos.append(pos)
        plot_colors.append('#ecf0f1')
        plot_lbl.append(f'{GRP[g]}\nPre (n={len(sub)})')
        plot_data.append(sub['post'].values)
        plot_pos.append(pos+1)
        plot_colors.append(GCL[g])
        plot_lbl.append(f'{GRP[g]}\nPost')
        pos+=3

    if plot_data:
        bp=ax.boxplot(
            plot_data, positions=plot_pos,
            widths=0.65, patch_artist=True,
            medianprops=dict(color='black',lw=2),
            whiskerprops=dict(lw=1.2),
            capprops=dict(lw=1.2),
            flierprops=dict(marker='o',ms=4,alpha=0.5))
        for patch,col in zip(bp['boxes'],plot_colors):
            patch.set_facecolor(col)
            patch.set_alpha(0.85)

        for g,p0 in zip(['G1','G2','G3'],[1,4,7]):
            sub=df_l[df_l['group']==g]
            if sub.empty: continue
            for _,row in sub.iterrows():
                ax.plot([p0,p0+1],
                        [row['pre'],row['post']],
                        color=GCL[g],
                        alpha=0.25,lw=0.8)
            ax.plot([p0,p0+1],
                    [sub['pre'].mean(),
                     sub['post'].mean()],
                    color=GCL[g],lw=2.5,zorder=5)

            if len(sub)>=3:
                try:
                    _,pv=wilcoxon(
                        sub['pre'].values/100,
                        sub['post'].values/100,
                        alternative='less')
                    stars=('***' if pv<0.001 else
                           '**'  if pv<0.01  else
                           '*'   if pv<0.05  else 'ns')
                    yy=sub['post'].max()+4
                    ax.plot([p0,p0+1],[yy,yy],
                            color=GCL[g],lw=1)
                    ax.text(p0+0.5,yy+0.5,stars,
                            ha='center',fontsize=10,
                            color=GCL[g])
                except:
                    pass

    ax.set_xticks(plot_pos)
    ax.set_xticklabels(plot_lbl, fontsize=8)
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0,115)
    ax.set_title('(b) Pre/Post by Group', fontsize=10)
    ax.yaxis.grid(True,alpha=0.35,linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig('/content/figure4_learning_gain.png',
                dpi=300,bbox_inches='tight')
    plt.show()
    print("✅ Figure 4 sauvegardée")

    fig,axes=plt.subplots(1,2,figsize=(12,5))
    fig.suptitle(
        'Figure 5 — CL Validation (Real Data)',
        fontsize=13,fontweight='bold')

    ax=axes[0]
    cl_data,cl_pos,cl_cols,cl_lbl=[],[],[],[]
    pos=1
    for g in ['G1','G2','G3']:
        sub=df_l[df_l['group']==g]
        if sub.empty or sub['mean_cl'].isna().all():
            pos+=2; continue
        cl_data.append(sub['mean_cl'].dropna().values)
        cl_pos.append(pos)
        cl_cols.append(GCL[g])
        cl_lbl.append(f'{GRP[g]}\n(n={len(sub)})')
        pos+=2
    if cl_data:
        bp2=ax.boxplot(
            cl_data,positions=cl_pos,widths=0.6,
            patch_artist=True,
            medianprops=dict(color='black',lw=2),
            whiskerprops=dict(lw=1.2),
            capprops=dict(lw=1.2),
            flierprops=dict(marker='o',ms=4,alpha=0.5))
        for patch,col in zip(bp2['boxes'],cl_cols):
            patch.set_facecolor(col)
            patch.set_alpha(0.8)

        for i,(g,p0) in enumerate(
                zip(['G1','G2','G3'],[1,3,5])):
            sub=df_l[df_l['group']==g]
            vals=sub['mean_cl'].dropna().values
            if len(vals)==0: continue
            jitter=np.random.normal(p0,0.05,len(vals))
            ax.scatter(jitter,vals,
                       color=GCL[g],alpha=0.6,
                       s=30,zorder=4)

    ax.axhline(0.60,color=ORANGE,linestyle='--',
               lw=1.5,label='CL-critical (0.60)')
    ax.axhline(0.30,color=GREEN, linestyle=':',
               lw=1.2,alpha=0.7,
               label='Confidence (0.30)')
    ax.set_xticks(cl_pos)
    ax.set_xticklabels(cl_lbl,fontsize=9)
    ax.set_ylabel('Mean $\\widehat{\\mathrm{CL}}$')
    ax.set_ylim(0,1)
    ax.set_title('(a) CL Distribution by Group',
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.yaxis.grid(True,alpha=0.35,linestyle='--')
    ax.set_axisbelow(True)

    ax=axes[1]
    for g in ['G1','G2','G3']:
        sub=df_l[df_l['group']==g]
        if sub.empty: continue
        ax.scatter(sub['prior'],sub['mean_cl'],
                   c=GCL[g],alpha=0.65,s=50,
                   label=GRP[g],
                   edgecolors='black',lw=0.5,
                   zorder=3)

    valid_mask = (~df_l['mean_cl'].isna() &
                  ~df_l['prior'].isna())
    if valid_mask.sum() >= 3:
        xv=df_l.loc[valid_mask,'prior'].values
        yv=df_l.loc[valid_mask,'mean_cl'].values
        r_v,p_v=stats.pearsonr(xv,yv)
        m_v,b_v=np.polyfit(xv,yv,1)
        xs=np.linspace(0,1,100)
        ax.plot(xs,m_v*xs+b_v,'k--',lw=2,
                label=f'r={r_v:.3f}',zorder=4)

        ax.text(0.97,0.95,
                f'r = {r_v:.3f}\np = {p_v:.3f}',
                transform=ax.transAxes,
                fontsize=9,va='top',ha='right',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='white',
                          edgecolor='#ccc',alpha=0.9))

    ax.set_xlabel('Prior Knowledge [0,1]')
    ax.set_ylabel('Mean CL [0,1]')
    ax.set_xlim(-0.05,1.05)
    ax.set_ylim(-0.05,1.05)
    ax.set_title('(b) CL vs Prior Knowledge',
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3,linestyle='--')

    plt.tight_layout()
    plt.savefig('/content/figure5_cl_validation.png',
                dpi=300,bbox_inches='tight')
    plt.show()
    print("✅ Figure 5 sauvegardée")

print(f"\n{'═'*60}")
print(f"  📋 CHIFFRES POUR LaTeX")
print(f"{'═'*60}")

for g in ['G1','G2','G3']:
    sub=df_l[df_l['group']==g]
    if sub.empty: continue
    print(f"\n  {GRP[g]} (n={len(sub)}) :")
    print(f"    Prior  = {sub['prior'].mean():.2f} "
          f"± {sub['prior'].std():.2f}")
    print(f"    Pre    = {sub['pre'].mean():.1f} "
          f"± {sub['pre'].std():.1f} %")
    print(f"    Post   = {sub['post'].mean():.1f} "
          f"± {sub['post'].std():.1f} %")
    print(f"    ΔG     = {sub['dg'].mean():.3f} "
          f"± {sub['dg'].std():.3f}")
    print(f"    CL     = {sub['mean_cl'].mean():.3f} "
          f"± {sub['mean_cl'].std():.3f}")
    print(f"    F2     = {sub['n_form2'].sum()} soumissions")

ws_r2=stats_out.get('wilcoxon',{})
cl_r2=stats_out.get('cl',{})
pe_r2=stats_out.get('pearson',{})

print(f"\n  Tests :")
print(f"    N total = {len(df_l)}")
print(f"    N avec Form2 = {(df_l['n_form2']>0).sum()}")
print(f"    N avec Gradio = {df_l['has_gradio'].sum()}")
if ws_r2:
    print(f"    Wilcoxon W={ws_r2.get('W',0):.1f} | "
          f"p={ws_r2.get('p',1):.4f} | "
          f"d={ws_r2.get('d',0):.2f}")
if cl_r2:
    print(f"    CL t={cl_r2.get('t',0):.3f} | "
          f"p={cl_r2.get('p',1):.4f} | "
          f"réd={cl_r2.get('red',0):.1f}%")

print(f"\n✅ Figures → /content/")
print(f"   figure4_learning_gain.png")
print(f"   figure5_cl_validation.png")

pe_tlx = stats_out.get('pearson_tlx', {})
if pe_tlx:
    print(f"\n  Pearson CL_q vs NASA-TLX (sous-échelles indépendantes) :")
    print(f"    r = {pe_tlx.get('r',0):.4f} | "
          f"p = {pe_tlx.get('p',1):.4f} | "
          f"N = {pe_tlx.get('n',0)}")
    print(f"    {'✅ p<0.05' if pe_tlx.get('p',1)<0.05 else '⚠️  ns'}")

print(f"\n  AILS_Results :")
print(f"    Lignes totales     = {len(df_r)}")
print(f"    Apprenants uniques = {df_r[C_EMAIL_R].nunique() if C_EMAIL_R else 'N/A'}")
print(f"    Modules couverts   = {df_r['module_id'].nunique() if 'module_id' in df_r.columns else 'N/A'}")
                                       
n_gen      = EXP_STATE.get('n_gen', 0)
n_first_ok = EXP_STATE.get('n_first_ok', 0)
n_retry    = EXP_STATE.get('n_retry', 0)
n_fallback = EXP_STATE.get('n_fallback', 0)
n_delivered = n_first_ok + (n_gen - n_first_ok - n_fallback)
pct_first  = 100 * n_first_ok / n_gen if n_gen else 0
pct_retry  = 100 * n_retry    / n_gen if n_gen else 0
pct_fallbk = 100 * n_fallback / n_gen if n_gen else 0
print(f'\n  6. Verification Engine (§3.2.0.4):')
print(f'     Exercises generated : {n_gen}')
print(f'     First-attempt OK    : {n_first_ok}  ({pct_first:.1f}%)')
print(f'     Retry (≥1 attempt)  : {n_retry}   ({pct_retry:.1f}%)')
print(f'     Fallback (κ<0.60)   : {n_fallback}    ({pct_fallbk:.1f}%)')

if 'score_raw' in df_r.columns:
    sc_all = pd.to_numeric(df_r['score_raw'], errors='coerce').dropna()
    print(f"    Score moyen        = {sc_all.mean():.2f}/10 ± {sc_all.std():.2f}")
if 'cl_value' in df_r.columns:
    cl_all = pd.to_numeric(df_r['cl_value'], errors='coerce').dropna()
    print(f"    CL moyen système   = {cl_all.mean():.3f} ± {cl_all.std():.3f}")
if 'mastered' in df_r.columns:
    n_mastered = (df_r['mastered'].astype(str).str.upper() == 'TRUE').sum()
    print(f"    Modules maîtrisés  = {n_mastered}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

                      
conditions = ['Static\nBaseline', 'No-CL', 'No-CF', 'No-XAI', 'Full AILS\n(Real)']
delta_G = [0.058, 0.100, 0.130, 0.168, 0.201]
cl_means = [0.718, 0.624, 0.512, 0.479, 0.473]

modules = ['XAI module\n(No-XAI → Full)', 'CF module\n(No-CF → Full)', 'CL module\n(No-CL → Full)']
marginal_contrib = [0.033, 0.071, 0.101]

                            
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.2, 1]})

                                                                                
                          
                                                                                
x = np.arange(len(conditions))
width = 0.35

                                                                     
ax1_twin = ax1.twinx()
rects2 = ax1_twin.bar(x + width/2, cl_means, width, label='Mean CL (Cognitive Load)',
                      color='#e06d53', hatch='//', alpha=0.85, zorder=2)
ax1_twin.set_ylabel('Mean Session Cognitive Load (CL)', color='#e06d53', fontweight='bold')
ax1_twin.set_ylim(0, 1.0)

                                              
rects1 = ax1.bar(x - width/2, delta_G, width, label='$\Delta\\bar{G}$ (Learning Gain)',
                 color='#1b365d', zorder=3)
ax1.set_ylabel('Normalised Learning Gain ($\Delta\\bar{G}$)', color='#1b365d', fontweight='bold')
ax1.set_ylim(0, 0.30)
ax1.set_xticks(x)
ax1.set_xticklabels(conditions)
ax1.set_zorder(ax1_twin.get_zorder() + 1)
ax1.patch.set_visible(False)                                       
ax1.grid(axis='y', linestyle='--', alpha=0.3, zorder=1)

                 
for rect in rects1:
    h = rect.get_height()
    ax1.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom',
                 fontsize=8.5, fontweight='bold', color='#1b365d')

for rect in rects2:
    h = rect.get_height()
    ax1_twin.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                      xytext=(0, 3), textcoords="offset points", ha='center', va='bottom',
                      fontsize=8.5, color='#b03a2e')

                  
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', framealpha=0.9)
ax1.set_title('(a) Learning Gain ($\Delta\\bar{G}$) & Cognitive Load (CL)', fontweight='bold')

                                                                                
                                        
                                                                                
colors = ['#2e7d32', '#1565c0', '#c62828']                    
y_pos = np.arange(len(modules))

                     
rects3 = ax2.barh(y_pos, marginal_contrib, color=colors, height=0.45, edgecolor='black', linewidth=0.8, zorder=3)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(modules)
ax2.set_xlim(0, 0.14)
ax2.set_xlabel('Marginal Gain Contribution ($\Delta\\bar{G}_{Full} - \Delta\\bar{G}_{ablated}$)', fontweight='bold')
ax2.set_title('(b) Marginal Contribution per Module', fontweight='bold')
ax2.grid(axis='x', linestyle='--', alpha=0.3, zorder=1)

                 
for rect in rects3:
    w = rect.get_width()
    ax2.annotate(f'+{w:.3f}', xy=(w, rect.get_y() + rect.get_height() / 2),
                 xytext=(6, 0), textcoords="offset points", ha='left', va='center',
                 fontsize=9, fontweight='bold')

                                   
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('Fig7_Corrected_Fixed.png', dpi=300)
plt.show()

In [ ]:
                                                                          
                                                                 
                                                                          
                                                                     
                                                                       
                                                                 
                                                             
 
                                                                       

                                                                                                   
                                                                       
import numpy as np
from scipy.stats import wilcoxon

MIN_N_FOR_TEST = 5                                                                     

if 'df_l' not in globals() or df_l.empty:
    raise RuntimeError(
        "df_l n'est pas défini — exécutez d'abord la Cellule 21 (analyse "
        "Google Sheets complète) pour construire dynamiquement la cohorte "
        "réelle à partir du contenu actuel de la Google Sheet."
    )

completers   = df_l[df_l['has_gradio']]
n_completers = len(completers)

print("=" * 70)
print("🧪 TABLE 17 — Wilcoxon Signed-Rank Test (Learning Gain)")
print("=" * 70)
print("Source des données : RÉELLES (Google Sheets, lecture dynamique)")
print(f"N total inscrits (Form 1)           = {len(df_l)}")
print(f"N avec dossier d'exercices complet  = {n_completers}")

if n_completers < MIN_N_FOR_TEST:
    print(f"\n⚠️  N={n_completers} insuffisant (< {MIN_N_FOR_TEST}) pour un test "
          f"statistiquement interprétable — collectez plus de données avant de "
          f"relancer cette cellule.")
else:
    diff = (completers['post'] - completers['pre']).values / 100.0
    diff = diff[~np.isnan(diff)]
    n_valid = len(diff)

                                                                                
    W, p_value = wilcoxon(diff, alternative='two-sided')
    cohens_d   = float(np.mean(diff) / np.std(diff, ddof=1))

    print(f"\nN (paires pre/post valides) = {n_valid}")
    print(f"Pre-test  : {completers['pre'].mean():.1f} ± {completers['pre'].std():.1f}")
    print(f"Post-test : {completers['post'].mean():.1f} ± {completers['post'].std():.1f}")
    print(f"\nW-statistic = {W:.1f}, p-value = {p_value:.4f}")
    print(f"Cohen's d (paired) = {cohens_d:.2f}")
    print(f"\n{'✅ p<0.05 — gain statistiquement significatif' if p_value < 0.05 else '⚠️  non significatif (p≥0.05) — population actuelle trop restreinte ou hétérogène'}")


## 📑 Partie 4 — Valeurs et figures de l'article (JEDM)

À exécuter **après** les cellules 1→25 (même session Colab). Chaque cellule remplit le
dictionnaire `RESULTS`, sauvegarde ses figures dans `article_results/` (PNG + PDF) et
affiche les lignes LaTeX à copier dans l'article. Une cellule dont les données sont
absentes s'arrête proprement avec un message (rien n'est inventé).

In [ ]:
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

OUT_DIR = 'article_results'
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS    = {}                                                              
TEX_TABLES = {}                                  

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11, 'axes.titlesize': 12,
    'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 120, 'savefig.facecolor': 'white',
})
C_BLUE, C_RED, C_GREEN, C_ORANGE, C_GRAY = '#1a237e', '#c0392b', '#1e8449', '#e67e22', '#7f8c8d'

def save_fig(fig, name):
    fig.savefig(f'{OUT_DIR}/{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(f'{OUT_DIR}/{name}.pdf', bbox_inches='tight')
    print(f'   💾 {OUT_DIR}/{name}.png / .pdf')

def boot_ci(x, stat=np.mean, n_boot=10000, alpha=0.05, seed=42):
    """IC bootstrap percentile (reproductible)."""
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    idx = np.random.RandomState(seed).randint(0, len(x), size=(n_boot, len(x)))
    vals = stat(x[idx], axis=1)
    return float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2)))

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def _find(df, kws):
    return next((c for c in df.columns if any(k in str(c).lower() for k in kws)), None)

def _read_table(var_name, sheet_tab, xlsx_names=('AILS_Database_corrigee.xlsx',)):
    """DataFrame déjà en mémoire → onglet Google Sheet → fichier xlsx local."""
    g = globals()
    if var_name in g and isinstance(g[var_name], pd.DataFrame) and not g[var_name].empty:
        return g[var_name].copy()
    try:
        return pd.DataFrame(sh.worksheet(sheet_tab).get_all_records())
    except Exception:
        pass
    for base in ('', '/content/', '/content/drive/MyDrive/'):
        for name in xlsx_names:
            path = base + name
            if os.path.exists(path):
                try:
                    return pd.read_excel(path, sheet_name=sheet_tab)
                except Exception:
                    continue
    return pd.DataFrame()

print('✅ R1 prêt — sorties dans', OUT_DIR)

In [ ]:
                                                                                
                                                                                       
                                                                                    
                                                                              
                                                                                
                                                                              
                                                                                             
                                                                                            
                                                                                            
ABL = dict(seed=42, eta=0.18, cl_relief=0.95, cl_noise=0.02, fit_boost=0.0)
CONDITIONS = {
    'Static': dict(cl_adapt=False, cf=False),
    'No-CL':  dict(cl_adapt=False, cf=True),
    'No-CF':  dict(cl_adapt=True,  cf=False),
    'Full':   dict(cl_adapt=True,  cf=True),
}

def draw_profiles(pop_config, seed):
    rng = np.random.RandomState(seed)
    c = lambda v: float(np.clip(v, 0.01, 0.99))
    out = []
    for gname, p in pop_config.items():
        for _ in range(p['n']):
            out.append(dict(
                group=p.get('label', gname),
                prior=c(rng.normal(*p['prior'])), conf=c(rng.normal(*p['conf'])),
                motiv=c(rng.normal(*p['motiv'])),
                h=c(rng.normal(*p['h_base'])), g=c(rng.normal(*p['g_base'])),
                e=c(rng.normal(*p['e_base'])), r=c(rng.normal(*p['r_base']))))
    return out

def run_learner(prof, cond, peers_pool, seed, lid):
    rng = np.random.RandomState(seed)
    cl = estimate_cl(prof['h'], prof['g'], prof['e'], prof['r'])[0]
    s0 = s = prof['prior']
    mastered, attempted, progress = [], set(), {}
    err, cls = [], []
    while True:
        cands = [m for m in get_candidate_modules(set(mastered)) if m not in attempted]
        if not cands:
            break
        learner = {
            'id': lid, 'username': f'abl_{lid}', 'prior_knowledge': prof['prior'],
            'confidence': prof['conf'], 'motivation': prof['motiv'],
            'mean_response_time': prof['r'], 'mean_gaze_disp': prof['g'],
            'mean_head_movement': prof['h'], 'mean_interaction_ev': prof['e'],
            'mean_score': s, 'mean_cl': cl,
            'mastered_modules': list(mastered), 'module_progress': progress}
        if cond['cf']:
            res = recommend_next_module(learner, peers_pool, cl, k=10)
            ranked = [c['module_id'] for c in (res['all_candidates'] if res else [])
                      if c['module_id'] not in attempted]
            mid = ranked[0] if ranked else cands[0]
        else:
            mid = cands[0]
        eta_eff = ABL['eta'] * (1 + ABL['fit_boost'] * (compat_score(learner, mid) - 0.5))
        s_pre, n_ex, ok = s, 0, False
        while n_ex < EXP_MAX_EXERCISES:
            s = score_evolution(s, cl, eta=eta_eff, xi=rng.uniform(0.8, 1.2))
            if cond['cl_adapt']:
                cl = max(0.05, cl * (ABL['cl_relief'] + rng.normal(0, ABL['cl_noise'])))
            else:
                cl = float(np.clip(cl * (1.0 + rng.normal(0, ABL['cl_noise'])), 0.05, 0.99))
            n_ex += 1
            err.append(1.0 - s); cls.append(cl)
            if n_ex >= N_WINDOW and s >= MODULES[mid]['theta'] and cl <= EXP_MASTERY_CL_MAX:
                ok = True
                break
        attempted.add(mid)
        if ok:
            mastered.append(mid)
        progress[mid] = {'score_pre': round(s_pre, 3), 'score_post': round(s, 3),
                         'score_current': round(s, 3), 'mean_cl': round(cl, 3),
                         'is_mastered': ok, 'nb_exercises': n_ex}
    return dict(gain=(s - s0) / (1 - s0) if s0 < 1 else np.nan,
                error=float(np.mean(err)) if err else np.nan,
                exercises=len(err), cl=float(np.mean(cls)) if cls else np.nan,
                n_mastered=len(mastered),
                ex_per_module=(len(err) / len(mastered)) if mastered else np.nan)

def run_ablation(seed, peers_pool):
    profiles = draw_profiles(POPULATION_SYN, seed)
    rows = []
    for name, cond in CONDITIONS.items():
        for i, prof in enumerate(profiles):
            r = run_learner(prof, cond, peers_pool, seed * 1000 + i, 5000 + i)
            r.update(condition=name, learner=i, group=prof['group'])
            rows.append(r)
    return pd.DataFrame(rows)

print('🧪 Simulation d\'ablation (N=30, seed=42)…')
df_abl = run_ablation(ABL['seed'], SYN_LEARNERS_CF)
METRICS = [('gain', 'Gain'), ('error', 'Error'), ('exercises', 'Exercises'), ('cl', 'CL')]
EXTRA   = ['n_mastered', 'ex_per_module']                                                                
order = list(CONDITIONS)
summ = df_abl.groupby('condition')[[m for m, _ in METRICS] + EXTRA].agg(['mean', 'std']).loc[order]
print(summ.round(3).to_string())

                                                                                             
paired = {}
for other in ['Static', 'No-CL', 'No-CF']:
    for m, _ in METRICS:
        a = df_abl[df_abl.condition == 'Full'].sort_values('learner')[m].values
        b = df_abl[df_abl.condition == other].sort_values('learner')[m].values
        d = a - b
        try:
            p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        except ValueError:
            p = np.nan
        paired[f'Full_vs_{other}_{m}'] = dict(mean_diff=float(d.mean()), p_wilcoxon=float(p))
print('\nFull vs autres (différence moyenne appariée, Wilcoxon p) :')
for k, v in paired.items():
    print(f"  {k:<26} Δ={v['mean_diff']:+.3f}  p={fmt_p(v['p_wilcoxon'])}")

if abs(paired['Full_vs_No-CF_gain']['mean_diff']) < 1e-12 and abs(paired['Full_vs_No-CF_cl']['mean_diff']) < 1e-12:
    print('\nℹ️ Full ≡ No-CF : dans ce modèle le choix du module n\'influence pas le score (fit_boost=0). '
          'L\'ablation ne montre donc que l\'effet mécanique de (1-CL). À écrire tel quel dans l\'annexe.')

                                                                                   
N_SEEDS_SENS = 10                                             
sens = []
for sd in range(1, N_SEEDS_SENS + 1):
    d = run_ablation(sd, SYN_LEARNERS_CF)
    sens.append(d.groupby('condition')[[m for m, _ in METRICS]].mean().loc[order].assign(seed=sd))
sens = pd.concat(sens).reset_index()
sens_summ = sens.groupby('condition')[[m for m, _ in METRICS]].agg(['mean', 'std']).loc[order]
print(f'\nSensibilité à la graine ({N_SEEDS_SENS} graines, moyenne des moyennes ± SD entre graines) :')
print(sens_summ.round(3).to_string())

RESULTS['ablation'] = dict(
    params=ABL, n=int(df_abl.learner.nunique()),
    seed42={c: {m: dict(mean=float(summ.loc[c, (m, 'mean')]), sd=float(summ.loc[c, (m, 'std')]))
                for m in [k for k, _ in METRICS] + EXTRA} for c in order},
    paired_full_vs=paired,
    seed_sensitivity={c: {m: dict(mean=float(sens_summ.loc[c, (m, 'mean')]), sd=float(sens_summ.loc[c, (m, 'std')]))
                          for m, _ in METRICS} for c in order})

rows_tex = []
for c in order:
    cells = [f"${summ.loc[c, (m, 'mean')]:.2f}{{\\pm}}{summ.loc[c, (m, 'std')]:.2f}$"
             if m != 'exercises' else f"${summ.loc[c, (m, 'mean')]:.1f}{{\\pm}}{summ.loc[c, (m, 'std')]:.1f}$"
             for m, _ in METRICS]
    rows_tex.append(f"{c} & " + ' & '.join(cells) + r' \\')
TEX_TABLES['ablation_sim'] = '\n'.join(rows_tex)
print('\n📋 Lignes LaTeX pour Table (ablation_sim) :\n' + TEX_TABLES['ablation_sim'])

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
cols = [C_GRAY, C_ORANGE, C_BLUE, C_GREEN]
for ax, (m, lab) in zip(axes, METRICS):
    means = [summ.loc[c, (m, 'mean')] for c in order]
    sds = [summ.loc[c, (m, 'std')] for c in order]
    ax.bar(order, means, yerr=sds, color=cols, alpha=.85, capsize=3)
    for k, c in enumerate(order):
        v = df_abl[df_abl.condition == c][m].values
        ax.scatter(np.random.RandomState(k).normal(k, .06, len(v)), v, s=8, color='k', alpha=.35, zorder=3)
    ax.set_title(lab); ax.tick_params(axis='x', rotation=25)
fig.suptitle('Simulation consistency check (N=30, seed=42) — not evidence of effectiveness', y=1.04, fontsize=11)
fig.tight_layout(); save_fig(fig, 'figA1_ablation_sim'); plt.show()

In [ ]:
                                                                                       
                                                                                         
PREPOST_PRE_COL   = None                           
PREPOST_POST_COLS = None                                                                 

def analyse_prepost(df, pre_col, post_col, label):
    d = df[[pre_col, post_col]].apply(pd.to_numeric, errors='coerce').dropna()
    pre, post = d[pre_col].values, d[post_col].values
    n, diff = len(d), post - pre
    ci = boot_ci(diff)
    nz = diff[diff != 0]
    W = stats.wilcoxon(diff, alternative='two-sided', method='auto')
    ranks = stats.rankdata(np.abs(nz))
    Wp, Wm = ranks[nz > 0].sum(), ranks[nz < 0].sum()
    ng = np.where(pre < 100, (post - pre) / (100 - pre), np.nan)
    lr = stats.linregress(pre, diff)
    out = dict(label=label, n=int(n), pre_mean=float(pre.mean()), pre_sd=float(pre.std(ddof=1)),
               post_mean=float(post.mean()), post_sd=float(post.std(ddof=1)),
               diff_mean=float(diff.mean()), diff_ci95=ci, W=float(W.statistic), p=float(W.pvalue),
               n_nonzero=int(len(nz)), ties_or_zeros=bool(len(nz) < n or len(np.unique(np.abs(nz))) < len(nz)),
               rank_biserial=float((Wp - Wm) / (Wp + Wm)) if (Wp + Wm) else np.nan,
               cohen_dz=float(diff.mean() / diff.std(ddof=1)),
               ng_mean=float(np.nanmean(ng)), ng_sd=float(np.nanstd(ng, ddof=1)),
               ng_median=float(np.nanmedian(ng)), n_pre100=int((pre >= 100).sum()),
               spearman_pre_vs_diff=[float(x) for x in stats.spearmanr(pre, diff)],
               slope_diff_on_pre=float(lr.slope), slope_p=float(lr.pvalue))
    return out, pre, post

df_pp = _read_table('df_prepost', 'PrePost')
if df_pp.empty:
    print('⚠️ R3 : aucune table pré/post trouvée (df_prepost / onglet PrePost / xlsx). Cellule ignorée.')
else:
    print('Colonnes PrePost :', list(df_pp.columns))
    pre_col = PREPOST_PRE_COL or _find(df_pp, ['pre'])
    post_cols = PREPOST_POST_COLS or [c for c in df_pp.columns if 'post' in str(c).lower()]
    print(f'→ pré = {pre_col} | post = {post_cols}  (modifier PREPOST_* si incorrect)')
    RESULTS['prepost'] = {}
    for pc in post_cols:
        res, pre, post = analyse_prepost(df_pp, pre_col, pc, pc)
        RESULTS['prepost'][pc] = res
        print(f"\n— {pc} —  n={res['n']}")
        print(f"  Pré  {res['pre_mean']:.1f} ± {res['pre_sd']:.1f} | Post {res['post_mean']:.1f} ± {res['post_sd']:.1f}")
        print(f"  Différence {res['diff_mean']:+.1f}  IC95% bootstrap [{res['diff_ci95'][0]:.1f}, {res['diff_ci95'][1]:.1f}]")
        print(f"  Wilcoxon (bilatéral) W={res['W']:.1f}, p={fmt_p(res['p'])} | r_rang-bis.={res['rank_biserial']:.2f} | dz={res['cohen_dz']:.2f}"
              f"{'  (ex æquo/zéros → approx. normale)' if res['ties_or_zeros'] else ''}")
        print(f"  ΔG normalisé : {res['ng_mean']:.2f} ± {res['ng_sd']:.2f} (médiane {res['ng_median']:.2f}; {res['n_pre100']} apprenant(s) à 100 exclu(s))")
        print(f"  Pré vs différence : Spearman ρ={res['spearman_pre_vs_diff'][0]:.2f} (p={fmt_p(res['spearman_pre_vs_diff'][1])}) "
              f"→ si ρ<0, régression vers la moyenne possible")
        TEX_TABLES[f'prepost_{pc}'] = (
            f"Full cohort & ${res['pre_mean']:.1f}\\pm{res['pre_sd']:.1f}$ & ${res['post_mean']:.1f}\\pm{res['post_sd']:.1f}$ & "
            f"${res['diff_mean']:+.1f}$ [{res['diff_ci95'][0]:.1f}, {res['diff_ci95'][1]:.1f}] & ${res['ng_mean']:.2f}$ \\\\")
        print('  📋 LaTeX :', TEX_TABLES[f'prepost_{pc}'])

        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        for a, b in zip(pre, post):
            ax[0].plot([0, 1], [a, b], color=C_GRAY, alpha=.45, lw=1)
        ax[0].plot([0, 1], [pre.mean(), post.mean()], color=C_RED, lw=3, marker='o')
        ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['Pre-test', 'Post-test']); ax[0].set_ylabel('Score (0–100)')
        ax[0].set_title(f'Paired scores (n={len(pre)})')
        ax[1].scatter(pre, post - pre, color=C_BLUE, alpha=.8)
        xs = np.linspace(pre.min(), pre.max(), 50)
        ax[1].plot(xs, res['slope_diff_on_pre'] * xs + (np.mean(post - pre) - res['slope_diff_on_pre'] * pre.mean()), color=C_RED)
        ax[1].axhline(0, color='k', lw=.6)
        ax[1].set_xlabel('Pre-test score'); ax[1].set_ylabel('Post − pre')
        ax[1].set_title('Pre-test vs change (regression-to-the-mean check)')
        fig.tight_layout(); save_fig(fig, f'fig_prepost_{str(pc)[:20].replace(" ", "_")}'); plt.show()

In [ ]:
                                                                                                             
                                                         

                                                                             
def _test_clq10():
    nan = np.nan
    base = [4] * 10
                                                        
    a = [1, 1, 1, 1, 1, 7, 7, 1, 1, 1]                                          
    assert compute_clq10(a) == 0.0, 'Q6/Q7=7 doivent devenir 1 (borne basse)'
    b = [7, 7, 7, 7, 7, 1, 1, 7, 7, 7]                                          
    assert compute_clq10(b) == 1.0, 'Q6/Q7=1 doivent devenir 7 (borne haute)'
    assert abs(compute_clq10(base) - 0.5) < 1e-12, 'milieu d\'échelle = 0,5 (8−4=4)'
    q6_up = base.copy(); q6_up[5] = 5
    q1_up = base.copy(); q1_up[0] = 5
    step = (1 / 6) / 10
    assert abs(compute_clq10(q6_up) - (0.5 - step)) < 1e-12, 'Q6 ↑ doit DIMINUER le score'
    assert abs(compute_clq10(q1_up) - (0.5 + step)) < 1e-12, 'Q1 ↑ doit AUGMENTER le score'
                                                          
    rng = np.random.RandomState(0)
    for _ in range(2000):
        v = list(rng.randint(1, 8, 10)); s = compute_clq10(v)
        assert 0.0 <= s <= 1.0, f'hors bornes : {v} → {s}'
                                                                                       
    invalids = [base[:9], base + [4], [None] + base[1:], [nan] + base[1:], ['' ] + base[1:],
                [0] + base[1:], [8] + base[1:], [3.5] + base[1:], ['abc'] + base[1:], ['agree'] + base[1:]]
    for v in invalids:
        assert np.isnan(compute_clq10(v)), f'devrait être NaN : {v}'
        assert validate_clq10_answers(v)[0] is False
    assert validate_clq10_answers(['5'] + base[1:])[0], 'texte numérique "5" accepté'
                                                                                           
    a = [3, 2, 5, 6, 1, 2, 7, 4, 3, 5]
    manual = np.mean([((8 - s) if i in (5, 6) else s) - 1 for i, s in enumerate(a)]) / 6
    assert abs(compute_clq10(a) - manual) < 1e-12
    perm = a.copy(); perm[0], perm[4] = perm[4], perm[0]
    assert abs(compute_clq10(perm) - compute_clq10(a)) < 1e-12
    assert abs(compute_clq10(dict(zip(CLQ10_IDS, a))) - compute_clq10(a)) < 1e-12, 'dict ≡ liste'
    assert len(CLQ10_COLS) == len(set(CLQ10_COLS)) == 10 and CLQ10_REVERSED == {'Q6', 'Q7'}
    print('✅ Tests compute_clq10 : recodage Q6/Q7, bornes [0,1], manquants/invalides, cohérence — OK')
_test_clq10()

if 'df_form2_clean' not in globals() or df_form2_clean.empty or 'df_form2_clq_src' not in globals():
    print('⚠️ R4 : df_form2_clean / df_form2_clq_src absents (exécuter les cellules 4→7). Analyse ignorée.')
else:
    f2 = df_form2_clean.copy()
    f2['clq10'] = clq10_series(df_form2_clq_src.loc[f2.index])                                           
    print(f'CL_q (compute_clq10) : {int(f2.clq10.notna().sum())}/{len(f2)} questionnaires valides | moyenne = {f2.clq10.mean():.3f}')
                                                                  
    chk = (f2['clq10'] - f2['clq_score_norm']).abs().max()
    assert (np.isnan(chk) or chk < 1e-3), f'incohérence cellule 7 / 29 : {chk}'
    RESULTS['clq_version_check'] = dict(n=int(f2.clq10.notna().sum()), mean_clq10=float(f2.clq10.mean()), max_abs_diff_vs_cell7=float(chk))
                                                                                                                     
    try:
        _src = globals().get('_tab') or globals().get('_load')
        _cl = _src('Cognitive_Load') if _src else None
        if _cl is not None and 'CL_Score' in _cl.columns:
            _x = clq10_series(_cl)
            print(f'   contrôle vs Cognitive_Load.CL_Score : écart max = {(_x - _cl.CL_Score).abs().max():.2e} (tolérance 1e-3, arrondi à 4 décimales)')
            assert (_x - _cl.CL_Score).abs().max() < 1e-3
    except Exception as _e:
        print('   (contrôle Cognitive_Load ignoré :', _e, ')')

    c_mail, c_ts = _find(f2, ['email']), _find(f2, ['timestamp', 'horodat'])
    if not c_mail:
        print('⚠️ colonne email introuvable dans Form 2.')
    else:
        f2['_mail'] = f2[c_mail].astype(str).str.strip().str.lower()
        if c_ts:
            f2[c_ts] = pd.to_datetime(f2[c_ts], errors='coerce')
            f2 = f2.sort_values([ '_mail', c_ts])
        g = f2.dropna(subset=['clq10']).groupby('_mail')['clq10']
        n_sub = g.size()
        multi = n_sub[n_sub >= 2].index
        first = g.first()[multi].values; last = g.last()[multi].values
        diff = last - first
        res = dict(n_submitters=int(len(n_sub)), n_submissions=int(n_sub.sum()),
                   mean_per_submitter=float(n_sub.mean()), n_with_pair=int(len(multi)),
                   n_single=int((n_sub == 1).sum()))
        if len(multi) >= 5:
            w = stats.wilcoxon(diff, alternative='two-sided')
            res.update(first_mean=float(first.mean()), last_mean=float(last.mean()),
                       diff_mean=float(diff.mean()), diff_ci95=boot_ci(diff),
                       W=float(w.statistic), p=float(w.pvalue), cohen_dz=float(diff.mean() / diff.std(ddof=1)))
        RESULTS['cl'] = res
        print(f"\nSoumissions : {res['n_submissions']} par {res['n_submitters']} apprenants "
              f"(moy. {res['mean_per_submitter']:.1f}) | paires 1ʳᵉ/dernière : {res['n_with_pair']} | une seule : {res['n_single']}")
        if 'W' in res:
            print(f"CL_q 1ʳᵉ={res['first_mean']:.3f} → dernière={res['last_mean']:.3f} | Δ={res['diff_mean']:+.3f} "
                  f"IC95% [{res['diff_ci95'][0]:.3f}, {res['diff_ci95'][1]:.3f}] | W={res['W']:.1f}, p={fmt_p(res['p'])}, dz={res['cohen_dz']:.2f}")

                                                                                         
        if c_ts and f2[c_ts].notna().any():
            gaps = f2.groupby('_mail')[c_ts].apply(lambda s: s.diff().dropna().dt.total_seconds() / 60).explode().dropna().astype(float)
            chk = {}
            if len(gaps) >= 5:
                mode = gaps.round().mode().iloc[0]
                chk['modal_gap_min'] = float(mode)
                chk['share_gaps_within_1min_of_mode'] = float((np.abs(gaps - mode) <= 1).mean())
            days = f2[c_ts].dt.date.value_counts()
            chk['most_common_day'] = str(days.index[0]); chk['share_on_most_common_day'] = float(days.iloc[0] / days.sum())
            mono = (f2.dropna(subset=['clq10']).groupby('_mail')['clq10']
                      .apply(lambda s: len(s) >= 3 and bool(np.all(np.diff(s.values) < 0))))
            chk['n_strictly_decreasing_trajectories'] = int(mono.sum())
            RESULTS['cl_timing_check'] = chk
            print('\n🕵️ Contrôle des horodatages :', json.dumps(chk, indent=2, ensure_ascii=False))
            if chk.get('share_gaps_within_1min_of_mode', 0) > .5:
                print('⚠️ Plus de la moitié des intervalles sont quasi identiques : vérifier que ces lignes sont de vraies réponses avant publication.')

        fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
        for a, b in zip(first, last):
            ax[0].plot([0, 1], [a, b], color=C_GRAY, alpha=.5)
        if len(multi):
            ax[0].plot([0, 1], [first.mean(), last.mean()], color=C_RED, lw=3, marker='o')
        ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['First', 'Last']); ax[0].set_ylabel('CL$_q$')
        ax[0].set_title(f'CL$_q$ within learner (n={len(multi)})')
        ax[1].hist(f2['clq10'].dropna(), bins=12, color=C_BLUE, alpha=.8)
        ax[1].set_xlabel('CL$_q$'); ax[1].set_ylabel('Submissions'); ax[1].set_title('Distribution of CL$_q$')
        fig.tight_layout(); save_fig(fig, 'fig_cl_questionnaire'); plt.show()

In [ ]:
                                                                                  
                                                                                          
                                                                                       
                                                                          
def _rank_metrics(rank):
    return dict(p1=float(rank == 1), r3=float(rank is not None and rank <= 3),
                r5=float(rank is not None and rank <= 5),
                ndcg3=(1 / math.log2(rank + 1)) if (rank is not None and rank <= 3) else 0.0,
                mrr=(1 / rank) if rank else 0.0)

def _random_expected(c, covered):
    if not covered or c == 0:
        return dict(p1=0, r3=0, r5=0, ndcg3=0, mrr=0)
    return dict(p1=1 / c, r3=min(3, c) / c, r5=min(5, c) / c,
                ndcg3=sum(1 / math.log2(r + 1) for r in range(1, min(3, c) + 1)) / c,
                mrr=sum(1 / r for r in range(1, c + 1)) / c)

def cf_loo_real(real, k=10):
    events = []
    for l in real:
        m = list(l.get('mastered_modules', []))
        if len(m) < 2:
            continue
        true_next, fake_m = m[-1], m[:-1]
        fake = {**l, 'mastered_modules': fake_m}
        pool = [x for x in real if str(x['id']) != str(l['id'])]
        cands = get_candidate_modules(set(fake_m))
        covered = true_next in cands
        ranks = {}
        res = recommend_next_module(fake, pool, l['mean_cl'], k=k) if pool else None
        sc = {c['module_id']: c for c in res['all_candidates']} if res else {}
        order_idx = {mid: i for i, mid in enumerate(MODULE_ORDER)}
        pop = {mid: sum(mid in x.get('mastered_modules', []) for x in pool) for mid in cands}
        rankings = {
            'AILS (CF + compat)': [c['module_id'] for c in res['all_candidates']] if res else [],
            'CF only':   sorted(cands, key=lambda mid: (-sc.get(mid, {}).get('vote_cf', 0), order_idx[mid])),
            'Compat only': sorted(cands, key=lambda mid: (-sc.get(mid, {}).get('compat_score', 0), order_idx[mid])),
            'Prereq. successor': sorted(cands, key=lambda mid: order_idx[mid]),
            'Popularity': sorted(cands, key=lambda mid: (-pop[mid], order_idx[mid])),
        }
        row = dict(learner=l['id'], covered=covered, n_cand=len(cands))
        for name, lst in rankings.items():
            rk = lst.index(true_next) + 1 if true_next in lst else None
            for mk, v in _rank_metrics(rk).items():
                row[f'{name}|{mk}'] = v
        for mk, v in _random_expected(len(cands), covered).items():
            row[f'Random (expected)|{mk}'] = v
        events.append(row)
    return pd.DataFrame(events)

if 'REAL_LEARNERS' not in globals() or len(REAL_LEARNERS) < 3:
    print('⚠️ R5 : REAL_LEARNERS absent/insuffisant (exécuter la cellule 14). Ignorée.')
else:
    ev = cf_loo_real(REAL_LEARNERS)
    methods = ['AILS (CF + compat)', 'CF only', 'Compat only', 'Prereq. successor', 'Popularity', 'Random (expected)']
    mlab = [('p1', 'Precision@1'), ('r3', 'Recall@3'), ('r5', 'Recall@5'), ('ndcg3', 'NDCG@3'), ('mrr', 'MRR')]
    n_ev = len(ev)
    print(f'Cohorte réelle : {len(REAL_LEARNERS)} apprenants | événements LOO : {n_ev} | '
          f'vérité terrain dans les candidats : {int(ev.covered.sum())}/{n_ev}')
    tab = pd.DataFrame({m: {lab: ev[f'{m}|{k}'].mean() for k, lab in mlab} for m in methods})
    print(tab.round(3).to_string())
    print(f"Taille moyenne de l'ensemble de candidats (DAG) : {ev.n_cand.mean():.1f} (min {ev.n_cand.min()}, max {ev.n_cand.max()})")
    if ev.n_cand.mean() < 3:
        print('⚠️ Ensemble de candidats très petit : le DAG fait déjà presque tout le travail, les métriques sont proches du plafond '
              'même pour « Random » — à discuter dans l\'article avant de conclure sur la qualité du CF.')
    RESULTS['cf'] = dict(n_learners=len(REAL_LEARNERS), n_events=int(n_ev), n_covered=int(ev.covered.sum()),
                         mean_n_candidates=float(ev.n_cand.mean()),
                         metrics={m: {lab: dict(mean=float(ev[f'{m}|{k}'].mean()),
                                                ci95=boot_ci(ev[f'{m}|{k}'].values)) for k, lab in mlab} for m in methods})
    TEX_TABLES['cf_metrics'] = '\n'.join(
        f"{lab} & " + ' & '.join(f"${tab.loc[lab, m]:.2f}$" for m in methods) + r' \\' for _, lab in mlab)
    print('\n📋 LaTeX (colonnes :', ' | '.join(methods), ')\n' + TEX_TABLES['cf_metrics'])

    fig, ax = plt.subplots(figsize=(10, 3.8))
    w = 0.13; x = np.arange(len(mlab))
    pal = [C_GREEN, C_BLUE, C_ORANGE, C_GRAY, '#8e44ad', '#bdc3c7']
    for i, m in enumerate(methods):
        means = [RESULTS['cf']['metrics'][m][lab]['mean'] for _, lab in mlab]
        lo = [means[j] - RESULTS['cf']['metrics'][m][lab]['ci95'][0] for j, (_, lab) in enumerate(mlab)]
        hi = [RESULTS['cf']['metrics'][m][lab]['ci95'][1] - means[j] for j, (_, lab) in enumerate(mlab)]
        ax.bar(x + (i - 2.5) * w, means, w, yerr=[np.maximum(lo, 0), np.maximum(hi, 0)], label=m, color=pal[i], capsize=2)
    ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in mlab]); ax.set_ylim(0, 1.05)
    ax.set_title(f'Leave-one-out on real cohort (events={n_ev}) with baselines')
    ax.legend(fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(.5, -.12))
    fig.tight_layout(); save_fig(fig, 'fig_cf_baselines'); plt.show()

In [ ]:
                                                                                   
                                                                                             
EXPERT_COL, EXERCISE_COL = 'expert', 'exercise'
CRITERIA_LABELS = ['Pedagogical relevance', 'Difficulty appropriateness', 'Curriculum alignment',
                   'Clarity of statement', 'Error-type targeting', 'Hint usefulness',
                   'Solution correctness', 'Feedback quality', 'Level suitability', 'Learning progression']

def fleiss_kappa(M):
    """M : (n_items, n_categories) = nb d'évaluateurs ayant choisi chaque catégorie."""
    M = np.asarray(M, float); N, _ = M.shape; n = M[0].sum()
    P_i = ((M ** 2).sum(1) - n) / (n * (n - 1))
    p_j = M.sum(0) / (N * n)
    Pe = (p_j ** 2).sum()
    return float((P_i.mean() - Pe) / (1 - Pe))

def krippendorff_alpha_ordinal(R, cats=(1, 2, 3, 4, 5)):
    cats = list(cats); k = len(cats); o = np.zeros((k, k))
    for row in R:
        r = [cats.index(int(v)) for v in row if not np.isnan(v)]
        m = len(r)
        if m < 2:
            continue
        for i in range(m):
            for j in range(m):
                if i != j:
                    o[r[i], r[j]] += 1.0 / (m - 1)
    n_c = o.sum(1); n = n_c.sum()
    def d2(a, b):
        lo, hi = min(a, b), max(a, b)
        return (n_c[lo:hi + 1].sum() - (n_c[lo] + n_c[hi]) / 2) ** 2
    Do = sum(o[a, b] * d2(a, b) for a in range(k) for b in range(k)) / n
    De = sum(n_c[a] * n_c[b] * d2(a, b) for a in range(k) for b in range(k)) / (n * (n - 1))
    return float(1 - Do / De)

def icc2k(R):
    """ICC(2,k) : modèle à deux facteurs aléatoires, moyenne de k évaluateurs. R : (n_items, k)."""
    R = np.asarray(R, float); n, k = R.shape; gm = R.mean()
    ms_r = k * ((R.mean(1) - gm) ** 2).sum() / (n - 1)
    ms_c = n * ((R.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((R - R.mean(1, keepdims=True) - R.mean(0, keepdims=True) + gm) ** 2).sum()
    ms_e = sse / ((n - 1) * (k - 1))
    return float((ms_r - ms_e) / (ms_r + (ms_c - ms_e) / n))

df_ex = _read_table('df_expert', 'Expert_Ratings')
if df_ex.empty and os.path.exists('expert_ratings.csv'):
    df_ex = pd.read_csv('expert_ratings.csv')
if df_ex.empty:
    print('⚠️ R6 : notes brutes des experts absentes (df_expert / onglet Expert_Ratings / expert_ratings.csv). '
          'Sans elles, ni κ ni les moyennes ne sont reproductibles. Cellule ignorée.')
else:
    crit = [c for c in df_ex.columns if c not in (EXPERT_COL, EXERCISE_COL)]
    df_ex[crit] = df_ex[crit].apply(pd.to_numeric, errors='coerce')
    experts = sorted(df_ex[EXPERT_COL].unique()); n_exo = df_ex[EXERCISE_COL].nunique()
    print(f'{len(experts)} experts × {n_exo} exercices × {len(crit)} critères')
    crit_means = df_ex[crit].mean()
    ex_means = df_ex.groupby(EXERCISE_COL)[crit].mean().mean(1)                             
    overall = float(df_ex[crit].mean().mean())
                                                
    long = df_ex.melt(id_vars=[EXPERT_COL, EXERCISE_COL], value_vars=crit, var_name='criterion', value_name='r')
    wide = long.pivot_table(index=[EXERCISE_COL, 'criterion'], columns=EXPERT_COL, values='r')
    R = wide.values
    cats = np.arange(1, 6)
    M = np.stack([(R == c).sum(1) for c in cats], 1)
    res = dict(n_experts=len(experts), n_exercises=int(n_exo), overall_mean=overall,
               sd_across_exercises=float(ex_means.std(ddof=1)), sd_across_criteria=float(crit_means.std(ddof=1)),
               fleiss_kappa=fleiss_kappa(M), krippendorff_alpha_ordinal=krippendorff_alpha_ordinal(R),
               icc2k=icc2k(R), criterion_means={str(c): float(v) for c, v in crit_means.items()})
    RESULTS['expert'] = res
    print(f"Moyenne globale = {overall:.2f} | SD entre exercices = {res['sd_across_exercises']:.2f} | SD entre critères = {res['sd_across_criteria']:.2f}")
    print(f"Accord : Fleiss κ = {res['fleiss_kappa']:.2f} | Krippendorff α (ordinal) = {res['krippendorff_alpha_ordinal']:.2f} | ICC(2,k) = {res['icc2k']:.2f}")
    print('→ préciser dans l\'article quelle SD et quel coefficient sont rapportés (κ de Cohen = 2 évaluateurs).')
    labs = CRITERIA_LABELS if len(crit) == 10 else [str(c) for c in crit]
    TEX_TABLES['expert_scores'] = '\n'.join(f'{l} & ${v:.1f}$ \\\\' for l, v in zip(labs, crit_means.values)) +\
        f"\n\\midrule\nOverall mean & ${overall:.2f}\\pm{res['sd_across_exercises']:.2f}$ \\\\"
    print('\n📋 LaTeX :\n' + TEX_TABLES['expert_scores'])

    ang = np.linspace(0, 2 * np.pi, len(crit), endpoint=False).tolist(); ang += ang[:1]
    fig = plt.figure(figsize=(6.2, 6.2)); ax = fig.add_subplot(111, polar=True)
    pal = [C_BLUE, C_ORANGE, C_GREEN, C_RED, C_GRAY]
    for i, e in enumerate(experts):
        v = df_ex[df_ex[EXPERT_COL] == e][crit].mean().tolist(); v += v[:1]
        ax.plot(ang, v, lw=1, alpha=.7, color=pal[i % 5], label=f'Expert {e}')
    v = crit_means.tolist(); v += v[:1]
    ax.plot(ang, v, lw=2.5, color='k', label='Mean'); ax.fill(ang, v, alpha=.12, color='k')
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(labs, fontsize=8); ax.set_ylim(1, 5)
    ax.set_title(f'Expert ratings (mean={overall:.2f}, Fleiss κ={res["fleiss_kappa"]:.2f})', y=1.08)
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=8)
    save_fig(fig, 'figure6_expert_radar'); plt.show()

In [ ]:
                                                                                                
                                                                                             
ESS_GROUP_COL, ESS_SCORE_COL, ESS_ITEM_COLS = 'group', 'ess', None

df_e = _read_table('df_ess', 'ESS')
if df_e.empty:
    print('⚠️ R7 : table ESS absente (df_ess / onglet ESS). Cellule ignorée.')
else:
    if ESS_SCORE_COL not in df_e.columns:
        cols = ESS_ITEM_COLS or [c for c in df_e.columns if c != ESS_GROUP_COL]
        df_e[ESS_SCORE_COL] = df_e[cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    gl = df_e[ESS_GROUP_COL].astype(str).str.lower()
    full = df_e.loc[gl.str.contains('full'), ESS_SCORE_COL].dropna().values
    nox = df_e.loc[gl.str.contains('no'), ESS_SCORE_COL].dropna().values
    n1, n2 = len(full), len(nox)
    one = stats.mannwhitneyu(full, nox, alternative='greater', method='auto')
    two = stats.mannwhitneyu(full, nox, alternative='two-sided', method='auto')
    r_rb = 2 * one.statistic / (n1 * n2) - 1
    rng = np.random.RandomState(42)
    boots = [2 * stats.mannwhitneyu(rng.choice(full, n1), rng.choice(nox, n2)).statistic / (n1 * n2) - 1 for _ in range(2000)]
    hl = float(np.median([a - b for a in full for b in nox]))                        
    res = dict(n_full=int(n1), n_noxai=int(n2), mean_full=float(full.mean()), sd_full=float(full.std(ddof=1)),
               mean_noxai=float(nox.mean()), sd_noxai=float(nox.std(ddof=1)),
               U_full=float(one.statistic), U_noxai=float(n1 * n2 - one.statistic),
               p_one_tailed=float(one.pvalue), p_two_sided=float(two.pvalue),
               rank_biserial=float(r_rb), rank_biserial_ci95=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
               hodges_lehmann_shift=hl)
    RESULTS['ess'] = res
    print(f"Full n={n1}: {res['mean_full']:.2f}±{res['sd_full']:.2f} | NoXAI n={n2}: {res['mean_noxai']:.2f}±{res['sd_noxai']:.2f}")
    print(f"Mann–Whitney : U_Full={res['U_full']:.1f} (U_NoXAI={res['U_noxai']:.1f}), p unilatéral={fmt_p(res['p_one_tailed'])}, "
          f"p bilatéral={fmt_p(res['p_two_sided'])}")
    print(f"r rang-bisérial = {r_rb:.2f} IC95% [{res['rank_biserial_ci95'][0]:.2f}, {res['rank_biserial_ci95'][1]:.2f}] | décalage HL = {hl:+.2f}")
    fig, ax = plt.subplots(figsize=(4.6, 4))
    ax.boxplot([nox, full], labels=['NoXAI', 'Full'], widths=.5)
    for i, v in enumerate([nox, full], 1):
        ax.scatter(np.random.RandomState(i).normal(i, .05, len(v)), v, s=14, alpha=.6, color=C_BLUE)
    ax.set_ylabel('ESS (1–5)'); ax.set_title('Explanation satisfaction')
    fig.tight_layout(); save_fig(fig, 'fig_ess'); plt.show()

In [ ]:
                                                                                                 
                                                                                                 
                                                                                               
                                                                                           
                                                                                  
GEN_REQUESTS = globals().get('GEN_REQUESTS', [])
if 'generate_exercise_rag' in globals() and not getattr(generate_exercise_rag, '_wrapped', False):
    _orig_gen = generate_exercise_rag
    def generate_exercise_rag(*a, **k):
        keys = ['n_gen', 'n_first_ok', 'n_retry', 'n_fallback']
        b = {x: EXP_STATE.get(x, 0) for x in keys}
        out = _orig_gen(*a, **k)
        d = {x: EXP_STATE.get(x, 0) - b[x] for x in keys}
        outcome = ('failed_none' if out is None else 'fallback' if d['n_fallback'] > 0
                   else 'first_attempt' if d['n_first_ok'] > 0 else 'after_retry')
        GEN_REQUESTS.append(dict(module=(a[0] if a else k.get('module_id')), attempts=d['n_gen'], outcome=outcome))
        return out
    generate_exercise_rag._wrapped = True
    print('✅ Journalisation par requête activée (à partir de maintenant).')

def summarize_generation():
    if GEN_REQUESTS:
        g = pd.DataFrame(GEN_REQUESTS); n = len(g); vc = g.outcome.value_counts()
        print(f'Requêtes journalisées : {n}')
        for o in ['first_attempt', 'after_retry', 'fallback', 'failed_none']:
            print(f'  {o:<14} {vc.get(o, 0):>4}  ({100 * vc.get(o, 0) / n:.1f}%)')
        RESULTS['generation'] = dict(level='request', n=n, **{o: int(vc.get(o, 0)) for o in vc.index})
    s = {x: EXP_STATE.get(x, 0) for x in ['n_gen', 'n_first_ok', 'n_retry', 'n_fallback']}
    print(f'Compteurs bruts (niveau TENTATIVE) : {s}')
    RESULTS.setdefault('generation', {})['attempt_counters'] = s
    print('Rappel : dans le code, 0.60 ≤ c < 0.70 → exercice signalé à l\'enseignant ET livré (pas de nouvelle tentative) ;'
          '\n         les nouvelles tentatives viennent d\'un JSON invalide, d\'un exercice incomplet ou d\'une similarité > 0.60.')
if 'EXP_STATE' in globals():
    summarize_generation()
else:
    print('⚠️ R8 : EXP_STATE absent (exécuter la cellule 20).')

In [ ]:
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
with open(f'{OUT_DIR}/results_values.json', 'w', encoding='utf-8') as f:
    json.dump(_jsonable(RESULTS), f, indent=2, ensure_ascii=False)
with open(f'{OUT_DIR}/results_tables.tex', 'w', encoding='utf-8') as f:
    for k, v in TEX_TABLES.items():
        f.write(f'%% ---- {k} ----\n{v}\n\n')
print('Sections calculées :', list(RESULTS))
print('Tableaux LaTeX    :', list(TEX_TABLES))
try:
    import shutil
    shutil.make_archive('article_results', 'zip', OUT_DIR)
    from google.colab import files
    files.download('article_results.zip')
except Exception as e:
    print('ℹ️ Archive créée localement (article_results.zip) ; téléchargement auto indisponible :', e)

## 🎚️ Partie 5 — Sensibilité aux poids (simulation, base synthétique)

**Statut : analyse de robustesse, PAS une validation.** Le simulateur contient par construction
deux hypothèses déclarées (cf. `SENS`) : (i) le gain suit Eq. 11 et décroît avec la charge,
(ii) la charge d'un exercice augmente avec l'écart module–niveau (`lam`). Un jeu de poids
« meilleur » ici l'est SOUS CES HYPOTHÈSES. Protocole : le jeu est choisi sur des graines de
sélection, puis évalué sur des graines de test disjointes. Prérequis : cellules 1→10 et R1.

In [ ]:
import numpy as np, pandas as pd, math, time, itertools
from scipy import stats
assert 'save_fig' in globals(), "Exécuter d'abord la cellule R1 (outils communs)."

SENS = dict(budget=24,                                                              
            max_per_module=15, eta=0.18, cl_relief=0.95, cl_noise=0.02,
            lam=0.5,                                                                                   
            K=10, n_learners=30,
            seeds_select=list(range(1, 31)), seeds_test=list(range(101, 131)))

MODS  = list(MODULE_ORDER)
D_NRM = {m: (MODULES[m]['difficulty'] - 1) / 4.0 for m in MODS}
PRE   = {m: set(MODULES[m]['prerequisites']) for m in MODS}
THETA = {m: MODULES[m]['theta'] for m in MODS}
MIDX  = {m: j for j, m in enumerate(MODS)}

def _draw_profiles(pop_config, seed):
    rng = np.random.RandomState(seed)
    c = lambda v: float(np.clip(v, 0.01, 0.99))
    out = []
    for gname, p in pop_config.items():
        for _ in range(p['n']):
            out.append(dict(prior=c(rng.normal(*p['prior'])), conf=c(rng.normal(*p['conf'])),
                            motiv=c(rng.normal(*p['motiv'])), h=c(rng.normal(*p['h_base'])),
                            g=c(rng.normal(*p['g_base'])), e=c(rng.normal(*p['e_base'])),
                            r=c(rng.normal(*p['r_base']))))
    return out

                                                                             
def sched(cold, crit, steady, thr=0.60, name=''):
    return dict(kind='schedule', cold=cold, crit=crit, steady=steady, thr=thr, name=name)
POL_ACTUAL     = sched((.6, .4), (.4, .6), (.6, .4), name='ACTUAL (Eq. 3)')
POL_CURRICULUM = dict(kind='fixed',  name='Curriculum order')
POL_RANDOM     = dict(kind='random', name='Random candidate')

class _Peers:
    """Pool de pairs simulés : matrice de profils (cosine) + Δ de score par module."""
    PV = None; DELTA = None

def _vec(prof, s, cl, n_mast):
    return np.array([prof['prior'], prof['conf'], prof['motiv'], prof['r'], prof['g'],
                     prof['h'], prof['e'], s, cl, n_mast / len(MODS)])

def _candidates(mastered, attempted):
    return [m for m in MODS if m not in mastered and m not in attempted and PRE[m] <= mastered]

def _compat(mid, prior, mastered):
    """Formule de l'article (Eq. 6) : 0,6·ℓ + 0,4·κ, avec le niveau INITIAL p_u."""
    pre = PRE[mid]
    cov = len(pre & mastered) / len(pre) if pre else 1.0
    return 0.6 * (1.0 - abs(D_NRM[mid] - prior)) + 0.4 * cov

def _scores(prof, s, cl, mastered, cands):
    v = _vec(prof, s, cl, len(mastered))
    nv = np.linalg.norm(v)
    sims = _Peers.PV @ (v / nv) if nv > 0 else np.zeros(len(_Peers.PV))
    top = np.argsort(-sims)[:SENS['K']]
    vote = {m: float(np.sum(sims[top] * _Peers.DELTA[top, MIDX[m]])) for m in cands}
    comp = {m: _compat(m, prof['prior'], mastered) for m in cands}
    return vote, comp

def _choose(pol, prof, s, cl, mastered, cands, n_int, u_t, diag=None):
    if len(cands) == 1 or pol['kind'] == 'fixed':
        return cands[0]
    if pol['kind'] == 'random':
        return cands[int(u_t * len(cands)) % len(cands)]
    vote, comp = _scores(prof, s, cl, mastered, cands)
    regime = 'cold' if n_int < 3 else ('crit' if cl >= pol['thr'] else 'steady')
    w2, w4 = pol[regime]
    if diag is not None:                                                                                  
        tops = {round(w, 1): max(cands, key=lambda m: (w * vote[m] + (1 - w) * comp[m], -MIDX[m]))
                for w in np.linspace(0, 1, 11)}
        diag.append(dict(n_cand=len(cands), n_distinct=len(set(tops.values())),
                         mean_vote=float(np.mean(list(vote.values()))),
                         mean_compat=float(np.mean(list(comp.values())))))
    return max(cands, key=lambda m: (w2 * vote[m] + w4 * comp[m], -MIDX[m]))

def _run_one(prof, pol, xi, nz, u, budget, diag=None):
    cl = estimate_cl(prof['h'], prof['g'], prof['e'], prof['r'])[0]
    s0 = s = prof['prior']
    mastered, attempted, progress, cls = set(), set(), {}, []
    t = n_int = 0
    while t < budget:
        cands = _candidates(mastered, attempted)
        if not cands:
            break
        mid = _choose(pol, prof, s, cl, mastered, cands, n_int, u[min(t, len(u) - 1)], diag)
        s_pre, n_ex, ok = s, 0, False
        while n_ex < SENS['max_per_module'] and t < budget:
            cl_eff = float(np.clip(cl + SENS['lam'] * (D_NRM[mid] - s), 0.05, 0.99))
            s = min(1.0, s + SENS['eta'] * (1 - s) * (1 - cl_eff) * xi[t])
            cl = max(0.05, cl * (SENS['cl_relief'] + nz[t]))
            n_ex += 1; t += 1; cls.append(cl_eff)
            if n_ex >= N_WINDOW and s >= THETA[mid] and cl_eff <= EXP_MASTERY_CL_MAX:
                ok = True
                break
        attempted.add(mid); n_int += n_ex
        if ok:
            mastered.add(mid)
        progress[mid] = (s_pre, s)
    return dict(gain=(s - s0) / (1 - s0) if s0 < 1 else np.nan, n_mastered=len(mastered),
                cl=float(np.mean(cls)) if cls else np.nan, progress=progress,
                final=_vec(prof, s, float(np.mean(cls)) if cls else cl, len(mastered)))

def _streams(seed, i, T):
    rng = np.random.RandomState(seed * 1000 + i)
    return rng.uniform(.8, 1.2, T), rng.normal(0, SENS['cl_noise'], T), rng.uniform(size=T)

def build_peers(seed=2026):
    profs = _draw_profiles(POPULATION_CF, seed)
    PV, DELTA = [], np.zeros((len(profs), len(MODS)))
    for i, p in enumerate(profs):
        xi, nz, u = _streams(seed, i, 400)
        r = _run_one(p, POL_CURRICULUM, xi, nz, u, budget=400)
        for m, (a, b) in r['progress'].items():
            DELTA[i, MIDX[m]] = max(0.0, b - a)
        PV.append(r['final'])
    _Peers.PV = np.array(PV) / np.linalg.norm(PV, axis=1, keepdims=True)
    _Peers.DELTA = DELTA
    return len(profs)

def eval_policy(pol, seed, diag=None):
    profs = _draw_profiles(POPULATION_SYN, seed)[:SENS['n_learners']]
    out = []
    for i, p in enumerate(profs):
        xi, nz, u = _streams(seed, i, SENS['budget'] + 1)
        out.append(_run_one(p, pol, xi, nz, u, SENS['budget'], diag))
    return dict(gain=np.nanmean([o['gain'] for o in out]), n_mastered=np.mean([o['n_mastered'] for o in out]),
                cl=np.nanmean([o['cl'] for o in out]))

def run_policies(policies, seeds):
    rows = []
    for name, pol in policies.items():
        for sd in seeds:
            rows.append(dict(policy=name, seed=sd, **eval_policy(pol, sd)))
    return pd.DataFrame(rows)

n_peers = build_peers()
print(f'✅ R10a prêt — pool de {n_peers} pairs simulés ; budget={SENS["budget"]} exercices ; lam={SENS["lam"]}')

In [ ]:
rng = np.random.RandomState(0)
viol, n_tests = 0, 2000
pairs = [(.7, .3), (.4, .6), (.2, .8), (.5, .5), (.9, .1)]
for _ in range(n_tests):
    s_cb = rng.uniform(size=6); cl = rng.uniform()
    ranks = [tuple(np.argsort(-(w1 * s_cb + w3 * (1 - cl)))) for w1, w3 in pairs]
    viol += len(set(ranks)) > 1
print(f'Classement des exercices identique pour tous les couples (w1,w3), w1>0 : '
      f'{n_tests - viol}/{n_tests} tirages')
print('→ Le terme w3·(1−CL) est le même pour tous les candidats : (w1,w3) ne modifie que l\'échelle du score, '
      'pas le choix. Seul (w2,w4) (choix du module) peut influencer la sélection ; si w1=0 tous les exercices sont à égalité.')
RESULTS['weights_sensitivity_analytic'] = dict(n_tests=n_tests, n_violations=int(viol))

In [ ]:
t0 = time.time()
GRID = np.round(np.linspace(0, 1, 11), 1)
policies = {
    'Curriculum order': POL_CURRICULUM, 'Random candidate': POL_RANDOM,
    'ACTUAL (Eq. 3)': POL_ACTUAL,
    'Uniform 0.5/0.5': sched((.5, .5), (.5, .5), (.5, .5), name='Uniform'),
    'No regime switch (0.6/0.4)': sched((.6, .4), (.6, .4), (.6, .4), name='Static'),
    'Compat only (0/1)': sched((0, 1), (0, 1), (0, 1), name='Compat'),
    'CF only (1/0)': sched((1, 0), (1, 0), (1, 0), name='CF'),
}
for w in GRID:
    policies[f'static w2={w:.1f}'] = sched((w, 1 - w), (w, 1 - w), (w, 1 - w))
for thr in (.5, .7, .8):
    policies[f'ACTUAL, CL thr={thr}'] = sched((.6, .4), (.4, .6), (.6, .4), thr=thr)
for wc, ws in itertools.product([0, .2, .4, .6, .8, 1], repeat=2):
    policies[f'heat crit={wc:.1f} steady={ws:.1f}'] = sched((.6, .4), (wc, 1 - wc), (ws, 1 - ws))

df_sel = run_policies(policies, SENS['seeds_select'])
sel = df_sel.groupby('policy')[['gain', 'n_mastered', 'cl']].mean()
cand_pool = [p for p in policies if p not in ('Curriculum order', 'Random candidate')]
best_name = sel.loc[cand_pool, 'gain'].idxmax()
print(f'⏱️ sélection : {time.time() - t0:.0f}s | meilleur sur graines de SÉLECTION : {best_name} '
      f'(gain={sel.loc[best_name, "gain"]:.4f}) | ACTUAL = {sel.loc["ACTUAL (Eq. 3)", "gain"]:.4f}')

eval_names = ['Curriculum order', 'Random candidate', 'ACTUAL (Eq. 3)', 'Uniform 0.5/0.5',
              'Compat only (0/1)', 'CF only (1/0)', best_name] + [f'static w2={w:.1f}' for w in GRID]
eval_names = list(dict.fromkeys(eval_names))
df_test = run_policies({k: policies[k] for k in eval_names}, SENS['seeds_test'])

def _paired(a_name, b_name, metric='gain'):
    a = df_test[df_test.policy == a_name].sort_values('seed')[metric].values
    b = df_test[df_test.policy == b_name].sort_values('seed')[metric].values
    d = a - b
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return dict(mean_diff=float(d.mean()), ci95=boot_ci(d), p=float(p),
                sd_between_seeds=float(b.std(ddof=1)))

summary = df_test.groupby('policy')[['gain', 'n_mastered', 'cl']].agg(['mean', 'std']).loc[eval_names]
print('\nGraines de TEST (moyenne, SD entre graines) :')
print(summary.round(4).to_string())
cmp = {k: _paired(best_name, k) for k in ['ACTUAL (Eq. 3)', 'Uniform 0.5/0.5', 'Curriculum order', 'Random candidate']
       if k != best_name}
print(f'\nMeilleur ({best_name}) − autre, gain (graines de test) :')
for k, v in cmp.items():
    print(f"  vs {k:<20} Δ={v['mean_diff']:+.4f}  IC95% [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]  p={fmt_p(v['p'])}")
rng_gain = df_test[df_test.policy.str.startswith('static w2')].groupby('policy').gain.mean()
print(f"\nÉtendue du gain sur la grille w2∈[0,1] : {rng_gain.max() - rng_gain.min():.4f} "
      f"(SD entre graines ≈ {df_test[df_test.policy == 'ACTUAL (Eq. 3)'].gain.std():.4f})")

RESULTS['weights_sensitivity'] = dict(
    settings={k: v for k, v in SENS.items() if not k.startswith('seeds')},
    n_seeds_select=len(SENS['seeds_select']), n_seeds_test=len(SENS['seeds_test']),
    best_on_selection=best_name,
    test={k: {m: dict(mean=float(summary.loc[k, (m, 'mean')]), sd=float(summary.loc[k, (m, 'std')]))
              for m in ['gain', 'n_mastered', 'cl']} for k in eval_names},
    best_vs=cmp, grid_range_gain=float(rng_gain.max() - rng_gain.min()))

In [ ]:
diag = []
for sd in SENS['seeds_test'][:10]:
    eval_policy(POL_ACTUAL, sd, diag)
dg = pd.DataFrame(diag)
multi = dg[dg.n_cand >= 2]
res_dec = dict(n_decisions=int(len(dg)), share_with_2plus_candidates=float(len(multi) / max(len(dg), 1)),
               share_top_choice_changes_with_w2=float((multi.n_distinct > 1).mean()) if len(multi) else np.nan,
               mean_vote=float(multi.mean_vote.mean()) if len(multi) else np.nan,
               mean_compat=float(multi.mean_compat.mean()) if len(multi) else np.nan)
RESULTS['weights_sensitivity']['decision_level'] = res_dec
print('Décisions de module (ACTUAL, 10 graines de test) :', json.dumps(res_dec, indent=1))
print('→ part des décisions où le choix change selon w2 ; si faible, les poids pèsent peu. '
      'Vote (somme sur K=10 pairs) et Compat n\'ont pas la même échelle : comparer mean_vote et mean_compat.')

                                                                 
g = df_test[df_test.policy.str.startswith('static w2')]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
xs, ms, lo, hi = [], [], [], []
for w in GRID:
    v = g[g.policy == f'static w2={w:.1f}'].gain.values
    xs.append(w); ms.append(v.mean()); ci = boot_ci(v); lo.append(ci[0]); hi.append(ci[1])
ax[0].plot(xs, ms, 'o-', color=C_BLUE); ax[0].fill_between(xs, lo, hi, alpha=.2, color=C_BLUE)
for name, col, ls in [('ACTUAL (Eq. 3)', C_GREEN, '--'), ('Curriculum order', C_GRAY, ':'), ('Random candidate', C_RED, ':')]:
    ax[0].axhline(df_test[df_test.policy == name].gain.mean(), color=col, ls=ls, label=name)
ax[0].set_xlabel('$w_2$ (weight of CF vote; $w_4=1-w_2$)'); ax[0].set_ylabel('Normalized gain (test seeds)')
ax[0].set_title('Gain vs. static weight'); ax[0].legend(fontsize=8)
                                                                                              
vals = [0, .2, .4, .6, .8, 1]
H = np.array([[sel.loc[f'heat crit={wc:.1f} steady={ws:.1f}', 'gain'] for wc in vals] for ws in vals])
im = ax[1].imshow(H, origin='lower', cmap='viridis')
ax[1].set_xticks(range(6)); ax[1].set_xticklabels(vals); ax[1].set_yticks(range(6)); ax[1].set_yticklabels(vals)
ax[1].set_xlabel('$w_2$ in CL-critical regime'); ax[1].set_ylabel('$w_2$ in steady regime')
ax[1].set_title('Gain (selection seeds); range = %.4f' % (H.max() - H.min()))
fig.colorbar(im, ax=ax[1], fraction=.046); fig.tight_layout(); save_fig(fig, 'fig_weight_sensitivity'); plt.show()

rows_tex = []
for k in ['Curriculum order', 'Random candidate', 'ACTUAL (Eq. 3)', 'Uniform 0.5/0.5', 'Compat only (0/1)', 'CF only (1/0)', best_name]:
    rows_tex.append(f"{k.replace('_', ' ')} & ${summary.loc[k, ('gain', 'mean')]:.3f}\\pm{summary.loc[k, ('gain', 'std')]:.3f}$ & "
                    f"${summary.loc[k, ('n_mastered', 'mean')]:.2f}$ & ${summary.loc[k, ('cl', 'mean')]:.2f}$ \\\\")
TEX_TABLES['weight_sensitivity'] = '\n'.join(dict.fromkeys(rows_tex))
print('\n📋 LaTeX (Gain ± SD entre graines, modules maîtrisés, CL moyen) :\n' + TEX_TABLES['weight_sensitivity'])

In [ ]:
                                                                                                
                                                                                                
_SENS_SAVE = dict(SENS)
scen_pols = {
    'Curriculum': POL_CURRICULUM, 'Random': POL_RANDOM, 'ACTUAL': POL_ACTUAL,
    'Uniform': sched((.5, .5), (.5, .5), (.5, .5)),
    'Compat only': sched((0, 1), (0, 1), (0, 1)), 'CF only': sched((1, 0), (1, 0), (1, 0)),
    'w2=0.2': sched((.2, .8), (.2, .8), (.2, .8)), 'w2=0.8': sched((.8, .2), (.8, .2), (.8, .2)),
}
rows_sc = []
for eta, lam, bud in itertools.product([0.05, 0.10, 0.18], [0.5, 1.5], [12, 24]):
    SENS.update(eta=eta, lam=lam, budget=bud); build_peers()
    d = run_policies(scen_pols, SENS['seeds_test'][:15])
    m = d.groupby('policy')[['gain', 'n_mastered']].mean()
    wset = ['ACTUAL', 'Uniform', 'Compat only', 'CF only', 'w2=0.2', 'w2=0.8']
    sd_seed = d[d.policy == 'ACTUAL'].gain.std()
    rows_sc.append(dict(eta=eta, lam=lam, budget=bud,
                        gain_actual=m.loc['ACTUAL', 'gain'], gain_random=m.loc['Random', 'gain'],
                        gain_curriculum=m.loc['Curriculum', 'gain'],
                        max_gap_weight_sets=m.loc[wset, 'gain'].max() - m.loc[wset, 'gain'].min(),
                        best_weight_set=m.loc[wset, 'gain'].idxmax(),
                        mastered_actual=m.loc['ACTUAL', 'n_mastered'], sd_between_seeds=sd_seed))
SENS.update(_SENS_SAVE); build_peers()                                           
sc = pd.DataFrame(rows_sc)
print(sc.round(4).to_string(index=False))
print(f"\nÉcart maximal de gain entre jeux de poids, tous scénarios : {sc.max_gap_weight_sets.max():.4f} "
      f"(gain moyen ≈ {sc.gain_actual.mean():.2f}) ; écart Random − ACTUAL maximal : "
      f"{(sc.gain_random - sc.gain_actual).abs().max():.4f}")
RESULTS['weights_sensitivity']['scenarios'] = sc.to_dict(orient='records')
TEX_TABLES['weight_scenarios'] = '\n'.join(
    f"{r.eta:.2f} & {r.lam:.1f} & {int(r.budget)} & ${r.gain_actual:.3f}$ & ${r.max_gap_weight_sets:.4f}$ & "
    f"${r.gain_random - r.gain_actual:+.4f}$ \\\\" for r in sc.itertuples())
print('\n📋 LaTeX (η, λ, budget, gain ACTUAL, écart max entre jeux de poids, Random−ACTUAL) :\n' + TEX_TABLES['weight_scenarios'])

## 🔎 Partie 6 — Vérification et régénération des figures de l'article

Recalcule depuis la base les figures *CL audit*, *pré/post* et *DAG des modules*, contrôle (`assert`) leurs chiffres clés et sauvegarde PNG + PDF dans `article_results/`.

In [ ]:
                                                                                
                                                          
                                                                                           
                                                                                   
                                                                        
                                                                                
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch

FIG_DIR = 'article_results'; os.makedirs(FIG_DIR, exist_ok=True)
XLSX_CANDIDATES = ['AILS_base_corrigee.xlsx', 'AILS_base_corrigee (1)(1).xlsx', 'AILS_base_corrigee (1).xlsx', 'AILS_base_corrigee__1_.xlsx', 'AILS_Database_corrigee.xlsx']
BLUE, RED, ORANGE = '#2c5282', '#c0392b', '#e67e22'
CL_CRITICAL = 0.60
REVERSED_ITEMS = ['Q6_raw', 'Q7_raw']                                                            

def _load(tab):
    """Google Sheet (variable `sh`) → sinon fichier xlsx local."""
    try:
        return pd.DataFrame(sh.worksheet(tab).get_all_records())
    except Exception:
        pass
    _p = find_ails_xlsx(XLSX_CANDIDATES)                                                                      
    if _p: return pd.read_excel(_p, sheet_name=tab)
    raise FileNotFoundError(f"Onglet '{tab}' introuvable (Sheet ou xlsx).")

def _style():
    return plt.rc_context({'font.family': 'DejaVu Sans', 'font.size': 11,
                           'axes.spines.top': False, 'axes.spines.right': False,
                           'savefig.facecolor': 'white'})

def _save(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight')
    print(f'   💾 {FIG_DIR}/{name}.png / .pdf')

                                                                              
def figure_cl_audit():
    cl = _load('Cognitive_Load').copy()
    Q = [f'Q{i}_raw' for i in range(1, 11)]
    items = cl[Q].astype(float)
    for c in REVERSED_ITEMS:
        items[c] = 8 - items[c]
    cl['sd_items'] = items.std(axis=1, ddof=1)                                                     
    assert 'compute_clq10' in globals(), 'exécuter d’abord la cellule CLq (compute_clq10)'
    cl['clq']      = cl[Q].apply(lambda r: compute_clq10(list(r)), axis=1)                                    
                                                                                                                                             
    DAY_PRE, DAY_POST = 0, 22                                                                                  
    assert set(cl['day_index'].unique()) <= {DAY_PRE, DAY_POST}, f"jours inattendus : {sorted(cl['day_index'].unique())}"
    cl['retained'] = cl['day_index'] == DAY_PRE
    ret, exc = cl[cl.retained], cl[~cl.retained].copy()
    exc['q_num'] = exc.sort_values(['learner_id', 'Exercise_ID']).groupby('learner_id').cumcount() + 1

                                           
    assert len(ret) == 17 and ret.learner_id.nunique() == 17, 'jour 0 ≠ 17'
    assert len(exc) == 39, 'jour 22 ≠ 39'
    assert exc.learner_id.nunique() == 12, 'étudiants du jour exclu ≠ 12'
    assert (cl.clq - cl['CL_Score']).abs().max() < 1e-3, 'CL_q recalculé ≠ colonne CL_Score'
    print(f"[A] jour 0 n={len(ret)} | jour 22 n={len(exc)} ({exc.learner_id.nunique()} étudiants)")
    print(f"    SD intra : jour 0 médiane={ret.sd_items.median():.2f} [{ret.sd_items.min():.2f}–{ret.sd_items.max():.2f}] | "
          f"jour 22 médiane={exc.sd_items.median():.2f} [{exc.sd_items.min():.2f}–{exc.sd_items.max():.2f}]")
    print(f"    CL_q jour 0 : médiane={ret.clq.median():.3f}, max={ret.clq.max():.2f} "
          f"({int((ret.clq > CL_CRITICAL).sum())} au-dessus de {CL_CRITICAL})")

    rng = np.random.RandomState(0)
    with _style():
        fig, ax = plt.subplots(1, 3, figsize=(15.5, 4.6), gridspec_kw={'width_ratios': [1.05, 1.1, 1.3]})
             
        bp = ax[0].boxplot([ret.sd_items, exc.sd_items], positions=[1, 2], widths=.5, showfliers=False,
                           medianprops=dict(color=ORANGE, lw=1.5))
        for x, d, c in [(1, ret.sd_items, BLUE), (2, exc.sd_items, RED)]:
            ax[0].scatter(x + rng.uniform(-.08, .08, len(d)), d, s=18, color=c, alpha=.8, zorder=3)
        ax[0].set_xticks([1, 2]); ax[0].set_xticklabels([f'Day 0\n(n={len(ret)})', f'Day 22\n(n={len(exc)})'])
        ax[0].set_ylabel('SD of the 10 items within a questionnaire'); ax[0].set_title('(a) Within-questionnaire spread', fontsize=11)
             
        ax[1].boxplot(ret.clq, positions=[1], widths=.5, showfliers=False, medianprops=dict(color=ORANGE, lw=1.5))
        ax[1].scatter(1 + rng.uniform(-.08, .08, len(ret)), ret.clq, s=22, color=BLUE, alpha=.85, zorder=3)
        ax[1].axhline(CL_CRITICAL, color=ORANGE, ls='--', lw=1.3, label=f'CL-critical ({CL_CRITICAL:.2f})')
        ax[1].set_ylim(0, 1); ax[1].set_xticks([1]); ax[1].set_xticklabels([f'Day-0 questionnaires\n({len(ret)} students)'])
        ax[1].set_ylabel('$CL_q$'); ax[1].set_title('(b) Day-0 CL scores', fontsize=11); ax[1].legend(frameon=False, loc='upper right', fontsize=8)
             
        for _, g in exc.sort_values('q_num').groupby('learner_id'):
            ax[2].plot(g.q_num, g.clq, '-o', color=RED, alpha=.7, lw=1.2, ms=3.5)
        ax[2].set_xticks([1, 2, 3, 4]); ax[2].set_ylim(0, 1); ax[2].set_xlabel('Questionnaire number (same student)')
        ax[2].set_ylabel('$CL_q$'); ax[2].set_title(f'(c) Day 22: trajectories ({exc.learner_id.nunique()} students)', fontsize=11)
        fig.tight_layout(); _save(fig, 'figure_cl_audit'); plt.show()

                                                                           
def figure_prepost():
    try:
        d = _load('Gains_Apprentissage')[['learner_id', 'pre_score', 'post_score']]
    except Exception:                                                            
        f1 = _load('Form Responses 1')
        d = pd.DataFrame({'pre_score': f1['Pre-test score (/20)'], 'post_score': f1['Post-test score (/20)']})
    d = d.apply(pd.to_numeric, errors='coerce').dropna(subset=['pre_score', 'post_score'])
    pre, post = d.pre_score.values, d.post_score.values
    lr = stats.linregress(pre, post)
    assert len(d) == 33, f'n={len(d)} ≠ 33'
    assert round(lr.slope, 2) == 0.74 and round(lr.intercept, 1) == 28.2, 'droite ≠ post = 0.74·pre + 28.2'
    print(f"[B] n={len(d)} | pré={pre.mean():.1f}±{pre.std(ddof=1):.1f} | post={post.mean():.1f}±{post.std(ddof=1):.1f} | "
          f"post = {lr.slope:.2f}·pre + {lr.intercept:.1f} (R²={lr.rvalue**2:.2f})")
    print(f"    ⚠️ Pré/post sont sur 0–100 ici ; la colonne source de Form Responses 1 est étiquetée « (/20) » → à harmoniser.")

    with _style():
        fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={'width_ratios': [1.05, 1]})
        for a, b in zip(pre, post):
            ax[0].plot([0, 1], [a, b], '-o', color='#a0aec0', lw=1, ms=3, alpha=.9)
        ax[0].plot([0, 1], [pre.mean(), post.mean()], '-o', color='#1f4e79', lw=4, ms=8)
        ax[0].set_xlim(-.25, 1.25); ax[0].set_ylim(0, 100); ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['Pre-test', 'Post-test'])
        ax[0].set_ylabel('Score (0-100)'); ax[0].set_title(f'(a) Individual trajectories (n={len(d)})', fontsize=11)
        ax[1].scatter(pre, post, s=40, color=BLUE, alpha=.9, zorder=3)
        ax[1].plot([0, 100], [0, 100], '--', color='gray', lw=1.2, label='No change')
        xs = np.array([pre.min(), pre.max()])
        ax[1].plot(xs, lr.slope * xs + lr.intercept, color=RED, lw=2.5, label=f'Fit: post = {lr.slope:.2f} pre + {lr.intercept:.1f}')
        ax[1].set_xlim(0, 100); ax[1].set_ylim(0, 100); ax[1].set_xlabel('Pre-test score'); ax[1].set_ylabel('Post-test score')
        ax[1].set_title('(b) Post-test vs. pre-test', fontsize=11); ax[1].legend(frameon=False, loc='lower right', fontsize=9)
        fig.tight_layout(); _save(fig, 'figure_prepost'); plt.show()

                                                                                
def figure_dag():
    mods = globals().get('MODULES') or {                                                             
        'M01': dict(title='Variables & types', difficulty=1, prerequisites=[], theta=.75),
        'M02': dict(title='Conditionals', difficulty=1, prerequisites=['M01'], theta=.75),
        'M03': dict(title='Loops', difficulty=2, prerequisites=['M01', 'M02'], theta=.75),
        'M04': dict(title='Arrays', difficulty=2, prerequisites=['M01', 'M03'], theta=.78),
        'M05': dict(title='Functions', difficulty=3, prerequisites=['M02', 'M03'], theta=.78),
        'M06': dict(title='Recursion', difficulty=4, prerequisites=['M05'], theta=.80),
        'M07': dict(title='Matrices', difficulty=3, prerequisites=['M04'], theta=.78),
        'M08': dict(title='Formal logic', difficulty=3, prerequisites=['M02'], theta=.80),
        'M09': dict(title='Complexity & sorting', difficulty=4, prerequisites=['M03', 'M05'], theta=.82),
        'M10': dict(title='Graphs', difficulty=5, prerequisites=['M06', 'M07', 'M09'], theta=.82)}
    EN = {'M01': 'Variables\n& types', 'M02': 'Conditionals', 'M03': 'Loops', 'M04': 'Arrays', 'M05': 'Functions',
          'M06': 'Recursion', 'M07': 'Matrices', 'M08': 'Formal\nlogic', 'M09': 'Complexity\n& sorting', 'M10': 'Graphs'}
    edges = [(p, m) for m, v in mods.items() for p in v['prerequisites']]
    print(f"[C] {len(mods)} modules, {len(edges)} arêtes (déduites de MODULES) : " + ', '.join(f'{a}→{b}' for a, b in edges))

    col = {'M01': 0, 'M02': 1, 'M03': 2, 'M08': 2, 'M04': 3, 'M05': 3, 'M06': 4, 'M07': 4, 'M09': 4, 'M10': 5}
    row = {'M06': 3, 'M03': 1.5, 'M04': 1.5, 'M01': 0, 'M02': 0, 'M07': 0, 'M10': 0, 'M08': -1.5, 'M05': -1.5, 'M09': -3}
    DX, W, H = 3.1, 2.25, 1.0
    pos = {m: (col[m] * DX, row[m]) for m in mods}
    fills = {1: '#d6e9f8', 2: '#aacbe2', 3: '#7fb1d3', 4: '#5399c8', 5: '#2e6da4'}
    box = lambda m: (pos[m][0] - W / 2, pos[m][0] + W / 2, pos[m][1] - H / 2, pos[m][1] + H / 2)

    def bezier(p0, p1, rad, n=40):                                                          
        (x0, y0), (x1, y1) = p0, p1
        cx, cy = (x0 + x1) / 2 + rad * (y1 - y0), (y0 + y1) / 2 - rad * (x1 - x0)
        t = np.linspace(0, 1, n)[:, None]
        return (1 - t) ** 2 * np.array(p0) + 2 * (1 - t) * t * np.array([cx, cy]) + t ** 2 * np.array(p1)

    def hits(curve, skip):                                                                          
        k = 0
        for m in mods:
            if m in skip: continue
            x0, x1, y0, y1 = box(m)
            k += int(((curve[:, 0] > x0 - .05) & (curve[:, 0] < x1 + .05) & (curve[:, 1] > y0 - .05) & (curve[:, 1] < y1 + .05)).sum())
        return k

    with _style():
        fig, ax = plt.subplots(figsize=(15, 6.4)); ax.set_aspect('equal'); ax.axis('off')
        ax.set_xlim(-1.4, 5 * DX + 1.4); ax.set_ylim(-4.6, 3.8)
        patches = {}
        for m, v in mods.items():
            x0, x1, y0, y1 = box(m)
            pilot = (m == 'M04')
            p = FancyBboxPatch((x0, y0), W, H, boxstyle='round,pad=0,rounding_size=.12', zorder=3,
                               fc=fills[v['difficulty']], ec=RED if pilot else '#2d4059', lw=4 if pilot else 1.3)
            ax.add_patch(p); patches[m] = p
            tc = 'white' if v['difficulty'] >= 4 else 'black'
            ax.text(pos[m][0], pos[m][1] + .13, f"{m}  {EN[m]}", ha='center', va='center', fontsize=9.5, color=tc, zorder=4)
            ax.text(pos[m][0], pos[m][1] - .30, rf"$\theta$={v['theta']:.2f}", ha='center', va='center', fontsize=9.5, color=tc, zorder=4)
        n_bad = 0
        for a, b in edges:
            best = min(((hits(bezier(pos[a], pos[b], r), {a, b}), abs(r), r)
                        for r in [0, .12, -.12, .22, -.22, .35, -.35, .5, -.5]))
            n_bad += best[0] > 0
            ax.add_patch(FancyArrowPatch(pos[a], pos[b], connectionstyle=f'arc3,rad={best[2]}', arrowstyle='-|>', mutation_scale=14,
                                         lw=1.6, color='#555', patchA=patches[a], patchB=patches[b], shrinkA=2, shrinkB=2, zorder=2))
        if n_bad: print(f'   ⚠️ {n_bad} arête(s) frôlent encore un module — ajuster `row`/`col`.')
        leg = [Patch(fc=fills[d], ec='#2d4059', label=f'Difficulty {d}') for d in range(1, 6)] +\
              [Patch(fc='white', ec=RED, lw=3, label='Module of the pilot (M04)')]
        ax.legend(handles=leg, loc='lower left', ncol=3, frameon=False, fontsize=10, bbox_to_anchor=(0, -.02))
        _save(fig, 'figure_dag'); plt.show()

for _fn in (figure_cl_audit, figure_prepost, figure_dag):
    try:
        _fn()
    except AssertionError as e:
        print(f'❌ {_fn.__name__} : la figure publiée ne correspond pas aux données → {e}')


## 📊 Partie 7 — Calcul des résultats de l'article (script pour les reviewers)

Cette cellule calcule, depuis la base normalisée, **tous les résultats rapportés dans l'article** : participants, questionnaires CL (les 56 sont analysés, aucun n'est écarté), flux des données, test pré/post (RQ1), questionnaire CL (RQ2), logs de décision (RQ3) et Annexe C (sensibilité de α et des corrélations aux sous-ensembles de questionnaires ; nombre de profils de réponses partagés comparé à celui attendu par hasard). Les moyennes par étudiant sont arrondies à 1e-10 avant tout classement pour que les ex æquo exacts restent des ex æquo. Rien n'est saisi à la main ; les résultats sont affichés et exportés dans `article_results/resultats_article.json`. Les Annexes A et B (simulations) sont produites par les cellules 9 à 15.

In [ ]:
                                                                                
                                                                           
                                                                         
                                                                                                                                 
                                                                                      
                                                                                   
                                                                                
import os, json, numpy as np, pandas as pd
from scipy import stats

XLSX_CANDIDATES = ['AILS_base_corrigee.xlsx', 'AILS_base_corrigee (1)(1).xlsx', 'AILS_base_corrigee (1).xlsx', 'AILS_base_corrigee__1_.xlsx', 'AILS_Database_corrigee.xlsx']
N_BOOT, SEED = 5000, 42
REVERSED = ['Q6_raw', 'Q7_raw']                                                                     
THRESH = [0.30, 0.60, 0.80]                                                            

def _tab(name):
    try:
        return pd.DataFrame(sh.worksheet(name).get_all_records())
    except Exception:
        pass
    _p = find_ails_xlsx(XLSX_CANDIDATES)                                                                      
    if _p: return pd.read_excel(_p, sheet_name=name)
    raise FileNotFoundError(f"Onglet '{name}' introuvable (Sheet ou xlsx).")

def boot_ci(v, stat, n=N_BOOT, seed=SEED):
    """IC95% bootstrap percentile ; `stat` s'applique ligne par ligne à la matrice des rééchantillons."""
    v = np.asarray(v, float); idx = np.random.RandomState(seed).randint(0, len(v), (n, len(v)))
    return tuple(float(q) for q in np.nanpercentile(stat(v[idx]), [2.5, 97.5]))

def cronbach(M):
    M = np.asarray(M, float); k = M.shape[1]
    return k / (k - 1) * (1 - M.var(0, ddof=1).sum() / M.sum(1).var(ddof=1))

def H(t): print('\n' + '═' * 78 + f'\n{t}\n' + '═' * 78)
R = {}                                                                                                

assert 'compute_clq10' in globals(), 'exécuter d’abord la cellule CLq (compute_clq10)'
prof, cl, logs = _tab('Learner_Profile'), _tab('Cognitive_Load'), _tab('AILS_Results')
gain, form1 = _tab('Gains_Apprentissage').set_index('learner_id'), _tab('Form Responses 1')
pre, post = gain.pre_score.astype(float), gain.post_score.astype(float)

                                                                                
H('1. PARTICIPANTS (§4.2)')
R['participants'] = dict(n=len(prof), age_mean=prof.Age.mean(), age_sd=prof.Age.std(ddof=1),
                         age_min=int(prof.Age.min()), age_max=int(prof.Age.max()),
                         gender=prof.Gender.value_counts().to_dict(), education=prof.Education_Level.value_counts().to_dict())
p = R['participants']
print(f"n = {p['n']} | âge {p['age_mean']:.1f} ± {p['age_sd']:.1f} (de {p['age_min']} à {p['age_max']})")
print('Genre     :', p['gender']); print('Éducation :', p['education'])
dd = form1['Day index']
print(f"Formulaire profil (Form 1) : jours relatifs de {int(dd.min())} à {int(dd.max())} ; répartition {dd.value_counts().sort_index().to_dict()}")

                                                                                          
H('2. QUESTIONNAIRES CL — AUDIT QUALITÉ (§4.4, tableau « data flow », figure « CL audit »)')
Q = [f'Q{i}_raw' for i in range(1, 11)]
items = cl[Q].astype(float).copy()
for c in REVERSED: items[c] = 8 - items[c]                                                                   
X = (items - 1) / 6                                                                        
cl['clq'] = cl[Q].apply(lambda r: compute_clq10(list(r)), axis=1).values                                    
assert np.allclose(cl['clq'].values, X.mean(axis=1).values, equal_nan=True), 'CLq ≠ moyenne des items recodés'
                                                                                                                                           
cl['sd_items'] = items.std(axis=1, ddof=1).values                                                               
cl['vec'] = cl[Q].astype(int).astype(str).agg('-'.join, axis=1)                                
DAY_PRE, DAY_POST = 0, 22                                                                                                               
d0  = (cl.day_index == DAY_PRE).values
d22 = (cl.day_index == DAY_POST).values
_other = ~(d0 | d22)
if _other.any(): print(f"⚠️ {int(_other.sum())} questionnaire(s) hors jours {DAY_PRE}/{DAY_POST} (jours {sorted(cl.day_index[_other].unique().tolist())}) : exclus des comparaisons pré/post, conservés dans alpha_all")
assert d0.any() and d22.any(), f'jours attendus {DAY_PRE} et {DAY_POST} absents : {sorted(cl.day_index.unique().tolist())}'                                                                             
n_per = cl[d22].groupby('learner_id').size(); mins = cl.loc[d22, 'minutes_since_previous_form'].dropna()
n_stud = cl.groupby('vec').learner_id.nunique()                                                                    
dup = cl[d22].vec.isin(n_stud[n_stud >= 2].index)                                                               
R['audit_cl'] = dict(n_forms=len(cl), n_students=cl.learner_id.nunique(), n_day0=int(d0.sum()), n_day22=int(d22.sum()),
                     n_students_day22=cl[d22].learner_id.nunique(), forms_per_student_day22=sorted(n_per.unique().tolist()),
                     interval_min_median=float(mins.median()), interval_min_range=[float(mins.min()), float(mins.max())],
                     sd_items_mean_day0=float(cl.sd_items[d0].mean()), sd_items_mean_day22=float(cl.sd_items[d22].mean()),
                     alpha_day0=cronbach(X[d0]), alpha_day22=cronbach(X[d22]), alpha_all=cronbach(X),
                     day22_forms_duplicated_across_students=int(dup.sum()), n_duplicated_patterns=int((n_stud >= 2).sum()))
a = R['audit_cl']
print(f"{a['n_forms']} questionnaires de {a['n_students']} étudiants : {a['n_day0']} au jour 0 (1/étudiant) et {a['n_day22']} au jour 22 (jour du post-test) — les 56 sont analysés")
print(f"Jour 22 : {a['n_students_day22']} étudiants, {a['forms_per_student_day22']} questionnaires chacun ; intervalle médian {a['interval_min_median']:.0f} min (de {a['interval_min_range'][0]:.0f} à {a['interval_min_range'][1]:.0f})")
print(f"SD intra-questionnaire moyenne : jour 0 {a['sd_items_mean_day0']:.2f} | jour 22 {a['sd_items_mean_day22']:.2f}")
print(f"α de Cronbach : jour 0 {a['alpha_day0']:.2f} | jour 22 {a['alpha_day22']:.3f} | 56 questionnaires {a['alpha_all']:.3f}")
print(f"Profils de réponses identiques entre étudiants : {a['n_duplicated_patterns']} profils, {a['day22_forms_duplicated_across_students']} des {a['n_day22']} questionnaires du jour 22 (au jour 0 : {int(cl[d0].vec.isin(n_stud[n_stud >= 2].index).sum())})")


                                                                                                                             
def n_shared(Rm, L):
    g = {}
    for r, l in zip(map(tuple, Rm), L): g.setdefault(r, set()).add(l)
    pats = [p for p, s in g.items() if len(s) >= 2]
    return len(pats), sum(len(g[tuple(r)]) >= 2 for r in Rm)
R22, L22 = cl.loc[d22, Q].astype(int).values, cl.loc[d22, 'learner_id'].values
Z = items[d22].values                                                                                      
mu, sdv = Z.mean(1), Z.std(1, ddof=1); rng = np.random.default_rng(SEED)
def simulate(model, B=2000):
    out = []
    for _ in range(B):
        if model == 'shuffle':                                                                                           
            S = np.array([rng.permutation(z) for z in Z])
        else:                                                                                                                          
            k = rng.integers(0, len(Z), len(Z)); S = np.clip(np.rint(rng.normal(mu[k][:, None], np.maximum(sdv[k][:, None], 0.3), (len(Z), 10))), 1, 7)
        S = S.copy(); S[:, [5, 6]] = 8 - S[:, [5, 6]]                                         
        out.append(n_shared(S.astype(int), L22))
    return np.array(out)
obs_pat, obs_forms = n_shared(R22, L22)
R['coincidences_jour22'] = dict(observed_patterns=obs_pat, observed_forms=obs_forms)
print(f"\nCoïncidences entre étudiants au jour 22 : observé = {obs_pat} profils partagés ({obs_forms} questionnaires). Attendu sous deux modèles de hasard (2000 simulations) :")
for mdl, lab in [('shuffle', "ordre des réponses mélangé dans chaque formulaire"), ('noise', "niveau + bruit discrétisé")]:
    o = simulate(mdl)
    R['coincidences_jour22'][mdl] = dict(expected_patterns=float(o[:, 0].mean()), ci95=[float(v) for v in np.percentile(o[:, 0], [2.5, 97.5])], p_ge_observed=float((o[:, 0] >= obs_pat).mean()))
    print(f"  {lab:<52s}: {o[:,0].mean():.1f} profils [{np.percentile(o[:,0],2.5):.0f}–{np.percentile(o[:,0],97.5):.0f}] ; P(≥ observé) = {(o[:,0] >= obs_pat).mean():.3f}")

logged_ids = set(logs.learner_id)
print('\nTableau « Data flow of the pilot »')
flow = [('Étudiants inscrits (comptes de test retirés)', len(prof)), ('Étudiants avec pré-test et post-test', len(gain)),
        ('Étudiants avec ≥1 questionnaire CL (56 questionnaires)', cl.learner_id.nunique()),
        ('Questionnaires CL au jour 0 (1 par étudiant)', f"{d0.sum()} ({cl[d0].learner_id.nunique()} étudiants)"),
        ('Questionnaires CL au jour 22 (3 ou 4 par étudiant)', f"{d22.sum()} ({cl[d22].learner_id.nunique()} étudiants)"),
        ('Étudiants avec décisions loguées', len(logged_ids & set(gain.index))),
        ('Étudiants avec les deux tests et sans décision loguée', len(set(gain.index) - logged_ids)),
        ('Étudiants sans aucun questionnaire CL', len(set(gain.index) - set(cl.learner_id)))]
for k, v in flow: print(f'  {k:<58s} {v}')
R['flux'] = dict(flow)

                                                                                         
H('3. RQ1 — TEST ÉCRIT PRÉ / POST (tableau « Written pre/post test », figure « Pre/post test »)')
d = (post - pre).values; ng = d / (100 - pre.values)                                                  
dz = d.mean() / d.std(ddof=1)
ci_d, ci_ng = boot_ci(d, lambda m: m.mean(1)), boot_ci(ng, lambda m: m.mean(1))
ci_dz = boot_ci(d, lambda m: m.mean(1) / m.std(1, ddof=1))
w = stats.wilcoxon(d); rp, rs = stats.pearsonr(pre, d), stats.spearmanr(pre, d)
lr = stats.linregress(pre, post)
pk = prof.set_index('learner_id').loc[gain.index, 'Q_PriorKnowledge']; r_pk = stats.spearmanr(pk, pre)
R['rq1'] = dict(n=len(d), pre_mean=pre.mean(), pre_sd=pre.std(ddof=1), pre_range=[float(pre.min()), float(pre.max())],
                post_mean=post.mean(), post_sd=post.std(ddof=1), post_range=[float(post.min()), float(post.max())],
                diff_mean=d.mean(), diff_sd=d.std(ddof=1), diff_ci95=ci_d, diff_range=[float(d.min()), float(d.max())], n_improved=int((d > 0).sum()),
                ng_mean=ng.mean(), ng_sd=ng.std(ddof=1), ng_ci95=ci_ng,
                ratio_of_means=(post.mean() - pre.mean()) / (100 - pre.mean()), dz=dz, dz_ci95=ci_dz,
                wilcoxon_W=float(w.statistic), wilcoxon_p=float(w.pvalue), pearson_pre_diff=rp[0], spearman_pre_diff=rs[0],
                fit_slope=lr.slope, fit_intercept=lr.intercept, fit_r2=lr.rvalue ** 2,
                prior_knowledge_vs_pre=dict(rho=r_pk[0], p=r_pk[1]))
print(f"{'Quantité':<52s}Valeur")
print(f"{'Pré-test, moyenne (SD) [min–max]':<52s}{pre.mean():.1f} ({pre.std(ddof=1):.1f}) [{pre.min():.0f}–{pre.max():.0f}]")
print(f"{'Post-test, moyenne (SD) [min–max]':<52s}{post.mean():.1f} ({post.std(ddof=1):.1f}) [{post.min():.0f}–{post.max():.0f}]")
print(f"{'Différence moyenne (SD) [IC95%]':<52s}{d.mean():.1f} ({d.std(ddof=1):.1f}) [{ci_d[0]:.1f}, {ci_d[1]:.1f}]   (min {d.min():.0f}, max {d.max():.0f} ; {int((d>0).sum())}/{len(d)} en hausse)")
print(f"{'Gain normalisé ΔG moyen (SD) [IC95%]':<52s}{ng.mean():.3f} ({ng.std(ddof=1):.3f}) [{ci_ng[0]:.3f}, {ci_ng[1]:.3f}]")
print(f"{'Rapport des moyennes (post−pré)/(100−pré)':<52s}{R['rq1']['ratio_of_means']:.3f}")
print(f"{'d_z apparié [IC95%]':<52s}{dz:.2f} [{ci_dz[0]:.2f}, {ci_dz[1]:.2f}]")
print(f"{'Wilcoxon (bilatéral)':<52s}W = {w.statistic:.0f}, p = {w.pvalue:.2e}")
print(f"{'Corrélation pré ↔ différence':<52s}Pearson r = {rp[0]:.2f} | Spearman ρ = {rs[0]:.2f}")
print(f"{'Droite post = a·pré + b':<52s}post = {lr.slope:.2f}·pré + {lr.intercept:.1f}  (R² = {lr.rvalue**2:.2f})")
print(f"{'Connaissance préalable déclarée ↔ pré-test':<52s}Spearman ρ = {r_pk[0]:.2f}, p = {r_pk[1]:.2f}")
print('\nDescription par niveau d\'éducation :')
edu = prof.set_index('learner_id').loc[gain.index, 'Education_Level']
print(pd.DataFrame({'pré': pre, 'post': post, 'diff': d, 'ΔG': ng}).groupby(edu.values).agg(['count', 'mean']).round(2).iloc[:, [0, 1, 3, 5, 7]].to_string())

print('\nExposition aux logs AILS :')
lg = gain.index.isin(logged_ids); mw = stats.mannwhitneyu(d[lg], d[~lg], alternative='two-sided')
R['exposition'] = dict(n_logged=int(lg.sum()), n_not_logged=int((~lg).sum()),
                       diff_logged_mean=d[lg].mean(), diff_logged_sd=d[lg].std(ddof=1), diff_not_logged_mean=d[~lg].mean(), diff_not_logged_sd=d[~lg].std(ddof=1),
                       mannwhitney_p=float(mw.pvalue), ng_logged=ng[lg].mean(), ng_not_logged=ng[~lg].mean())
e = R['exposition']
print(f"  Avec décision loguée (n={e['n_logged']}) : {e['diff_logged_mean']:.1f} pts (SD {e['diff_logged_sd']:.1f}) | ΔG {e['ng_logged']:.2f}")
print(f"  Sans décision loguée (n={e['n_not_logged']}) : {e['diff_not_logged_mean']:.1f} pts (SD {e['diff_not_logged_sd']:.1f}) | ΔG {e['ng_not_logged']:.2f}")
print(f"  Mann–Whitney bilatéral : p = {mw.pvalue:.2f}")

                                                                                                                                
H('4. RQ2 — QUESTIONNAIRE DE CHARGE COGNITIVE (§5.3) — les 56 questionnaires')
def alpha_cluster_ci(M, ids, B=N_BOOT, seed=SEED):
    """IC95% bootstrap de α en rééchantillonnant les ÉTUDIANTS (et tous leurs questionnaires)."""
    r = np.random.RandomState(seed); u = np.unique(ids); ix = {s: np.where(ids == s)[0] for s in u}
    v = [cronbach(M[np.concatenate([ix[s] for s in r.choice(u, len(u))])]) for _ in range(B)]
    return np.nanpercentile(v, [2.5, 97.5])
al_lo, al_hi = alpha_cluster_ci(X.values, cl.learner_id.values)                                                      
al_lo_rows, al_hi_rows = boot_ci(np.arange(len(cl)), lambda I: np.array([cronbach(X.values[i]) for i in I.astype(int)]))                                                
la = cl.groupby('learner_id').clq.mean().round(10)                                                                         
ng_s = pd.Series(ng, index=gain.index)
r_pre, r_ng = stats.spearmanr(pre.loc[la.index], la), stats.spearmanr(ng_s.loc[la.index], la)
e = cl[d22].sort_values(['learner_id', 'Exercise_ID']).copy(); e['pos'] = e.groupby('learner_id').cumcount() + 1
fl = e.groupby('learner_id').clq.agg(['first', 'last']).round(10); wl = stats.wilcoxon(fl['first'], fl['last'])
by_pos = e.groupby('pos').clq.agg(['count', 'mean'])
R['rq2'] = dict(n_forms=len(cl), n_students=int(la.size), alpha=cronbach(X), alpha_ci95=[al_lo, al_hi], alpha_ci95_rows_naive=[al_lo_rows, al_hi_rows],
                clq_form_mean=cl.clq.mean(), clq_form_sd=cl.clq.std(ddof=1), n_forms_above_060=int((cl.clq > 0.60).sum()),
                clq_learner_mean=la.mean(), clq_learner_sd=la.std(ddof=1), clq_learner_range=[float(la.min()), float(la.max())],
                n_learners_above_060=int((la > 0.60).sum()),
                spearman_clq_pre=dict(rho=r_pre[0], p=r_pre[1]), spearman_clq_ng=dict(rho=r_ng[0], p=r_ng[1]),
                within_learner=dict(n=len(fl), first_mean=fl['first'].mean(), last_mean=fl['last'].mean(), mean_change=(fl['last'] - fl['first']).mean(),
                                    n_increased=int((fl['last'] > fl['first']).sum()), wilcoxon_p=float(wl.pvalue),
                                    mean_by_position={int(k): [int(v['count']), float(v['mean'])] for k, v in by_pos.iterrows()}))
print(f"{len(cl)} questionnaires, {la.size} étudiants : α = {cronbach(X):.2f} [IC95% {al_lo:.2f}, {al_hi:.2f}]  (IC bootstrap par étudiant ; α à interpréter avec prudence : profils de réponses dupliqués, voir Annexe C)")
print(f"CL_q par questionnaire : moyenne {cl.clq.mean():.2f} (SD {cl.clq.std(ddof=1):.2f}) ; {int((cl.clq>0.60).sum())}/{len(cl)} ({(cl.clq>0.60).mean():.0%}) au-dessus de 0,60")
print(f"CL_q par étudiant     : moyenne {la.mean():.2f} (SD {la.std(ddof=1):.2f}), min {la.min():.2f}, max {la.max():.2f} ; {int((la>0.60).sum())} au-dessus de 0,60")
print(f"Spearman CL_q (étudiant) ↔ pré-test        : ρ = {r_pre[0]:.2f}, p = {r_pre[1]:.2f}")
print(f"Spearman CL_q (étudiant) ↔ gain normalisé  : ρ = {r_ng[0]:.2f}, p = {r_ng[1]:.2f}   (exploratoire, non corrigé pour la multiplicité)")
print(f"\nIntra-étudiant ({len(fl)} étudiants avec plusieurs questionnaires, jour 22) : 1ᵉʳ = {fl['first'].mean():.2f} → dernier = {fl['last'].mean():.2f} ; "
      f"{int((fl['last']>fl['first']).sum())}/{len(fl)} en hausse ; Wilcoxon p = {wl.pvalue:.4f}")
print('  moyenne par rang du questionnaire :', {int(k): (int(v['count']), round(float(v['mean']), 2)) for k, v in by_pos.iterrows()})

                                                                                               
H('5. RQ3 — LOGS DE DÉCISION (tableau « Logged decisions », §5.4)')
per = logs.groupby('learner_id').size()
state = pd.cut(logs.cl_value, [-1, *THRESH, 2], right=False, labels=['Confidence', 'Exploration', 'Hesitation', 'Confusion']).value_counts()
e1, e2 = logs[logs.exercise_index == 1].score_norm, logs[logs.exercise_index == 2].score_norm
two = logs[logs.learner_id.isin(per[per == 2].index)].groupby('exercise_index')[['score_norm', 'cl_value']].mean()
qd = logs.learner_id.map(cl.groupby('learner_id').CL_Score.apply(list))
close = sum(any(abs(c - v) <= 0.01 for v in (vs if isinstance(vs, list) else [])) for c, vs in zip(logs.cl_value, qd))
R['rq3'] = dict(n_decisions=len(logs), n_students=int(per.size), two_decisions=int((per == 2).sum()), one_decision=int((per == 1).sum()),
                modules=sorted(logs.module_id.unique()), difficulty=sorted(logs.difficulty.unique().tolist()),
                cl_ge_060=int((logs.cl_value >= .60).sum()), cl_gt_050=int((logs.cl_value > .50).sum()),
                three_hints=int(((logs.cl_value > .5) | (logs.difficulty == 1)).sum()),
                recomputed_states=state.to_dict(), logged_labels=logs.cl_state.value_counts().to_dict(),
                mastered=int(logs.mastered.sum()), mastered_exercise_index=sorted(logs[logs.mastered.astype(bool)].exercise_index.unique().tolist()),
                score_ex1=[float(e1.mean()), float(e1.std(ddof=1)), int(len(e1))], score_ex2=[float(e2.mean()), float(e2.std(ddof=1)), int(len(e2))],
                paired_score=[float(two.score_norm[1]), float(two.score_norm[2])], paired_cl=[float(two.cl_value[1]), float(two.cl_value[2])],
                cl_matches_questionnaire=int(close), shap_message_empty=int(logs.shap_message.isna().sum()),
                error_type_general=int((logs.error_type == 'general').sum()), concatenated_rows=int(logs.next_action.astype(str).str.contains('general').sum()))
r3 = R['rq3']
print(f"{r3['n_decisions']} décisions de {r3['n_students']} étudiants ({r3['two_decisions']} avec 2, {r3['one_decision']} avec 1) ; modules {r3['modules']}, difficulté {r3['difficulty']}")
print(f"CL loggé ≥ 0,60 : {r3['cl_ge_060']} ({r3['cl_ge_060']/len(logs):.0%}) | CL loggé > 0,50 : {r3['cl_gt_050']} ({r3['cl_gt_050']/len(logs):.0%})")
print(f"Règle des 3 indices (Éq. hint_cond) : {r3['three_hints']} ({r3['three_hints']/len(logs):.0%}) | indices affichés (CL ≥ 0,60) : {r3['cl_ge_060']} ({r3['cl_ge_060']/len(logs):.0%})")
print('États recalculés avec les seuils de §3.1 :', {k: int(v) for k, v in state.items()})
print('Labels présents dans le log              :', r3['logged_labels'])
print('  CL loggé par label :', {k: (round(float(g.min()), 2), round(float(g.max()), 2)) for k, g in logs.groupby('cl_state').cl_value})
print(f"Décisions « mastered » : {r3['mastered']} (exercice n° {r3['mastered_exercise_index']})")
print(f"Score normalisé, 1ᵉʳ exercice : {e1.mean():.2f} (SD {e1.std(ddof=1):.2f}), n={len(e1)} | 2ᵉ exercice : {e2.mean():.2f} (SD {e2.std(ddof=1):.2f}), n={len(e2)}")
print(f"Étudiants avec 2 exercices (n={int(per.eq(2).sum())}) : score {two.score_norm[1]:.2f} → {two.score_norm[2]:.2f} ; CL loggé {two.cl_value[1]:.2f} → {two.cl_value[2]:.2f}")
print(f"CL loggé à ±0,01 d'une valeur questionnaire du même étudiant : {close}/{len(logs)} | explication vide : {r3['shap_message_empty']}/{len(logs)} | error_type « general » : {r3['error_type_general']}/{len(logs)} | lignes concaténées : {r3['concatenated_rows']}")

                                                                                                                                                
H('6. ANNEXE C — SENSIBILITÉ DE α ET DES CORRÉLATIONS (aucun questionnaire n\'est écarté de l\'analyse principale)')
share = cl.vec.map(n_stud) >= 2                                                                                                    
first_form = (cl.groupby('learner_id').cumcount() == 0).values
SCEN = {'A. Les 56 questionnaires (analyse principale)': np.ones(len(cl), bool), 'B. Jour 0 seul': d0, 'C. Jour 22 seul': d22,
        'D. Sans les questionnaires à profil partagé': ~share.values, 'E. Premier questionnaire de chaque étudiant': first_form}
rows = []
for nm, m in SCEN.items():
    ids = cl.learner_id.values[m]; lv = pd.Series(cl.clq.values[m]).groupby(ids).mean().round(10)
    lo_, hi_ = alpha_cluster_ci(X.values[m], ids); rp_, rg_ = stats.spearmanr(pre.loc[lv.index], lv), stats.spearmanr(ng_s.loc[lv.index], lv)
    rows.append({'Scénario': nm, 'Q.': int(m.sum()), 'Étud.': len(lv), 'α': round(cronbach(X.values[m]), 3), 'IC95% (par étudiant)': f'[{lo_:.2f}, {hi_:.2f}]',
                 'CL_q moy.': round(lv.mean(), 2), 'ρ pré': round(rp_[0], 2), 'p': round(rp_[1], 2), 'ρ gain': round(rg_[0], 2), 'p ': round(rg_[1], 2)})
sens = pd.DataFrame(rows); print(sens.to_string(index=False))
la_all = pd.Series(cl.clq.values).groupby(cl.learner_id.values).mean().round(10)
loo_rho = [stats.spearmanr(ng_s.loc[la_all.drop(s).index], la_all.drop(s))[0] for s in la_all.index]
loo_alpha = [cronbach(X.values[cl.learner_id.values != s]) for s in la_all.index]
print(f"Retrait d'un étudiant à la fois (56 questionnaires) : α de {np.nanmin(loo_alpha):.3f} à {np.nanmax(loo_alpha):.3f} ; ρ(CL_q, gain) de {min(loo_rho):.2f} à {max(loo_rho):.2f}")
R['annexe_c'] = dict(scenarios=sens.to_dict(orient='records'), leave_one_student_out=dict(alpha_range=[float(np.nanmin(loo_alpha)), float(np.nanmax(loo_alpha))], rho_gain_range=[float(min(loo_rho)), float(max(loo_rho))]),
                     day22=dict(n_students=int(len(fl)), mean_by_position=R['rq2']['within_learner']['mean_by_position'], mean_change=R['rq2']['within_learner']['mean_change'], wilcoxon_p=R['rq2']['within_learner']['wilcoxon_p']))
print(f"Jour 22 : changement moyen premier → dernier questionnaire = {R['rq2']['within_learner']['mean_change']:+.2f} (Wilcoxon p = {R['rq2']['within_learner']['wilcoxon_p']:.4f})")

                                                                
os.makedirs('article_results', exist_ok=True)
json.dump(R, open('article_results/resultats_article.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2,
          default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('\n💾 article_results/resultats_article.json  (tous les résultats ci-dessus, lisibles par machine)')


## Partie 8 — Contrôles de cohérence code ↔ article ↔ base (v5)
Cellule autonome (xlsx uniquement). Elle vérifie l'échelle des scores, `CL_Score_Mean`, les deux versions de Form 1, le démarrage à froid, les seuils de maîtrise et l'absence de dates calendaires.

In [ ]:
                                                                                
                                                                
                                                                                                  
                                                                                
import numpy as np, pandas as pd
_p = find_ails_xlsx() if 'find_ails_xlsx' in globals() else None
assert _p, 'base xlsx introuvable'
X8 = pd.read_excel(_p, sheet_name=None)
prof8, f1_8, res8, gain8 = X8['Learner_Profile'], X8['Form Responses 1'], X8['AILS_Results'], X8['Gains_Apprentissage']
print('Base :', _p, '| onglets :', list(X8))

                                                 
print('\n[1] Échelle pré/post — en-têtes :', [c for c in f1_8.columns if 'test score' in c])
print('    max pré =', gain8.pre_score.max(), '| max post =', gain8.post_score.max(), '→ échelle 0–100 (les en-têtes « /20 » sont FAUX)')
assert gain8.post_score.max() > 20

                                                                                            
likert = ['Q_Comfortable','Q_PriorKnowledge','Q_UnderstandQuickly','Q_Confident','Q_NeedsHelp',
          'Q_PrefersStepByStep','Q_PrefersHints','Q_PrefersConcise','Q_Motivated','Q_LosesFocus']
recalc = prof8[likert].mean(axis=1, skipna=True)
print('\n[2] CL_Score_Mean = moyenne des items Likert de profil disponibles : écart max =',
      round(float((recalc - prof8.CL_Score_Mean).abs().max()), 4))
assert (recalc - prof8.CL_Score_Mean).abs().max() < 0.01
print('    items de profil manquants par étudiant :', prof8[likert].isna().sum(axis=1).value_counts().to_dict())
print('    étudiants sans Q_Motivated/Q_LosesFocus :', sorted(prof8.loc[prof8.Q_Motivated.isna(), 'learner_id']))

                                                                                   
shift = f1_8['Day index'] == 0
print('\n[3] Form 1 : lignes à jour relatif 0 =', int(shift.sum()), '| lignes où « motivated » est du texte non-Likert :',
      int(f1_8.loc[shift, 'I feel motivated when solving exercises in this subject'].astype(str).str.match(r'^\d').eq(False).sum()))
print('    → valeur manquante pour ces étudiants ; défaut 0,5 (safe_get) si NON imputée ; médiane si imputée (v4)')

                                            
print('\n[4] cold_start_state :', {p: cold_start_state(p) if 'cold_start_state' in globals() else 'cellule 10 non exécutée' for p in (0.10, 0.50, 0.80)})
rule = lambda p: ('M01–M03' if p>=.70 else 'M01' if p>=.40 else 'aucun')
pri = ((prof8.Q_PriorKnowledge-1)/4)
print('    prior (Form 1, (x−1)/4) → modules initialement maîtrisés :', pri.map(rule).value_counts().to_dict())

                                                         
print('\n[5] θ_m utilisé par _check_mastery (cellule 21) : M04 = 0.78 ; EXP_MASTERY_SCORE = 0.70 = repli si module inconnu, jamais atteint pour M04')
print('    flags « mastered » du log pilote :', int(res8.mastered.sum()), '| exercices concernés :', sorted(res8.loc[res8.mastered, 'exercise_index'].unique()))
print('    colonnes du log pilote :', list(res8.columns))
print('    colonnes clq_raw / cl_smoothed (build v5 du code) présentes dans le log pilote :', {'clq_raw','cl_smoothed'} <= set(res8.columns))

                                                                       
dt = [(n, c) for n, d in X8.items() for c in d.columns if pd.api.types.is_datetime64_any_dtype(d[c])]
print('\n[6] colonnes de type date dans la base :', dt or 'aucune', '| timestamps du log : tous vides =', bool(res8.timestamp.isna().all()))
print('    indices relatifs présents (Form 1) :', sorted(f1_8['Day index'].unique().tolist()), '| (Cognitive_Load) :', sorted(X8['Cognitive_Load'].day_index.unique().tolist()))
print('    → pas de dates calendaires ; les indices relatifs restent dans la base mais sortent du texte principal')

                                   
print('\n[7] n pré/post =', len(gain8), '| profils =', len(prof8), '| logs =', res8.learner_id.nunique(),
      '| CL =', X8['Cognitive_Load'].learner_id.nunique())
